# Step 08 — Redistribution: the zone targets onto the buildings

The VISUM structure data give every traffic zone a target per activity
category: workers, school, university and kindergarten places, daily and
non-daily shopping, leisure. This notebook shares each zone's targets out over
the buildings in it — and, inside the buildings that several occupants share,
over those occupants — in proportion to the **floor area each occupant
occupies**, so that every building carries its part of each category it holds.
The zone totals are kept exactly — for Workers as far as a zone's buildings can
realistically hold them; the rest goes to the zones around it (section 7). The
five labels without a zone total get an area and a visitor value instead. No
model call is made here.

| | |
|---|---|
| **Reads** | `data/output/07_building_pois_classified.gpkg` — layers `buildings` (step 06 plus `mid_labels_merged`) and `building_pois` (the 7,864 asked POIs with their labels); `data/output/04_buildings_enriched.gpkg` — layer `buildings`, the eaves heights, OSM levels and OSM twins; `06_buildings_classified.gpkg` — layer `building_pois`, every building-POI pair with its share; `01_all_pois.gpkg` / `01_all_buildings_osm.gpkg`, the OSM tags of the POIs and building polygons; `data/input/zone_targets.gpkg` — layer `regionbsstructuredata_zone` |
| **Writes** | `data/output/08_redistribution.gpkg` — layers `buildings` (one per building: id, zone, name, ALKIS function, class, the model's labels and the labels the rules use (`labels_used`), storeys, floor area and occupied floor, its value of each of the seven categories, the visitor values of the five other labels and `value_assumed`), `pois` (the asked POIs: id, building, zone, name, the model's labels and the labels used (`poi_labels_used`), class, occupied floor, the same values) and `zones` (target and placed value per category) — only the necessary columns; the rules, vacancy and fallback bookkeeping are shown in this notebook |
| **Needs** | `pyogrio`, `geopandas`, `shapely`, `pandas`; `lib/floor_rules.py`, `lib/vacancy.py`, `lib/edu_carriers.py`, `lib/storey_evidence.py`, `lib/label_rules.py`, `lib/staff_catering.py`, `lib/gap_storeys.py`, `lib/visitor_rates.py`, `lib/unit_sizes.py`, `lib/workers_headcount.py` |

## State of this notebook

| step | | status |
|---|---|---|
| **08.1** | **The inputs** — the buildings, heights, POIs and zone targets, checked against each other | **implemented** |
| **08.2** | **The zones** — every building in the zone that holds its representative point; the rings of touching zones | **implemented** |
| **08.3** | **Storeys and floor area** — the height storeys, the reclassification, the residential context, the structure rules | **implemented** |
| **08.4** | **The units and their occupied floor area** — POI groups, vacancy, the shares, building / POI / rest / remainder units, the rules on single categories | **implemented** |
| **08.5** | **The weights** — one row per unit and category, the areas of the five other labels, the last-carrier guard, the checks | **implemented** |
| **08.6** | **The allocation** — every zone target over its carriers, ring by ring where a zone has none; School by type | **implemented** |
| **08.7** | **Workers: realistic headcounts** — each unit up to the published size of its activity, what a zone cannot hold to the zones around it | **implemented** |
| **08.8** | **The checks, the diagnostics and the file** | **implemented** |

## Decisions (`config.py` STEP 08)

* **Units.** A building without asked POIs is one unit with its step-06
  labels; in a building with asked POIs every asked POI is a unit under its own
  labels, Workers included.
* **Rest units: keep all.** A label the building's step-06 labels carry but
  none of its asked POIs does stays as a rest unit; it carries no Workers. It
  lies on the floor the POIs leave free — at most the floor its class rule
  gives a sole occupant, one average occupant's share where the POIs fill it.
  A rest label goes where an asked POI of the building is its evident source
  and step 07 labelled that POI otherwise (below).
* **Site rows give no unit** (`share_of_site` is an audit column).
* **Fallback.** A zone with a target but no carrier of a category widens ring by
  ring through the zones that touch it until carriers are found; the ring is
  pooled and the move is listed in section 6.
* **Zones with a target of 0.** Their Workers places share the Workers targets
  of the nearest ring of zones that have one, each unit's weight split by those
  targets — VISUM counts e.g. the Autostadt's staff in the VW plant zone; the
  regional total stays VISUM's. Totals are ignored only for the five labels
  without a total. University, Kindergarten and School places do the same with
  the zones that touch them only — a zone border cuts through a campus (without
  this the TU's Biozentrum and library would get nothing, the Chemistry Centre
  next door 16 % of the region); School by type. A place joins only zones with
  places of their own: a zone with a target but no place keeps the ring
  fallback — its touching zones, or the next ring out where they hold no
  place — and a school place splits its weight by the zones' targets of its own
  types. A place with nothing it could take in the touching zones looks up to
  ring 3. A Workers total above 0 but under 1 % of the zone's own Bosserhof
  expectation (100 expected Workers or more) counts as 0 for this sharing,
  with the touching zones only; the zone keeps its small total and takes no
  joiners.
* **School by type.** The School target is three VISUM columns — primary
  (`SG_4_GSCH`), secondary (`SG_4_WFSCH`) and vocational pupils (`SG_4_BS`, the
  two Kollegs included). Each goes to the zone's school places of its own type;
  a zone without a place of a type gives that type to all its school places, so
  every zone keeps its School total. A place's type comes from its names, its
  model type or its ALKIS function, else it counts as primary and secondary.
* **Who carries School, University and Kindergarten.** The model's education
  labels are corrected from evidence already in the data
  (`EDU_CARRIER_RULES`, `lib/edu_carriers.py`): a place whose own data
  name another use (a former school, a village hall, adult education, a
  children's home, elderly day care, a youth club, continuing education ...),
  a non-teaching part of a campus (Mensa, administration, computing, boarding
  house ...) or a place without any evidence loses the label; real schools
  typed after their Mensa or Kita get it, vocational schools labelled
  university become vocational School, and the school types follow OSM isced /
  grades tags, vocational campuses and the Oskar-Kämmer sites. A School place in
  a zone with a School total of 0, known only from the cadastre or a bare OSM
  outline, joins no neighbour and takes no ring fallback; the Kita part of a
  school, Kita polygons and squeezed Kita POIs weigh their own size. A dropped label leaves its share empty, and the
  last-carrier guard keeps a zone's only carrier. By a web check (`EDU_CARRIER_LISTS`) the
  Ostfalia computing centre keeps University and the SOS-Mütterzentrum Salzgitter its Kita (at most the childcare
  cap); three CJD Jugenddorf buildings and the Bornhausen sports hall lose School.
* **The labels of steps 06 and 07 reconciled** (`LABEL_RULES`,
  `lib/label_rules.py`). Step 06 read each building whole, step
  07 each asked POI on its own, with the same definitions — and they still read
  the same kinds of place differently (fast food: daily shopping on 44 % of the
  one-POI buildings, on 6 % of the POIs). By its own OSM use and data: an eating
  place carries leisure, not shopping (a café inside a hotel, pool or clubhouse
  only loses shopping; shops with seating and staff canteens keep theirs); a
  pharmacy carries retail_daily, a fuel station too unless its data show no
  shop (then errands); a beverage market, kiosk, newsagent, tobacco or lottery
  shop retail_daily instead of retail_non_daily; a service (salon, travel
  agency, tailor, rental, tattoo, betting ...) no retail_non_daily; a leisure
  venue labelled sports (pool, ice rink, bowling, climbing) leisure too. A rest
  label goes where an asked POI of the building is its evident source and step
  07 labelled that POI otherwise ("the POI's answer wins"); every other rest
  label stays. A label taken off a POI does not come back as a rest unit, and a
  dropped label's share goes to the place's other labels; the last-carrier
  guard keeps a zone's only carrier.
* **Staff catering** (`STAFF_CATERING_RULES`,
  `lib/staff_catering.py`). Works canteens, staff cafés and snack bars on
  company sites, school and university Mensas and hospital cafeterias carry no
  daily or non-daily shopping and no leisure: their guests are the staff,
  pupils and students already counted as Workers, School or University. A unit
  is staff catering by its own name, type or OSM `amenity=canteen`, or as an
  eating place in a works building or on a company site — never a shop, a place
  in a store building or a canteen open to all by its name. The rest unit of
  such a building loses daily shopping and leisure too ("the canteen answers").
  A dropped label's share goes to the unit's other labels. Staff shops and
  kiosks inside a plant — a shop or kiosk in a works building on a named
  company site, not public by its own words — count as staff catering too and
  count no visitors (Siemens' staff coffee shop would otherwise hold 26 % of
  its zone's daily shopping).
* **The Workers sectors are only a check.** VISUM also splits Workers into
  three sectors, but with one fixed mix per city or district for all its zones;
  the notebook prints our mix next to it and never uses it.
* **Small places stay small.** On top of the one-floor rule, a building filed
  as small-scale retail or a restaurant occupies at most the typical size of
  its kind (748 / 438 m², the p90 of the mapped shops and restaurants), as do
  its rest unit and such POIs mapped as points; the cut floor carries nothing;
  food stores keep their size. A misclassified large place cannot take a
  large share. One whose own data name a big-box chain (the chain list of the
  one-storey rule) takes the big-box cap instead — or its format's size:
  drugstores 1,103 m², pet stores 1,500 m², beverage stores 1,487 m², textile
  and shoe discounters 748 m² (`UNIT_SIZE_FORMATS`).
* **Places without evidence count a quarter.** A building with no POI of its
  own, no name (ALKIS or on its OSM outline) and no site other than a landuse
  area — its labels are the model's reading of ALKIS or a landuse area — weighs
  a quarter in Retail_Daily, Retail_Non-Daily and Leisure.
* **The zone of a building** is the zone holding its representative point;
  buildings in no zone — mostly outside the study area — get nothing.
* **Vacant places get nothing** (`lib/vacancy.py`): a POI tagged vacant /
  disused / abandoned, a ruin or named
  "Leerstand", or whose step-07 type says so; a building whose own POIs are all
  such, that lies at least half under an OSM polygon tagged abandoned / disused
  / vacant / ruins without an active POI inside, or whose step-05 type says
  vacant, derelict, closed …; a vacant unit in a shared building keeps its floor
  empty and the active tenants keep theirs. Every such place is listed in
  section 4. A building that no longer stands — ALKIS (survey 2017/18) still
  holds it, OSM shows a construction or brownfield site or other buildings
  there — gives no unit either (`GONE_BUILDINGS`, 78; buildings on a site named
  after an active occupant stay as stand-ins for what is rebuilt there). The
  OSM tags that say a place is not in use count as they say it
  (`VACANCY_OSM_SIGNS`): a building under works or gone (`building=construction`
  that says what is built or carries a `check_date`, `demolished`, `razed`,
  `removed`), a lifecycle prefix (`disused:shop=*`, `was:*` …) where the object
  has no active tag, `opening_hours` closed / off / no, a past `end_date`, a
  note saying it for good ("dauerhaft geschlossen", "Leerstand"), `shop=no`; a
  building only these leave empty stays where it lies on the named site of an
  active occupant, and a tag a dated source contradicts is not read
  (`stale_ids`: Netto Hillerse, open again since December 2025; the Wolfsburg
  library Nordstadt, open on Thursdays). A place known to be closed on the date
  of the list gives no unit either, nor do the asked POIs in it, except one its
  entry keeps open (`keep_pois`: the Antiquariat in the annex of Burg
  Dankwarderode) (`CLOSED_BUILDINGS`, checked on `CLOSED_AS_OF`, each with
  what, since, until and a dated source: the Stadthalle Braunschweig, the
  Vieweghaus, Burg Dankwarderode, Bad Gliesmarode, the Klinikum site
  Holwedestraße, the St.-Elisabeth-Krankenhaus Salzgitter, two former real
  stores, the Grotrian-Steinweg factory, a hall under conversion, the Rote Wiese
  tennis hall, the former Freizeitcenter Seesen and the former prison
  Braunschweig-Rennelberg; an OSM sign or `GONE_BUILDINGS` keeps its own
  reason). The zone totals still hold: the zone's other
  carriers take the share, or the touching zones where the place was the
  zone's last carrier (the Holwedestraße cafeteria's daily shopping).
* **Floor area, not volume.** Every unit weighs the floor area it occupies,
  `O`: Workers `O × f`, every other zone category `O / k_red` — the Bosserhof
  rates are per 100 m² of floor area.
* **Labels are never merged.** Only the seven labels with a zone total are
  redistributed (work, school, university, childcare, retail_daily,
  retail_non_daily, leisure); business, errands, sports, meetup and lessons
  feed no category — each gets an area `O / k_all` and a value `area × visitor
  rate / 100` (below). Errands does not feed Retail_Daily: zones that lose
  their only carrier use the ring fallback; thin zones stay as they are.
* **Separate splits.** `k_red` counts the unit's redistributed non-work labels,
  so an extra errands or meetup label never shrinks a weight; `k_all` counts all
  its non-work labels; rest units take the building's.
* **Storeys**: the height rule — eaves height capped at the mean height, 3.5 m
  storeys, halls one storey, OSM `building:levels` over both — plus the
  structure rules: worship buildings, fire stations, sports halls and pools,
  hangars, auditoria, towers and parking (over OSM levels too) one storey;
  storeyed company uses on factory sites by height, canteens at most 2; cinemas
  at most 2, IKEA 2; the 9 multi-level furniture houses by height, at most 4;
  single-level boxes outside the hall classes one storey; one storey where only
  the height gave more — shops and restaurants below 6.5 m or under one-level
  OSM outlines, shops with store evidence, low shop boxes, farm halls, fuel
  canopies and station shops, a station shop carrying its canopy; big low
  office and service boxes (over 5,000 m², eaves at most 12 m) only with hall
  evidence — the 23 halls the web and OSM check found among them, or a
  hall-type OSM outline — and unless their own data say office floors; real
  offices keep their floors (`FLOOR_OFFICE_BOX`); sport halls and studios
  where only the height gave more, unless their own data say floors —
  clubhouses below 800 m², boathouses, golf and stadium buildings keep theirs,
  unless all their own OSM outlines say one level (`FLOOR_SPORT_BOX`);
  department stores and shopping centres 4.5 m storeys; malls their mapped levels, else at most 3;
  outlets at most 2. Not adopted: the deep-plan office test, halls over OSM
  levels, an alternative storey scheme (h / 3.8, a 4-storey cap, a
  10,000 m² rule).
  An OSM-only building on step 04's 2.0 m placeholder (OSM gives no height and
  no levels) of a class storeyed by kind, below 2,500 m² and under no hall
  outline takes the median storeys of the ALKIS buildings of its class and
  footprint band — the class-and-size median, in place of the mean height of
  the height rule for these rows (`FLOOR_OSM_GAP_STOREYS`, `lib/gap_storeys.py`).
* **Reclassification.** The 8 shopping centres with fewer than 15 POI pairs and
  a full-range retailer are hypermarkets (factor 1.2, one storey); the 8
  Designer Outlets Wolfsburg buildings are factory outlet centres (2.15, at
  most 2 storeys). Four big halls the model filed as
  shops or restaurants take the class of what they are — the Anssems trailer
  megastore, VW Halle 31, a works hall with a pump and Stanze Stahl
  (`CLASS_FIX_BUILDINGS`); so do the 23 halls among the
  big low 'office' boxes — factories, logistics and storage halls, a meat
  wholesaler, a dealer workshop, a brewery hall, a furniture market.
* **Residential context** (dwellings in the building): an ALKIS residential
  code unless an office-building head, a non-residential OSM twin or an
  unconfirmed low box says otherwise; or an OSM residential twin; or a
  residential noun in the type. A house twin never makes a non-residential code
  residential; a Kita is residential only by its type. It switches only the
  street-level and small-unit uses.
* **Building units** take the storeys their class gives a sole occupant,
  refined by the only POI's group and level tag: shops and restaurants one
  storey with and without dwellings (two by a level tag over 0;1, or for a
  multi-level chain without dwellings); banks and agencies one with dwellings,
  all without; practices and offices one with dwellings unless the head names a
  whole office building; Kitas 1 / 2; plain homes one storey with their labels.
* **POI units** take a floor family from their own group — GROUND (a share of
  the one ground floor), ONE, LOW2, CAP2 / CAP3 or ALL (what the others leave)
  — and keep `min(share × floor area, family storeys × footprint)`; the
  ground-floor units together at most one footprint; generic POIs never ALL; a
  level tag only moves a unit up; one facility of several POIs shares the
  sole-occupant floor; a POI mapped as a point is capped at the p90 of its
  group's mapped polygons (every group: a default cap where a group has fewer
  than 10 polygons), before the ground-floor budget, and is never the main
  occupant — the freed floor goes to the main occupants, with none it stays
  free, a remainder where there are no dwellings; an anchor — a food store or
  big box in its own big-box building, or a point of family ALL whose class is
  the building's — is the main occupant and not capped; holiday flats and
  student dormitories give no unit but keep their floor. The sizes of POI
  units (`UNIT_SIZE_RULES`): a point counts at least its group's median polygon in its
  building's shares (not a point that repeats a polygon of its building);
  tenant points outside the shopping and outlet centres take the point cap and
  are never the main occupant; in a mall the tenant points' shares follow their
  group's typical size; a pharmacy occupies at most 307 m², a fuel canopy
  without a station shop at most the fuel cap. By format
  (`UNIT_SIZE_FORMATS`, `lib/unit_sizes.py`): the easyApotheke
  format up to 400 m²; counters and kiosks at least the mapped kiosk median
  (53 m²) in their building's shares, not a small shop's; a mall tenant of a
  listed chain at least its published size in its building's shares, and in
  the malls with a published retail floor it weighs that size. **The floor of
  the malls** (`MALL_RETAIL_FLOOR`): the shops, restaurants and
  services of 7 malls occupy at most their published retail floor (the BraWo
  Carree 17,345 m² instead of 55,652 m²), one factor for all of them, so a
  unit can end below the sizes above; a mall's own outline POI (BraWoPark's)
  stands for the shops OSM does not map and keeps only the floor its tenants
  leave; their non-retail tenants keep their floor, the rest — car parks,
  aisles — carries nothing.
* **The shares** are re-split over the asked POIs once the landuse pairs are
  removed; a vacant pair keeps an empty slot.
* **Remainder.** In a building without dwellings the floor no unit occupies is
  one Workers-only unit: at the normal-office factor 2.9 in shop and office
  buildings (unmapped upper-floor tenants), at the building's own factor in any
  other (its own main use: a hospital at 1.25, a Kita at 2.3); with dwellings
  it carries nothing.
* **Factors.** A POI unit takes the factor of its own POI-level class (the
  building's for generic POIs). Rates fixed: vehicle repair 2.5, nursing homes
  1.25, small retail 3.5, and the headline means 2.36 / 1.30 / 1.725. An
  office building filed in an industry, yard or transport class
  — its type names a whole office building, no hall or works word — takes the
  normal-office factor 2.9 on its own units, and an industry POI that only
  repeats the building's OSM outline takes the building's factor.
* **Single categories.** Churches, volunteer fire stations, sports halls and
  clubhouses: Workers 0, their other labels stay — on the building and rest
  units, and on the POI units that are the church (worship, or a POI without a
  type), the station (fire station) or the hall, its club or the school using it (sport, community,
  school, generic, unmapped; pools, ice rinks and play centres keep theirs);
  other tenants keep their Workers. A volunteer fire station labelled only work
  gets no unit; a church or sports hall labelled only work keeps its unit,
  which then carries nothing. The no-evidence residential
  buildings: one storey, Workers only (those with a retail / commercial twin
  keep their labels). Holiday flats and dormitories: no unit. Where such a rule
  leaves a zone without its last carrier, the removed units keep that category
  — the last-carrier guard. The Workers-0 rules also reach fitness and
  wellness halls (staffed fitness and wellness places keep theirs), worship
  buildings by the storey rules' worship flag and worship POIs in any building,
  city fire stations whose own names or operators say volunteer (not a zone's
  only Workers carrier) and parking garages and decks. A building whose only
  sites are landuse areas does not count as without evidence for the
  no-evidence residential rule (`NO_EVIDENCE_LANDUSE`): 460 residential-coded
  buildings keep their model labels.
* **Non-occupants** (amenity features asked as POIs — showers, bookcases,
  lockers …) give no unit, decided by tag, and their host keeps the floor —
  behind `REDIST_DROP_NON_OCCUPANTS`, counted in section 4.
* **Visitor values** of the five labels without a zone total: visitor
  arrivals per average working day, staff excluded = area × rate / 100, the
  middle value of the rate table (`VISITOR_RATE_ANCHORS`; low / high kept
  for sensitivity runs); where no published rate exists, our pick, marked in
  `value_assumed`. Zeros: cafes that also carry leisure (meetup), hotels
  (business), fuel stations, parking (errands), plain homes (meetup), company
  academies and school or university buildings (lessons), staff canteens and
  occupational health on company sites (all five). Pharmacies and post offices
  per outlet; a small-format rate only up to its group's point size cap, the
  class rate on the rest. The rates as meant (`VISITOR_RULES`,
  `lib/visitor_rates.py`): the sports type words
  set the rate in every sports class (gyms, clubhouses, riding, shooting and
  tennis halls, pools); the counter rate covers at most 388 m², the
  public-office rate the rest; big low office boxes count business and errands
  on one storey; university libraries count no errands, research and private
  libraries the office rate; the cafe zero holds wherever the gastronomy rate
  applies, company academies by their own words count no lessons, nor do
  school sports halls, and a building in no zone gets no value; a leisure
  venue whose rate counts its audience counts meetup only beyond its own
  Leisure share, max(0, meetup − Leisure share) (VISUM's
  Leisure holds such audiences only in part; the KP-ONE festival meadow keeps
  0), and event venues take the event rate; a building whose only POI is a
  heritage feature (a castle, a monastery) takes its class rate unless it is
  filed as a culture or leisure facility (else the care home in Schloss
  Schliestedt would count meetup at the theatre rate); staff shops count no
  visitors. Salzgitter's errands and sports values are probably too low, and
  Peine's nearly as much (thin OSM coverage; a note in section 8).
* Values stay floats: no rounding, so the totals hold exactly.
* **Workers: realistic headcounts** (`WORKERS_*`,
  `lib/workers_headcount.py`, section 7). VISUM's Workers are a
  real headcount of jobs, and one multiplier per zone would inflate every
  carrier of an over-full zone alike (the Peiner Festsäle would hold 1,095
  Workers for an operator with 34 staff for two venues). On top of the
  allocation every Workers unit gets an activity type and the published band
  of its kind: its realistic headcount is its Bosserhof value, never above the
  band's high or its size rule; an establishment holds at most twice that and
  never more than the published maximum of its type, an open-ended employer at
  most 3 × its Bosserhof value (industry, logistics, transport 5 ×), the
  remainder of an establishment's building at most 1 ×, and 39 named
  institutions at most their published staff; no lower limits. Each zone
  fills its units at one level; what it cannot hold goes to the under-filled
  units of the zones around it (the touching zones first, then rings 2 and 3),
  never above their realistic headcount; a zone with a Workers total of 0 or
  near 0 takes part like any other. The remainder of an establishment's
  building and the technical structures (sewage plants, biogas and composting
  plants, cisterns, bunkers, greenhouses, silos, animal shelters) take no
  spill, and a conference or seminar centre is an event venue, not business
  services; a technical structure holds at most its realistic headcount, and
  its words are read closely. The regional total stays VISUM's; the exact zone
  totals do not hold for Workers where a zone holds more than its buildings
  realistically can.
* **OSM levels, culture halls, food stores and institutions.** OSM levels on
  ALKIS buildings (outlines over at least half of the footprint, an implied
  storey of 2.5–6 m, buildings counted by the height alone), 5 m storeys for
  tall culture halls, the food stores' published size rule as their realistic
  headcount in both directions (EDEKA Center and REWE Center count as
  hypermarkets), and the named institutions raised to their published staff
  from the over-filled units around them. Not applied: sharing Retail and
  Leisure totals of 0, an office default, flat floor weights.
* **The halls of the VW Wolfsburg plant** take their OSM levels (at most 4, no
  power plant); the plant zone keeps 63,605 of its 70,309 VISUM Workers, 58,530
  with one storey per hall.

In [1]:
import os, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError('Cannot find the pipeline root (the folder containing config.py). '
                       f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.')
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import re
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import shapely

from config import (
    TARGET_CRS, OUTPUT_DIR, POI_CLASSIFIED_FILE, ENRICHED_BUILDINGS_FILE, LLM_ACTIVITY_LABELS, LLM_BOSSERHOF_CLASSES,
    ZONE_TARGETS_FILE, ZONE_LAYER, ZONE_ID_COL, ZONE_COLUMN_MAP, MID_LABEL_TO_ACTIVITY, REDIST_AREA_LABELS,
    BOSSERHOF_WEIGHTS, BOSSERHOF_NORMALIZATION_MAP, BOSSERHOF_FGSV_RANGES,
    REDIST_WORKERS_WEIGHT, REDIST_KEEP_UNCOVERED_BUILDING_CATEGORIES, REDIST_MULTI_CATEGORY_SPLIT,
    REDIST_FALLBACK_MAX_RINGS, REDISTRIBUTION_FILE, CLASSIFIED_BUILDINGS_FILE,
    REDIST_EXCLUDE_VACANT, VACANCY_POI_USES, VACANCY_NAME_PATTERN, VACANCY_TEXT_PATTERN, VACANCY_PARTIAL_PATTERN,
    VACANCY_POI_TAG_PATTERN, VACANCY_BUILDING_VALUES, VACANCY_BUILDING_TAG_PATTERN, VACANCY_POLYGON_MIN_COVER,
    ALL_POIS_FILE, ALL_BUILDINGS_OSM_FILE,
    FLOOR_STOREY_HEIGHT_M, FLOOR_OSM_LEVELS_MAX, FLOOR_OSM_ALKIS_LEVELS, FLOOR_CULTURE_STOREY, FLOOR_OSM_HALL_LEVELS, HALL_BOSSERHOF_CLASSES, HALL_TYPE_HEAD_SPLIT, HALL_TYPE_PATTERN,
    HALL_TYPE_EXCLUDE_PATTERN,
    REDIST_REMAINDER_WORKERS_FACTOR, REDIST_REMAINDER_OFFICE_CLASSES, CHURCH_POI_GROUPS, FLOOR_TYPE_PATTERNS, FLOOR_FULL_TYPE_KEYS, FLOOR_ALKIS_CODES, FLOOR_CRAFT_CLASSES,
    FLOOR_INDUSTRY_CLASSES, FLOOR_SPORT_CLASSES, FLOOR_IKEA_CLASSES, FLOOR_BRAND_UNHALL_CLASSES, FLOOR_LOCATION_SPLIT,
    FLOOR_HALL_SITE_PATTERN, FLOOR_CANTEEN_MAX_STOREYS, FLOOR_CINEMA_MAX_STOREYS, FLOOR_IKEA_PATTERN, FLOOR_IKEA_EXCLUDE_PATTERN,
    FLOOR_IKEA_STOREYS, FLOOR_FURNITURE_HOUSE_MIN_EAVES_M, FLOOR_FURNITURE_HOUSE_MIN_FOOTPRINT_M2, FLOOR_FURNITURE_HOUSE_MAX_STOREYS,
    FLOOR_FURNITURE_HOUSE_EXCLUDE_TYPE, FLOOR_FURNITURE_HOUSE_EXCLUDE_NAME, FLOOR_BOX_CLASSES, FLOOR_BOX_TYPE_PATTERN,
    FLOOR_BOX_MIN_FOOTPRINT_M2, FLOOR_BOX_MAX_EAVES_M, FLOOR_MALL_BOX_MAX_EAVES_M, FLOOR_RETAIL_STOREY_HEIGHT_M,
    FLOOR_RETAIL_TALL_CLASSES, FLOOR_MALL_LEVELS, FLOOR_MALL_MIN_LEVEL_POIS, FLOOR_MALL_MAX_STOREYS, FLOOR_MALL_SMALL_MAX_STOREYS,
    FLOOR_MALL_SMALL_FOOTPRINT_M2, FLOOR_OUTLET_MAX_STOREYS,
    RECLASS_HYPERMARKET_PATTERN, RECLASS_HYPERMARKET_MAX_PAIRS, RECLASS_OUTLET_PATTERN, RECLASS_OUTLET_USES, RECLASS_OUTLET_EXCLUDE,
    RC_ALKIS_RANGE, RC_ALKIS_CODES, RC_NONRES_ALKIS_RANGE, RC_RES_TYPE_PATTERN, RC_OFFICE_BUILDING_PATTERN, RC_RES_TWINS,
    RC_HOUSE_TWINS, RC_HOUSE_MIN_STOREYS, RC_NONRES_TWINS, RC_BOX_MIN_FOOTPRINT_M2, RC_BOX_MAX_STOREYS,
    POI_GROUPS, POI_KEY_FALLBACK, POI_GROUP_CLASS, POI_CLASS_BY_USE, POI_FOOD_STORE_CLASS, POI_DISCOUNT_CLASS,
    POI_DISCOUNT_PATTERN, POI_HYPER_CLASS, POI_HYPER_PATTERN,
    UNIT_CLASS_STOREYS, UNIT_STREET_CLASSES, UNIT_GROUP_STOREYS, UNIT_COUNTER_ALL_USES, UNIT_LEVEL_TAG_CLASSES,
    UNIT_LEVEL_TAG_GROUPS, UNIT_HOME_STOREYS, MULTI_LEVEL_CHAINS,
    POI_GROUP_FAMILY, POI_MALL_CLASSES, POI_MALL_UNIT_GROUPS, POI_BIGBOX_CLASSES, POI_CULTURE_ALL_USES, POI_CHAIN_GROUPS,
    POI_NO_UNIT_GROUPS, POI_DORM_USES, REDIST_LANDUSE_USES, POINT_SIZE_CAP_QUANTILE, POINT_SIZE_CAP_OUTLINE_SHARE,
    POINT_SIZE_CAP_M2, REDIST_DROP_NON_OCCUPANTS, NON_OCCUPANT_TAGS, NON_OCCUPANT_KEEP_KEYS, NON_OCCUPANT_INFO_STAFFED_PATTERN,
    CHURCH_ALKIS_CODES, CHURCH_POI_USES, VOLUNTEER_FIRE_PATTERN, VOLUNTEER_FIRE_EXCLUDE_PATTERN, VOLUNTEER_FIRE_EXCLUDE_HEAD_PATTERN,
    VOLUNTEER_FIRE_CITY_PATTERN, VOLUNTEER_FIRE_CITY_AGS, VOLUNTEER_FIRE_POI_USES, SPORTS_HALL_ALKIS_CODES, SPORTS_HALL_TYPE_PATTERN,
    SPORTS_HALL_CLASSES, SPORTS_HALL_KEEP_PATTERN, SPORTS_HALL_KEEP_IDS, SPORTS_HALL_POI_GROUPS,
    NO_EVIDENCE_FUNCTION_PREFIX, NO_EVIDENCE_CLASSES, NO_EVIDENCE_KEEP_TWINS,
    HOLIDAY_FLAT_CLASS, HOLIDAY_FLAT_FUNCTION_PREFIX, HOLIDAY_FLAT_USES, REDIST_LAST_CARRIER_GUARD,
    POINT_SIZE_CAP_MIN_POLYGONS, POINT_SIZE_CAP_DEFAULT_M2, POI_ANCHOR_GROUPS, REDIST_SHARE_ZERO_TARGET_CATEGORIES,
    REDIST_SHARE_ZERO_TARGET_MAX_RINGS, REDIST_SHARE_ZERO_TARGET_FAR_RINGS, REDIST_SHARE_NEAR_ZERO_TARGET,
    VISITOR_RATE_SCENARIO, VISITOR_RATE_ANCHORS, VISITOR_PER_OUTLET, VISITOR_GROUP_RATES, VISITOR_CLASS_RATES, VISITOR_TYPE_RATES,
    VISITOR_DEFAULT_RATES, VISITOR_SPORT_USES, VISITOR_DANCE_USES, VISITOR_EVENTS_USES, VISITOR_CAR_USES, VISITOR_SMALL_FORMAT_GROUPS,
    VISITOR_STAFF_ONLY_PATTERN, VISITOR_STAFF_ONLY_GROUPS, VISITOR_STAFF_ONLY_CLASSES,
    SCHOOL_TYPE_COLUMNS, SCHOOL_TYPE_PATTERNS, SCHOOL_TYPE_STRIP_PATTERN, SCHOOL_TYPE_ALKIS, SCHOOL_TYPE_DEFAULT,
    WORKER_SECTOR_COLUMNS, WORKER_SECTOR_CLASSES, ZONE_DISTRICT_COL,
    SMALL_PLACE_CAP_GROUPS, SMALL_PLACE_EXEMPT_TYPE_PATTERN, REDIST_NO_POI_WEIGHT, REDIST_NO_POI_CATEGORIES,
    EDU_CARRIER_RULES, FLOOR_ONE_STOREY_RULES, LABEL_RULES,
    STAFF_CATERING_RULES, FLOOR_OSM_GAP_STOREYS, NO_EVIDENCE_LANDUSE, VISITOR_RULES, UNIT_SIZE_RULES, CHURCH_FLAG_EXCLUDE_PATTERN,
    VOLUNTEER_FIRE_OWN_TEXT_PATTERN, PARKING_OTHER_USE_PATTERN, OFFICE_IN_INDUSTRY_EXCLUDE_PATTERN, CLASS_FIX_BUILDINGS,
    EDU_CARRIER_LISTS, GONE_BUILDINGS,
    FLOOR_OFFICE_BOX, UNIT_SIZE_FORMATS, MALL_RETAIL_FLOOR, MALL_NONRETAIL_GROUPS,
    FLOOR_SPORT_BOX, VACANCY_OSM_SIGNS, CLOSED_BUILDINGS, CLOSED_AS_OF,
    WORKERS_HEADCOUNT, WORKERS_NO_SPILL, WORKERS_ACTIVITY_GROUPS, WORKERS_ACTIVITY_KEYWORDS, WORKERS_ACTIVITY_POI_USES,
    WORKERS_ACTIVITY_SHOP_USES, WORKERS_ACTIVITY_CLASSES, WORKERS_ACTIVITY_RULES, WORKERS_BANDS, WORKERS_SUBTYPES, WORKERS_SIZES,
    WORKERS_INSTITUTIONS,
)
from lib.checks import require_file, require_non_empty, require_unique, require_cols, require_crs
from lib import vacancy
from lib import floor_rules
from lib import edu_carriers
from lib import storey_evidence
from lib import label_rules
from lib import staff_catering
from lib import gap_storeys
from lib import visitor_rates
from lib import unit_sizes
from lib import workers_headcount

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 70)

CATS = list(ZONE_COLUMN_MAP)                    # the seven target categories, in config order
AREA_LABELS = list(REDIST_AREA_LABELS)          # the five labels without a zone total: an area, no redistribution
RED_LABELS = [l for l in MID_LABEL_TO_ACTIVITY if l != 'work']   # the six redistributed labels besides work
def col(prefix, cat):
    return f"{prefix}_{cat.replace('-', '_')}"  # e.g. assigned_Retail_Non_Daily
SCHOOL_TYPES = list(SCHOOL_TYPE_COLUMNS)        # primary, secondary, vocational: the School target by type
def scol(prefix, t):
    return f'{prefix}_School_{t}'               # e.g. assigned_School_primary
_RX_SCHOOL = floor_rules.compile_school_patterns(SCHOOL_TYPE_PATTERNS, SCHOOL_TYPE_STRIP_PATTERN)

if REDIST_WORKERS_WEIGHT not in ('floor_x_bosserhof', 'floor') or REDIST_MULTI_CATEGORY_SPLIT not in ('even', 'full'):
    raise ValueError(f'unknown REDIST_WORKERS_WEIGHT {REDIST_WORKERS_WEIGHT!r} or REDIST_MULTI_CATEGORY_SPLIT {REDIST_MULTI_CATEGORY_SPLIT!r}')
if set(REDIST_SHARE_ZERO_TARGET_CATEGORIES) - set(CATS):
    raise ValueError(f'REDIST_SHARE_ZERO_TARGET_CATEGORIES holds unknown categories: {sorted(set(REDIST_SHARE_ZERO_TARGET_CATEGORIES) - set(CATS))}')
if (set(REDIST_SHARE_ZERO_TARGET_MAX_RINGS) != set(REDIST_SHARE_ZERO_TARGET_CATEGORIES)
        or any(r is not None and (not isinstance(r, int) or r < 1) for r in REDIST_SHARE_ZERO_TARGET_MAX_RINGS.values())):
    raise ValueError('REDIST_SHARE_ZERO_TARGET_MAX_RINGS needs one entry per shared category: None or a ring of 1 or more')
for _k in ('storeyed_pattern', 'box_type_pattern', 'box_chain_pattern', 'box_chain_exclude', 'fuel_pattern', 'wash_pattern', 'canopy_pattern',
           'station_not_shop_head'):
    re.compile(FLOOR_ONE_STOREY_RULES[_k])                                # one storey where only the height says more
label_rules.check_config(LABEL_RULES, LLM_ACTIVITY_LABELS)            # the labels of steps 06 and 07 reconciled
gap_storeys.check_config(FLOOR_OSM_GAP_STOREYS)                       # placeholder storeys, staff catering, Workers-0 tests, unit sizes
for _k in ('words', 'public', 'estate_pattern', 'kind_hospital', 'kind_school', 'kind_university'):
    re.compile(STAFF_CATERING_RULES[_k])
for _k in (CHURCH_FLAG_EXCLUDE_PATTERN, VOLUNTEER_FIRE_OWN_TEXT_PATTERN, PARKING_OTHER_USE_PATTERN, OFFICE_IN_INDUSTRY_EXCLUDE_PATTERN):
    re.compile(_k)
if (set(UNIT_SIZE_RULES) != {'point_min_quantile', 'mall_quantile', 'pharmacy_cap_m2', 'chain_cap_group', 'canopy_cap_group'}
        or not 0 < UNIT_SIZE_RULES['point_min_quantile'] < 1 or not 0 < UNIT_SIZE_RULES['mall_quantile'] < 1
        or UNIT_SIZE_RULES['chain_cap_group'] not in POINT_SIZE_CAP_M2 or UNIT_SIZE_RULES['canopy_cap_group'] not in POINT_SIZE_CAP_M2):
    raise ValueError('UNIT_SIZE_RULES: the keys, quantiles in (0, 1) and groups with a point cap')
unit_sizes.check_config(UNIT_SIZE_FORMATS)                                # formats, office boxes, staff shops, malls, heritage POIs
re.compile(FLOOR_OFFICE_BOX['floors_pattern'])
for _k in ('head_pattern', 'public'):
    re.compile(STAFF_CATERING_RULES['shops'][_k])
_mrf = [i_ for v_ in MALL_RETAIL_FLOOR.values() for i_ in v_['ids']]
if (set(FLOOR_OFFICE_BOX['classes']) - set(LLM_BOSSERHOF_CLASSES) or len(_mrf) != len(set(_mrf))
        or any(not v_['m2'] > 0 for v_ in MALL_RETAIL_FLOOR.values())
        or not set((VISITOR_RULES.get('heritage_poi') or {}).get('keep_classes', ())) <= set(LLM_BOSSERHOF_CLASSES)):
    raise ValueError('FLOOR_OFFICE_BOX / VISITOR_RULES heritage_poi: a class not in the catalogue, or MALL_RETAIL_FLOOR: a building in '
                     'two malls or a floor not above 0')
for _k in [k_ for k_ in FLOOR_SPORT_BOX if k_.endswith('pattern')]:   # sport boxes, OSM signs of non-use, closed places
    re.compile(FLOOR_SPORT_BOX[_k])
for _k in ('opening_hours_pattern', 'note_pattern', 'note_exclude_pattern'):
    re.compile(VACANCY_OSM_SIGNS[_k])
_cl_bad = [b_ for b_, v_ in CLOSED_BUILDINGS.items() if not {'what', 'since', 'until', 'source'} <= set(v_)
           or pd.Timestamp(v_['since']) > pd.Timestamp(CLOSED_AS_OF)
           or (v_['until'] is not None and not re.fullmatch(r'\d{4}(-\d{2}(-\d{2})?)?', str(v_['until'])))]
_st_bad = [k_ for k_ in VACANCY_OSM_SIGNS['stale_ids'] if not re.fullmatch(r'(?:node|way|relation)/\d+', k_)]
if (set(FLOOR_SPORT_BOX['classes']) - set(LLM_BOSSERHOF_CLASSES) or not set(FLOOR_OFFICE_BOX['hall_ids']) <= set(CLASS_FIX_BUILDINGS)
        or VACANCY_OSM_SIGNS['end_date_before'] != CLOSED_AS_OF or _cl_bad or _st_bad):
    raise ValueError(f'FLOOR_SPORT_BOX: a class not in the catalogue, FLOOR_OFFICE_BOX: a hall without its class fix, '
                     f'VACANCY_OSM_SIGNS: end_date_before not CLOSED_AS_OF or stale ids that are no OSM ids {_st_bad}, or '
                     f'CLOSED_BUILDINGS: an entry without what / since / until / source, closed only after {CLOSED_AS_OF} or with '
                     f'an until that is no date: {_cl_bad}')
workers_headcount.check_config(WORKERS_HEADCOUNT, WORKERS_ACTIVITY_GROUPS, WORKERS_ACTIVITY_KEYWORDS,     # Workers: realistic headcounts (section 7)
                               WORKERS_ACTIVITY_POI_USES, WORKERS_ACTIVITY_SHOP_USES, WORKERS_ACTIVITY_CLASSES,
                               WORKERS_ACTIVITY_RULES, WORKERS_BANDS, WORKERS_SUBTYPES, WORKERS_SIZES, WORKERS_INSTITUTIONS,
                               WORKERS_NO_SPILL, bosserhof_classes=LLM_BOSSERHOF_CLASSES, fgsv_ranges=BOSSERHOF_FGSV_RANGES)
if set(CLASS_FIX_BUILDINGS.values()) - set(LLM_BOSSERHOF_CLASSES) or set(EDU_CARRIER_LISTS) != {'keep', 'drop', 'kita_cap'}:
    raise ValueError('CLASS_FIX_BUILDINGS: a class not in the catalogue, or EDU_CARRIER_LISTS without keep / drop / kita_cap')
if set(FLOOR_ONE_STOREY_RULES['station_extra_pairs'].values()) & set(FLOOR_ONE_STOREY_RULES['station_extra_pairs']):
    raise ValueError('FLOOR_ONE_STOREY_RULES: a listed station shop is itself a listed canopy')
if (set(REDIST_SHARE_ZERO_TARGET_FAR_RINGS) - set(REDIST_SHARE_ZERO_TARGET_CATEGORIES)
        or any(not isinstance(r, int) or REDIST_SHARE_ZERO_TARGET_MAX_RINGS[c] is None or r < REDIST_SHARE_ZERO_TARGET_MAX_RINGS[c]
               for c, r in REDIST_SHARE_ZERO_TARGET_FAR_RINGS.items())):
    raise ValueError('REDIST_SHARE_ZERO_TARGET_FAR_RINGS: shared categories with a ring limit only, each ring at least that limit')
if (set(REDIST_SHARE_NEAR_ZERO_TARGET) - set(REDIST_SHARE_ZERO_TARGET_CATEGORIES)
        or set(REDIST_SHARE_NEAR_ZERO_TARGET) & ({'School'} | set(REDIST_SHARE_ZERO_TARGET_FAR_RINGS))
        or any(set(v) != {'max_share', 'min_expected', 'max_ring'} or not 0 < v['max_share'] < 1 or not v['min_expected'] >= 0
               or (v['max_ring'] is not None and (not isinstance(v['max_ring'], int) or v['max_ring'] < 1))
               for v in REDIST_SHARE_NEAR_ZERO_TARGET.values())):
    raise ValueError('REDIST_SHARE_NEAR_ZERO_TARGET: shared categories other than School and without a far ring, each with '
                     'max_share in (0, 1), min_expected >= 0 and max_ring None or 1 or more')
_sch_src = ZONE_COLUMN_MAP['School']
if sorted(SCHOOL_TYPE_COLUMNS.values()) != sorted([_sch_src] if isinstance(_sch_src, str) else _sch_src):
    raise AssertionError('SCHOOL_TYPE_COLUMNS must be exactly the School columns of ZONE_COLUMN_MAP')
if not set(SCHOOL_TYPE_DEFAULT) | {t for v in SCHOOL_TYPE_ALKIS.values() for t in v} <= set(SCHOOL_TYPE_COLUMNS) or not SCHOOL_TYPE_DEFAULT:
    raise AssertionError('SCHOOL_TYPE_DEFAULT and SCHOOL_TYPE_ALKIS may only name types of SCHOOL_TYPE_COLUMNS')
for _k, _v in EDU_CARRIER_RULES.items():            # who carries School, University and Kindergarten: every pattern compiles
    if isinstance(_v, str):
        re.compile(_v)
if not set(EDU_CARRIER_RULES['school_alkis']) <= set(SCHOOL_TYPE_ALKIS):
    raise AssertionError("EDU_CARRIER_RULES['school_alkis'] may only name ALKIS school functions of SCHOOL_TYPE_ALKIS")
if (set(SMALL_PLACE_CAP_GROUPS.values()) - set(POINT_SIZE_CAP_M2) or set(REDIST_NO_POI_CATEGORIES) - set(CATS)
        or not 0 < REDIST_NO_POI_WEIGHT <= 1):
    raise AssertionError('SMALL_PLACE_CAP_GROUPS needs groups with a POINT_SIZE_CAP_M2, REDIST_NO_POI_CATEGORIES known categories, '
                         'REDIST_NO_POI_WEIGHT a weight in (0, 1]')
if set(WORKER_SECTOR_CLASSES) - set(WORKER_SECTOR_COLUMNS) or len(set(WORKER_SECTOR_COLUMNS) - set(WORKER_SECTOR_CLASSES)) != 1:
    raise AssertionError('WORKER_SECTOR_CLASSES must list the classes of every sector of WORKER_SECTOR_COLUMNS but one (the rest)')
if sorted(MID_LABEL_TO_ACTIVITY.values()) != sorted(CATS) or 'work' not in MID_LABEL_TO_ACTIVITY:
    raise AssertionError('MID_LABEL_TO_ACTIVITY must map exactly one label onto each of the seven categories (labels are never merged)')
if set(MID_LABEL_TO_ACTIVITY) & set(AREA_LABELS) or set(MID_LABEL_TO_ACTIVITY) | set(AREA_LABELS) != set(LLM_ACTIVITY_LABELS):
    raise AssertionError('every label must be either redistributed (MID_LABEL_TO_ACTIVITY) or an area label (REDIST_AREA_LABELS), never both')
_rids = ({r for d in VISITOR_GROUP_RATES.values() for r in d.values()} | {r for d in VISITOR_CLASS_RATES.values() for r in d.values()}
         | {r for p, fb in VISITOR_TYPE_RATES.values() for r in [fb] + [x for _, x in p]} | set(VISITOR_DEFAULT_RATES.values())
         | set(VISITOR_SPORT_USES)) - {'CMP'}
if (_rids - set(VISITOR_RATE_ANCHORS) or set(VISITOR_DEFAULT_RATES) != set(AREA_LABELS)
        or set(VISITOR_GROUP_RATES) - set(AREA_LABELS) or set(VISITOR_CLASS_RATES) != set(AREA_LABELS)
        or VISITOR_RATE_SCENARIO not in ('low', 'mid', 'high')
        or any(not (lo <= mid <= hi) for lo, mid, hi, *_ in list(VISITOR_RATE_ANCHORS.values()) + list(VISITOR_PER_OUTLET.values()))):
    raise AssertionError(f'the visitor-rate tables do not fit: unknown rate ids {sorted(_rids - set(VISITOR_RATE_ANCHORS))}, a label '
                         'without its tables, an unknown scenario or a rate whose low / mid / high are out of order')
_mid = {k: (lo + hi) / 2 for k, (lo, hi) in BOSSERHOF_FGSV_RANGES.items()}
_off = {k: (m, BOSSERHOF_WEIGHTS.get(k)) for k, m in _mid.items() if BOSSERHOF_WEIGHTS.get(k) is None or abs(BOSSERHOF_WEIGHTS[k] - m) > 1e-9}
if _off:
    raise AssertionError(f'BOSSERHOF_WEIGHTS is not the midpoint of the FGSV range for {_off}')
_lists = {'HALL_BOSSERHOF_CLASSES': HALL_BOSSERHOF_CLASSES, 'FLOOR_CRAFT_CLASSES': FLOOR_CRAFT_CLASSES,
          'FLOOR_INDUSTRY_CLASSES': FLOOR_INDUSTRY_CLASSES, 'FLOOR_SPORT_CLASSES': FLOOR_SPORT_CLASSES,
          'FLOOR_IKEA_CLASSES': FLOOR_IKEA_CLASSES, 'FLOOR_BRAND_UNHALL_CLASSES': FLOOR_BRAND_UNHALL_CLASSES,
          'FLOOR_BOX_CLASSES': FLOOR_BOX_CLASSES, 'FLOOR_RETAIL_TALL_CLASSES': FLOOR_RETAIL_TALL_CLASSES,
          'UNIT_CLASS_STOREYS': UNIT_CLASS_STOREYS, 'UNIT_STREET_CLASSES': UNIT_STREET_CLASSES,
          'UNIT_LEVEL_TAG_CLASSES': UNIT_LEVEL_TAG_CLASSES, 'UNIT_HOME_STOREYS': UNIT_HOME_STOREYS,
          'POI_MALL_CLASSES': POI_MALL_CLASSES, 'POI_BIGBOX_CLASSES': POI_BIGBOX_CLASSES, 'SPORTS_HALL_CLASSES': SPORTS_HALL_CLASSES,
          'NO_EVIDENCE_CLASSES': NO_EVIDENCE_CLASSES, 'POI_GROUP_CLASS': set(POI_GROUP_CLASS.values()),
          'POI_CLASS_BY_USE': {c for _, _, c in POI_CLASS_BY_USE},
          'VISITOR_CLASS_RATES': {c for d in VISITOR_CLASS_RATES.values() for c in d}, 'VISITOR_STAFF_ONLY_CLASSES': VISITOR_STAFF_ONLY_CLASSES,
          'REDIST_REMAINDER_OFFICE_CLASSES': REDIST_REMAINDER_OFFICE_CLASSES,
          'WORKER_SECTOR_CLASSES': set().union(*WORKER_SECTOR_CLASSES.values()), 'SMALL_PLACE_CAP_GROUPS': set(SMALL_PLACE_CAP_GROUPS)}
_outside = {k: sorted(set(v) - set(LLM_BOSSERHOF_CLASSES)) for k, v in _lists.items() if set(v) - set(LLM_BOSSERHOF_CLASSES)}
if _outside:
    raise AssertionError(f'class names outside the Bosserhof catalogue: {_outside}')
_groups = set(POI_GROUPS) | {'unmapped'}
_g_lists = {'POI_GROUP_FAMILY': POI_GROUP_FAMILY, 'UNIT_GROUP_STOREYS': UNIT_GROUP_STOREYS, 'POINT_SIZE_CAP_M2': POINT_SIZE_CAP_M2,
            'UNIT_LEVEL_TAG_GROUPS': UNIT_LEVEL_TAG_GROUPS, 'POI_MALL_UNIT_GROUPS': POI_MALL_UNIT_GROUPS,
            'POI_CHAIN_GROUPS': POI_CHAIN_GROUPS, 'POI_NO_UNIT_GROUPS': POI_NO_UNIT_GROUPS, 'SPORTS_HALL_POI_GROUPS': SPORTS_HALL_POI_GROUPS,
            'POI_ANCHOR_GROUPS': POI_ANCHOR_GROUPS, 'VISITOR_GROUP_RATES': {g for d in VISITOR_GROUP_RATES.values() for g in d},
            'VISITOR_SMALL_FORMAT_GROUPS': VISITOR_SMALL_FORMAT_GROUPS, 'VISITOR_STAFF_ONLY_GROUPS': VISITOR_STAFF_ONLY_GROUPS, 'CHURCH_POI_GROUPS': CHURCH_POI_GROUPS}
_unknown = {k: sorted(set(v) - _groups) for k, v in _g_lists.items() if set(v) - _groups}
if _unknown or set(POI_GROUP_FAMILY) != _groups:
    raise AssertionError(f'POI groups outside POI_GROUPS: {_unknown}; groups without a floor family: {sorted(_groups - set(POI_GROUP_FAMILY))}')

print('Root  :', ROOT_DIR)
print('Input :', POI_CLASSIFIED_FILE.name, '+', ENRICHED_BUILDINGS_FILE.name, '+', ZONE_TARGETS_FILE.name, f'(layer {ZONE_LAYER})')
print('Output:', REDISTRIBUTION_FILE.name)
print(f'Run   : weight = occupied floor area ({REDIST_WORKERS_WEIGHT}) · split {REDIST_MULTI_CATEGORY_SPLIT} over the redistributed labels · '
      f'rest units {"kept" if REDIST_KEEP_UNCOVERED_BUILDING_CATEGORIES else "dropped"} · '
      f'vacant places {"left out" if REDIST_EXCLUDE_VACANT else "kept"} · non-occupants {"dropped" if REDIST_DROP_NON_OCCUPANTS else "kept"} · '
      f'last-carrier guard {"on" if REDIST_LAST_CARRIER_GUARD else "off"} · fallback '
      + ('ring by ring until carriers are found' if REDIST_FALLBACK_MAX_RINGS is None else f'up to ring {REDIST_FALLBACK_MAX_RINGS}'))
print(f'  ok  {len(MID_LABEL_TO_ACTIVITY)} labels are redistributed ({", ".join(f"{l} -> {c}" for l, c in MID_LABEL_TO_ACTIVITY.items())}); '
      f'{len(AREA_LABELS)} get an area only ({", ".join(AREA_LABELS)})')
print(f'  ok  BOSSERHOF_WEIGHTS is the midpoint of the FGSV range for all {len(_mid)} classes that have one; '
      f'every class list and POI group of the floor rules is in the catalogue')
print(f'  ok  visitor rates: {len(VISITOR_RATE_ANCHORS) - 1} rates and {len(VISITOR_PER_OUTLET)} per-outlet '
      f'values, the {VISITOR_RATE_SCENARIO} value used; {sum(a[3] for a in VISITOR_RATE_ANCHORS.values()) + sum(a[3] for a in VISITOR_PER_OUTLET.values())} '
      'of them our own pick where no published rate exists (marked assumed in the output)')

Root  : <repo>
Input : 07_building_pois_classified.gpkg + 04_buildings_enriched.gpkg + zone_targets.gpkg (layer regionbsstructuredata_zone)
Output: 08_redistribution.gpkg
Run   : weight = occupied floor area (floor_x_bosserhof) · split even over the redistributed labels · rest units kept · vacant places left out · non-occupants dropped · last-carrier guard on · fallback ring by ring until carriers are found
  ok  7 labels are redistributed (work -> Workers, school -> School, university -> University, childcare -> Kindergarten, retail_daily -> Retail_Daily, retail_non_daily -> Retail_Non-Daily, leisure -> Leisure); 5 get an area only (business, errands, sports, meetup, lessons)
  ok  BOSSERHOF_WEIGHTS is the midpoint of the FGSV range for all 37 classes that have one; every class list and POI group of the floor rules is in the catalogue
  ok  visitor rates: 54 rates and 2 per-outlet values, the mid value used; 19 of them our own pick where no published rate exists (marked assumed in the

## 1. The inputs

* the **buildings** of step 07.6 — step 06's layer plus `mid_labels_merged`:
  footprint, volume, ALKIS function, Bosserhof class, the step-06 labels and
  type. Every class must have a Workers factor and every label must be known;
* the **heights** of step 04 — the eaves height, OSM `building:levels`, the
  OSM twin tag of each ALKIS building (`osm_building` for the OSM gap rows) and
  the POI names;
* the **asked POIs** of step 07.6 — each an own pair of step 06 with the same
  share, the asked shares of a building summing to at most 1 (a landuse or
  vacant pair holds the rest);
* every **building-POI pair** of step 06 and the **OSM tags** of step 01 —
  the own pairs share each building exactly; they give the sole POI of a
  building, the level tags, the landuse and vacant pairs, the sets of the single-category rules;
* the **zone targets** — the seven categories built from the zone columns by
  `ZONE_COLUMN_MAP`, missing values as 0, keyed by the unique VISUM zone
  number.

In [2]:
for _p, _what in ((POI_CLASSIFIED_FILE, 'buildings and asked POIs (step 07.6)'), (ENRICHED_BUILDINGS_FILE, 'eaves heights, OSM levels and twins (step 04)'),
                  (CLASSIFIED_BUILDINGS_FILE, 'building-POI pairs (step 06)'), (ZONE_TARGETS_FILE, 'VISUM zone targets'),
                  (ALL_POIS_FILE, 'the OSM tags of the POIs (step 01)'), (ALL_BUILDINGS_OSM_FILE, 'the OSM building polygons (step 01)')):
    require_file(_p, _what)
print()

# --- the buildings ---------------------------------------------------------------------------
bld = pyogrio.read_dataframe(POI_CLASSIFIED_FILE, layer='buildings',
                             columns=['building_id', 'source', 'label_en', 'name', 'ags', 'function', 'area_m2', 'volume_3d_m3',
                                      'n_pois', 'n_sites', 'bosserhof_class', 'mid_labels', 'mid_labels_merged', 'llm_type'])
require_non_empty(bld, 'buildings')
require_unique(bld, 'building_id', 'buildings')
require_crs(bld, TARGET_CRS, 'buildings')
if bld['volume_3d_m3'].isna().any() or (bld['volume_3d_m3'] <= 0).any():
    raise AssertionError('a building without a positive volume')
if bld['area_m2'].isna().any() or (bld['area_m2'] <= 0).any():
    raise AssertionError('a building without a positive footprint')

def _norm(cls):
    key = re.sub(r'[^a-z]+', ' ', str(cls).lower()).strip()
    return BOSSERHOF_NORMALIZATION_MAP.get(key, key)
def factor(cls):                                # the Workers factor of a step-05 class name
    return BOSSERHOF_WEIGHTS[_norm(cls)]
_classes = bld['bosserhof_class'].dropna().unique()
_unmapped = sorted(c for c in set(_classes) | {POI_HYPER_CLASS, 'factory outlet centers'} if _norm(c) not in BOSSERHOF_WEIGHTS)
if bld['bosserhof_class'].isna().any() or _unmapped:
    raise AssertionError(f'Bosserhof classes without a factor: {_unmapped}; buildings without a class: {int(bld["bosserhof_class"].isna().sum())}')
print(f'  ok  all {len(_classes)} Bosserhof classes of the buildings have a Workers factor '
      f'(a relative factor, {min(map(factor, _classes))} to {max(map(factor, _classes))})')
_bad = sorted({l for s in pd.concat([bld['mid_labels'], bld['mid_labels_merged']]) for l in s.split(';')} - set(LLM_ACTIVITY_LABELS))
if _bad:
    raise AssertionError(f'unknown building labels: {_bad}')
print(f'  ok  {len(bld):,} buildings, {bld["volume_3d_m3"].sum() / 1e6:,.1f} M m3 on {bld["area_m2"].sum() / 1e6:,.2f} km2 of footprint; '
      'every label is known')

# --- the eaves heights, OSM levels and OSM twins (step 04) ---------------------------------------
_e = pyogrio.read_dataframe(ENRICHED_BUILDINGS_FILE, layer='buildings', read_geometry=False,
                            columns=['building_id', 'area_m2', 'height_eaves_max_m', 'osm_levels', 'osm_twin_tag', 'osm_building', 'poi_names',
                                     'osm_twin_name'])
require_unique(_e, 'building_id', 'step-04 buildings')
_e = _e.set_index('building_id')
if not bld['building_id'].isin(_e.index).all():
    raise AssertionError('a building missing from step 04')
if not np.allclose(bld['area_m2'], bld['building_id'].map(_e['area_m2'])):
    raise AssertionError('the footprints of steps 04 and 07 differ')
bld['height_eaves_m'] = bld['building_id'].map(_e['height_eaves_max_m'])
_lv = pd.to_numeric(bld['building_id'].map(_e['osm_levels']), errors='coerce')
bld['osm_levels'] = _lv.where(_lv.between(1, FLOOR_OSM_LEVELS_MAX) & (_lv == np.floor(_lv)))
for _c in ('osm_twin_tag', 'osm_building', 'poi_names', 'osm_twin_name'):
    bld[_c] = bld['building_id'].map(_e[_c])
print(f'  ok  OSM building:levels (1 to {FLOOR_OSM_LEVELS_MAX}) mapped on {int(bld["osm_levels"].notna().sum()):,} buildings; '
      f'eaves height for {int(bld["height_eaves_m"].gt(0).sum()):,}, {int((~bld["height_eaves_m"].gt(0)).sum()):,} without (they use the mean height); '
      f'an OSM twin tag on {int(bld["osm_twin_tag"].notna().sum()):,} ALKIS buildings')

# --- the asked POIs ---------------------------------------------------------------------------
pois = pyogrio.read_dataframe(POI_CLASSIFIED_FILE, layer='building_pois',
                              columns=['poi_id', 'building_id', 'poi_use', 'share_in_building', 'poi_mid_labels', 'name', 'poi_type'])
require_unique(pois, 'poi_id', 'asked POIs')
require_crs(pois, TARGET_CRS, 'asked POIs')
_orphans = set(pois['building_id']) - set(bld['building_id'])
if _orphans:
    raise AssertionError(f'{len(_orphans):,} asked POIs on buildings not in the layer, e.g. {sorted(_orphans)[:3]}')
if not pois['share_in_building'].between(0, 1, inclusive='right').all():
    raise AssertionError('an asked POI with a share outside (0, 1]')
_ssum = pois.groupby('building_id')['share_in_building'].sum()
if (_ssum > 1 + 1e-9).any():
    raise AssertionError(f'the asked shares sum above 1 on {int((_ssum > 1 + 1e-9).sum())} buildings')
_bad = sorted({l for s in pois['poi_mid_labels'] for l in s.split(';')} - set(LLM_ACTIVITY_LABELS))
if _bad:
    raise AssertionError(f'unknown POI labels: {_bad}')
print(f'  ok  {len(pois):,} asked POIs on {pois["building_id"].nunique():,} buildings; their shares sum to 1 on '
      f'{int((_ssum > 1 - 1e-9).sum()):,} buildings and to less on {int((_ssum <= 1 - 1e-9).sum()):,} '
      f'(a landuse or vacant pair holds the rest; below 0.5 on {int((_ssum < 0.5).sum()):,})')

# --- every building-POI pair of step 06, and the OSM tags of the POIs (step 01) ----------------------
pairs06 = pyogrio.read_dataframe(CLASSIFIED_BUILDINGS_FILE, layer='building_pois', read_geometry=False,
                                 columns=['building_id', 'poi_id', 'how', 'poi_use', 'name', 'parent_name', 'share_in_building'])
own06 = pairs06[pairs06['how'] != 'site'].copy()                     # the own POIs; site rows give no unit
_k = pois.merge(own06[['building_id', 'poi_id', 'how', 'share_in_building']], on=['building_id', 'poi_id'], how='left', suffixes=('', '_06'))
if _k['how'].isna().any() or not np.allclose(_k['share_in_building'], _k['share_in_building_06']):
    raise AssertionError('an asked POI that is no own pair of step 06, or whose share differs from step 06')
pois['how'] = _k['how'].to_numpy()
_osum = own06.groupby('building_id')['share_in_building'].sum()
if not np.allclose(_osum, 1.0):
    raise AssertionError('the own pairs of a building do not share it exactly')
poi_attrs = pyogrio.read_dataframe(ALL_POIS_FILE, layer='pois', read_geometry=False,
                                   columns=['poi_id', 'poi_use', 'poi_use_tag', 'geom_kind', 'area_m2', 'name', 'amenity', 'shop',
                                            'tourism', 'office', 'leisure', 'craft', 'healthcare', 'operator', 'landuse', 'tags'])
require_unique(poi_attrs, 'poi_id', 'step-01 POIs')
if not own06['poi_id'].isin(poi_attrs['poi_id']).all():
    raise AssertionError('an own POI of step 06 missing from step 01')
print(f'  ok  {len(pairs06):,} building-POI pairs of step 06, {len(own06):,} of them own POIs (they share each building exactly); '
      f'{len(poi_attrs):,} OSM POIs of step 01 with their tags')
# the OSM signs of non-use (lib/vacancy.py): the building polygons, and the signs every step-01 POI carries as its tags say
# them (VACANCY_OSM_SIGNS)
polygons = vacancy.polygon_signs(ALL_BUILDINGS_OSM_FILE, bld, VACANCY_BUILDING_VALUES, VACANCY_BUILDING_TAG_PATTERN,
                                 rules=VACANCY_OSM_SIGNS)
poi_osm_signs = vacancy.poi_sign_texts(ALL_POIS_FILE, VACANCY_OSM_SIGNS)
print(f'  ok  {int((polygons["cover"] >= VACANCY_POLYGON_MIN_COVER).sum()):,} buildings lie at least {VACANCY_POLYGON_MIN_COVER:.0%} '
      f'under an OSM building polygon with a sign of non-use ({int((polygons["cover_old"] >= VACANCY_POLYGON_MIN_COVER).sum()):,} '
      'tagged abandoned / disused / vacant / ruins - disused / abandoned:building only =yes -, the others by a '
      'sign of non-use: a lifecycle prefix such as disused:shop=*, works or gone, a past end_date, a note ...); '
      f'{int(poi_osm_signs.ne("").sum()):,} step-01 POIs carry a sign of non-use')
# who carries School, University and Kindergarten: the OSM outlines that tell a use, and the geometry of the
# education POIs and Mensas (lib/edu_carriers.py, EDU_CARRIER_RULES)
edu_outlines = edu_carriers.outlines(ALL_BUILDINGS_OSM_FILE, bld, building_values=EDU_CARRIER_RULES['outline_buildings'],
                                     amenity_values=EDU_CARRIER_RULES['outline_amenities'])
edu_geoms = edu_carriers.poi_geometries(ALL_POIS_FILE, bld.crs)
print(f'  ok  {edu_outlines["building_id"].nunique():,} buildings under an OSM outline that tells a use (school, university, Kita, church, '
      f'fire station, ruins ...) or names a Mensa; the point or polygon of {len(edu_geoms):,} POIs')
# one storey where only the height says more: every OSM building outline over every footprint, with its
# building:levels and building tag, and the roof outlines over each footprint (lib/storey_evidence.py, FLOOR_ONE_STOREY_RULES)
storey_pairs, storey_roof = storey_evidence.osm_pairs(ALL_BUILDINGS_OSM_FILE, bld, roof_tags=FLOOR_ONE_STOREY_RULES['canopy_roof_tags'])
storey_feats = storey_evidence.osm_features(
    storey_pairs, role_min=FLOOR_ONE_STOREY_RULES['role_min'], outline_min_share=FLOOR_ONE_STOREY_RULES['outline_min_share'],
    two_min_share=FLOOR_ONE_STOREY_RULES['two_min_share'], res_tags=FLOOR_ONE_STOREY_RULES['res_outline_tags'],
    res_min_share=FLOOR_ONE_STOREY_RULES['res_outline_min_share'])
print(f'  ok  {len(storey_pairs):,} overlaps with an OSM building outline on {storey_pairs["building_id"].nunique():,} buildings; outlines with '
      f'building:levels over {int(storey_feats["lvn_cov"].notna().sum()):,} of them, roof or carport outlines over {len(storey_roof):,}')

# --- the zone targets -------------------------------------------------------------------------
_src = sorted({c for v in ZONE_COLUMN_MAP.values() for c in ([v] if isinstance(v, str) else v)})
_chk = list(WORKER_SECTOR_COLUMNS.values()) + [ZONE_DISTRICT_COL]     # the Workers sectors and the district: for the sector check only
zones = pyogrio.read_dataframe(ZONE_TARGETS_FILE, layer=ZONE_LAYER, columns=[ZONE_ID_COL, 'NAME'] + _src + _chk)
require_cols(zones, [ZONE_ID_COL, 'NAME'] + _src + _chk, ZONE_LAYER)
require_unique(zones, ZONE_ID_COL, ZONE_LAYER)
require_crs(zones, TARGET_CRS, ZONE_LAYER)
zones = zones.reset_index(drop=True)
for _c, _v in ZONE_COLUMN_MAP.items():
    zones[col('target', _c)] = zones[[_v] if isinstance(_v, str) else list(_v)].fillna(0).sum(axis=1).astype(float)
if (zones[[col('target', c) for c in CATS]] < 0).any().any():
    raise AssertionError('a negative zone target')
for _t, _c in SCHOOL_TYPE_COLUMNS.items():                          # School by type
    zones[scol('target', _t)] = zones[_c].fillna(0).astype(float)
if not np.allclose(zones[[scol('target', t) for t in SCHOOL_TYPES]].sum(axis=1), zones[col('target', 'School')]):
    raise AssertionError('the school types do not add up to the School target')
if not np.allclose(zones[list(WORKER_SECTOR_COLUMNS.values())].fillna(0).sum(axis=1), zones[col('target', 'Workers')]):
    raise AssertionError('the Workers sectors do not add up to the Workers target')
print(f'  ok  {len(zones):,} zones; the targets (zones with a target above 0):')
for c in CATS:
    print(f'        {c:<17} {zones[col("target", c)].sum():>12,.0f}   ({int((zones[col("target", c)] > 0).sum())} zones)')
_ns = (zones[[scol('target', t) for t in SCHOOL_TYPES]] > 0).sum(axis=1)
print('        School by type: ' + ' · '.join(f'{t} {zones[scol("target", t)].sum():,.0f} ({int((zones[scol("target", t)] > 0).sum())} zones)'
                                            for t in SCHOOL_TYPES) + f'; {int((_ns > 1).sum())} zones have pupils of more than one type')
print(f'  ok  the three school types add up to School and the three Workers sectors to Workers in every zone (the sectors only for a check)')

  ok  07_building_pois_classified.gpkg (51.6 MB)
  ok  04_buildings_enriched.gpkg (42.5 MB)
  ok  06_buildings_classified.gpkg (55.1 MB)
  ok  zone_targets.gpkg (5.6 MB)
  ok  01_all_pois.gpkg (14.2 MB)
  ok  01_all_buildings_osm.gpkg (165.4 MB)



  ok  buildings: 39,786 rows
  ok  buildings.building_id: unique and non-null (39,786)
  ok  buildings: CRS EPSG:25832
  ok  all 43 Bosserhof classes of the buildings have a Workers factor (a relative factor, 0.5 to 4.15)
  ok  39,786 buildings, 277.2 M m3 on 29.62 km2 of footprint; every label is known


  ok  step-04 buildings.building_id: unique and non-null (39,786)


  ok  OSM building:levels (1 to 60) mapped on 212 buildings; eaves height for 38,186, 1,600 without (they use the mean height); an OSM twin tag on 30,676 ALKIS buildings
  ok  asked POIs.poi_id: unique and non-null (7,864)
  ok  asked POIs: CRS EPSG:25832
  ok  7,864 asked POIs on 2,913 buildings; their shares sum to 1 on 2,696 buildings and to less on 217 (a landuse or vacant pair holds the rest; below 0.5 on 128)


  ok  step-01 POIs.poi_id: unique and non-null (27,843)
  ok  41,285 building-POI pairs of step 06, 22,633 of them own POIs (they share each building exactly); 27,843 OSM POIs of step 01 with their tags


  ok  204 buildings lie at least 50% under an OSM building polygon with a sign of non-use (81 tagged abandoned / disused / vacant / ruins - disused / abandoned:building only =yes -, the others by a sign of non-use: a lifecycle prefix such as disused:shop=*, works or gone, a past end_date, a note ...); 64 step-01 POIs carry a sign of non-use


  ok  3,385 buildings under an OSM outline that tells a use (school, university, Kita, church, fire station, ruins ...) or names a Mensa; the point or polygon of 27,843 POIs


  ok  52,008 overlaps with an OSM building outline on 37,891 buildings; outlines with building:levels over 6,460 of them, roof or carport outlines over 549
  ok  regionbsstructuredata_zone: has ['NO', 'NAME', 'SG_3_BE~17', 'SG_4_BS', 'SG_4_GSCH', 'SG_4_HS', 'SG_4_KITA', 'SG_4_WFSCH', 'SG_5_EK_TB', 'SG_5_EK~19', 'SG_6_FR~23', 'SG_3_BES~8', 'SG_3_BE~11', 'SG_3_BE~14', 'RO_LANDK~1']
  ok  regionbsstructuredata_zone.NO: unique and non-null (880)
  ok  regionbsstructuredata_zone: CRS EPSG:25832
  ok  880 zones; the targets (zones with a target above 0):
        Workers                591,700   (783 zones)
        School                 149,999   (306 zones)
        University              38,253   (20 zones)
        Kindergarten            37,323   (421 zones)
        Retail_Daily           908,859   (566 zones)
        Retail_Non-Daily       777,989   (575 zones)
        Leisure                222,936   (858 zones)
        School by type: primary 39,085 (210 zones) · secondary 77,622 (119 

## 2. The zones

A building belongs to the zone that holds its **representative point** — a
point guaranteed to lie inside the footprint, which the centroid of a concave
building is not. A point may lie in at most one zone; a building in none gets
no target. The zones are repaired with `make_valid` first (a few VISUM
polygons are invalid; the zones layer is written with the repaired outlines),
and the **rings** for the fallback are built here:
ring 1 of a zone is every zone that touches or overlaps it, ring 2 every zone
that touches ring 1, and so on.

In [3]:
_zgeom = shapely.make_valid(zones.geometry.values)
print(f'  ..  {int((~shapely.is_valid(zones.geometry.values)).sum())} invalid zone polygon(s) repaired with make_valid')
_pts = gpd.GeoDataFrame({'building_id': bld['building_id'].values},
                        geometry=bld.geometry.representative_point().values, crs=bld.crs)
_zz = gpd.GeoDataFrame({'zone_idx': np.arange(len(zones))}, geometry=_zgeom, crs=zones.crs)
_hit = gpd.sjoin(_pts, _zz, how='left', predicate='within')
_two = _hit['building_id'].duplicated(keep=False) & _hit['zone_idx'].notna()
if _two.any():
    raise AssertionError(f'{_hit.loc[_two, "building_id"].nunique()} buildings lie in two zones, '
                         f'e.g. {_hit.loc[_two, "building_id"].head(3).tolist()}')
bld['zone_idx'] = bld['building_id'].map(_hit.drop_duplicates('building_id').set_index('building_id')['zone_idx'])
bld['zone_no'] = bld['zone_idx'].map(zones[ZONE_ID_COL]).astype('Int64')
_none = bld['zone_idx'].isna()
print(f'  ok  {int((~_none).sum()):,} buildings in {bld["zone_idx"].nunique():,} zones, each in exactly one')
print(f'  ..  {int(_none.sum()):,} buildings in no zone get no target: {bld.loc[_none, "volume_3d_m3"].sum() / 1e6:,.2f} M m3 '
      f'({100 * bld.loc[_none, "volume_3d_m3"].sum() / bld["volume_3d_m3"].sum():.2f} % of the volume); by district key: '
      f'{bld.loc[_none, "ags"].astype("string").str[:5].fillna("none").value_counts().head(6).to_dict()}')
print(f'  ..  {int((~zones.index.isin(bld["zone_idx"].dropna().astype(int))).sum())} zone(s) hold no building at all')

# the rings: the zones that touch or overlap each zone
_i, _j = shapely.STRtree(_zgeom).query(_zgeom, predicate='intersects')
NEIGHBOURS = {k: set() for k in range(len(zones))}
for _a, _b in zip(_i, _j):
    if _a != _b:
        NEIGHBOURS[int(_a)].add(int(_b))
_deg = pd.Series({k: len(v) for k, v in NEIGHBOURS.items()})
print(f'  ok  ring 1 of a zone holds {_deg.min()} to {_deg.max()} zones, median {_deg.median():.0f}; '
      f'{int((_deg == 0).sum())} zone(s) touch no other')

  ..  4 invalid zone polygon(s) repaired with make_valid
  ok  39,560 buildings in 867 zones, each in exactly one
  ..  226 buildings in no zone get no target: 2.70 M m3 (0.97 % of the volume); by district key: {'03241': 138, '03254': 49, '03159': 21, '03351': 14, '03155': 4}
  ..  13 zone(s) hold no building at all
  ok  ring 1 of a zone holds 1 to 19 zones, median 6; 0 zone(s) touch no other


## 3. Storeys and floor area

Floor area FA = footprint A × storeys N. N starts from the **height
rule** (`storeys_08b`): `h` = the eaves height capped at the mean height (volume ÷ footprint),
the mean height where no eaves height exists; N = `max(1, round(h / 3.5 m))`;
a hall (a hall class, or a hall word in the head of the type, never with the
exclusion words) is one storey; OSM `building:levels` overrides both.

Before that, the **reclassification**: shopping centres that are hypermarkets
or factory outlet centres take that class. The **residential context** is read
on the height storeys. Then the **structure rules** apply in order where OSM
levels are not mapped (parking also where they are), a later rule overriding
an earlier one: hall classes with a residential head, a hall word before the
office word, crafts, storeyed company uses and canteens on factory sites,
worship, fire stations, sports halls and pools, hangars and auditoria, towers,
parking, cinemas, outlets and brand words that are no halls, IKEA, the
furniture houses, the single-level boxes (their words read plural-safe). Then
**one storey where only the height gave more** (3.4):
shops and restaurants below 6.5 m (a branch of a multi-level chain too) or
under OSM outlines that all say one level (service buildings only under their
own level-1 outline, without an attic, below 7.5 m), shops of 800 m² or more
with store words, a box chain or a box shop tag (eaves up to 8.75 m, 9 m from
1,500 m²), shops of 1,000 m² or more with eaves below 7 m, shops and
restaurants under an OSM barn, shed, greenhouse or roof outline or with an
ALKIS farm, depot or car-wash function, and station shops — never over OSM
levels, a structure rule, dwellings, a POI upstairs, or (for the shops) a
practice, office, school, Kita or hotel inside or a type naming upper floors;
and fuel-station canopies wherever OSM gives no levels; then big low office
and service boxes — the model's office or service class (before the class
fix), over 5,000 m², eaves at most 12 m — only with hall evidence (the 23 halls
the web and OSM check found, 3 of them one storey already as halls of their new
class; or a hall-type OSM outline over at least half the footprint) and unless
OSM levels, an OSM office outline or a type naming several floors say office
floors (the big low boxes are mostly halls the model typed 'office' from a
generic ALKIS function, and real offices keep their floors; `FLOOR_OFFICE_BOX`); last, sport halls and studios
(`FLOOR_SPORT_BOX`, `storey_evidence.sport_boxes`): a sport class
whose own data name a sport use, eaves up to 12.5 m — a hall activity (riding,
shooting, pool, tennis, climbing, bowling, indoor play, a sports hall) from
300 m² (riding and shooting at any size), a studio or a generic sports centre
from 300 m² where the height gave two storeys —, unless another non-sport
occupant, a mixed-use, shop or office type, a school, hotel, care home or
hospital, a storeyed sport use, a type naming floors or OSM levels say floors;
clubhouses below 800 m², boathouses, golf and stadium buildings keep theirs
(like the canteens of the industry rule: about half of the mapped clubhouses
have an upper or attic floor), and any candidate sport building whose own OSM
outlines all say one level counts one too. Then 4.5 m
storeys for department stores and shopping centres; then the mall caps. Two fixes: four big
halls the model filed as shops or restaurants — and the 23 halls
among the big low office boxes — take the class of what they are, before the
height storeys
(`CLASS_FIX_BUILDINGS`); and an OSM-only building on step 04's 2.0 m
placeholder — OSM gives no height and no levels — of a class storeyed by kind,
below 2,500 m² and under no hall outline, takes the median storeys of the
ALKIS buildings of its class and footprint band with a LoD2 eaves height, no
hall and no dwellings (3.2b, after the residential context and before the
structure rules; `FLOOR_OSM_GAP_STOREYS`, `lib/gap_storeys.py`). `storey_rule` records the
source of the height storeys and every rule that touched the building. The cell lists how many
buildings each rule touched and the largest change per rule.

Two storey rules come
before 3.4. **OSM levels on ALKIS buildings** (3.3b): step 04 keeps
`building:levels` on OSM rows only, so the OSM override alone reaches about 210
buildings; the OSM outlines over an ALKIS building give its storeys where
they cover at least half of the footprint, on buildings counted by the height
alone (no hall, no structure rule) and only where the implied storey height lies
between 2.5 and 6 m (`FLOOR_OSM_ALKIS_LEVELS`) — 4,373 buildings, 1,458 of them
one storey fewer than the height gave (schools, offices, hospitals). **Tall
culture halls** (3.3c): a museum, theatre, event hall, library or castle of the
class entertainment, culture counted at three storeys or more by the height
alone takes 5 m storeys (`FLOOR_CULTURE_STOREY`; 132 buildings, the
Staatstheater 7 → 5).

**OSM levels on the halls of the VW Wolfsburg
plant** (3.3d, `FLOOR_OSM_HALL_LEVELS`): in the plant zone a hall under OSM
outlines with levels takes them, at most 4 and only where the implied storey
lies between 2.5 and 6 m, never a power plant — 55 halls (Hallen 1–4 at 4
levels, the halls 12–14 and 16 at 3). With one storey the plant zone would keep only
58,530 of its 70,309 VISUM Workers against about 62,000–70,000 employees on the
site; with the levels it keeps 63,605. The region's other halls keep the hall rule.

In [4]:
# --- the building attributes the rules read --------------------------------------------------------
bld['code'] = pd.to_numeric(bld['function'].fillna('').str.extract(r'^31001_(\d{4})$')[0], errors='coerce')
bld['head'] = floor_rules.head(bld['llm_type'], HALL_TYPE_HEAD_SPLIT)
bld['mean_height_m'] = bld['volume_3d_m3'] / bld['area_m2']

# --- the reclassification: hypermarkets and outlet centres filed as shopping centres --------------------
bld['class_06'] = bld['bosserhof_class']
bld['bosserhof_class'], _hyper, _outlet = floor_rules.reclassify(
    bld, pairs06, mall_class='shopping centers', hyper_class=POI_HYPER_CLASS, outlet_class='factory outlet centers',
    hyper_pattern=RECLASS_HYPERMARKET_PATTERN, hyper_max_pairs=RECLASS_HYPERMARKET_MAX_PAIRS,
    outlet_pattern=RECLASS_OUTLET_PATTERN, outlet_uses=RECLASS_OUTLET_USES, outlet_exclude=RECLASS_OUTLET_EXCLUDE)
bld['workers_factor'] = bld['bosserhof_class'].map(factor)
# the class fix: four big halls the model filed as shops or restaurants take the class of what they are (Anssems trailer
# megastore, VW Halle 31, the works hall with the tankpool24 pump, Stanze Stahl) - CLASS_FIX_BUILDINGS; their factor and their
# storeys follow the new class. So do the 23 big low boxes typed 'office' that the web and OSM check found to be halls
_cfm = bld['building_id'].isin(list(CLASS_FIX_BUILDINGS))
if int(_cfm.sum()) != len(CLASS_FIX_BUILDINGS):
    raise AssertionError(f'CLASS_FIX_BUILDINGS: {len(CLASS_FIX_BUILDINGS) - int(_cfm.sum())} building(s) not found')
_r16_miss = (({i_ for v_ in MALL_RETAIL_FLOOR.values() for i_ in v_['ids']}
              | set((VISITOR_RULES.get('audience_zero') or {}).get('keep_zero_buildings', ()))) - set(bld['building_id']))
if _r16_miss:                                   # the listed malls and the venue kept at 0 must exist
    raise AssertionError(f'MALL_RETAIL_FLOOR / keep_zero_buildings: building(s) not found: {sorted(_r16_miss)}')
_cf_old = dict(zip(bld.loc[_cfm, 'building_id'], bld.loc[_cfm, 'bosserhof_class']))
bld.loc[_cfm, 'bosserhof_class'] = bld.loc[_cfm, 'building_id'].map(CLASS_FIX_BUILDINGS)
bld['workers_factor'] = bld['bosserhof_class'].map(factor)
print(f'  ok  the class fix (with the {len(FLOOR_OFFICE_BOX["hall_ids"])} halls among the office boxes): '
      f'{len(CLASS_FIX_BUILDINGS)} buildings '
      'take the class of what they are - ' + ' · '.join(f'{c_} {n_}' for c_, n_ in pd.Series(CLASS_FIX_BUILDINGS).value_counts().items())
      + ' (from ' + ' · '.join(f'{c_} {n_}' for c_, n_ in pd.Series(_cf_old).value_counts().items()) + ')')
print(f'  ok  reclassified: {int(_hyper.sum())} shopping centres -> {POI_HYPER_CLASS} (factor {factor(POI_HYPER_CLASS)}), '
      f'{int(_outlet.sum())} -> factory outlet centers (factor {factor("factory outlet centers")}); '
      f'{int(bld["class_06"].ne(bld["bosserhof_class"]).sum())} buildings change class')

# --- 3.1 the height storeys (storeys_08b) -----------------------------------------------------------
base = floor_rules.base_storeys(bld, storey_height=FLOOR_STOREY_HEIGHT_M, hall_classes=HALL_BOSSERHOF_CLASSES,
                                head_split=HALL_TYPE_HEAD_SPLIT, hall_pattern=HALL_TYPE_PATTERN, hall_exclude=HALL_TYPE_EXCLUDE_PATTERN)
print(f'  ok  the height storeys: height from ' + ' · '.join(f'{k} {v:,}' for k, v in base['height_source'].value_counts().items())
      + '; storeys from ' + ' · '.join(f'{k} {v:,}' for k, v in base['storeys_source'].value_counts().items())
      + f'; {(bld["area_m2"] * base["storeys"]).sum() / 1e6:,.2f} km2 of floor area')

# --- 3.2 the residential context -----------------------------------------------------------------------
rcx = floor_rules.residential_context(
    bld, base['storeys'], res_range=RC_ALKIS_RANGE, res_codes=RC_ALKIS_CODES, res_type=RC_RES_TYPE_PATTERN,
    office_whole=RC_OFFICE_BUILDING_PATTERN, res_twins=RC_RES_TWINS, house_twins=RC_HOUSE_TWINS, nonres_twins=RC_NONRES_TWINS,
    house_min_storeys=RC_HOUSE_MIN_STOREYS, box_min_m2=RC_BOX_MIN_FOOTPRINT_M2, box_max_storeys=RC_BOX_MAX_STOREYS,
    nonres_range=RC_NONRES_ALKIS_RANGE, kita_class='kindergartens')
bld['RC'] = rcx['RC']
bld['osm_tag'] = rcx['osm_tag']
_a = rcx['alk_res']
print(f'  ok  residential context on {int(bld["RC"].sum()):,} buildings: ALKIS residential code {int(_a.sum()):,}, of which taken back by '
      f'an office-building head {int((_a & rcx["office_whole"]).sum()):,}, a non-residential OSM twin {int((_a & rcx["twin_non"]).sum()):,}, '
      f'an unconfirmed low box {int(rcx["box_quirk"].sum()):,}; added by an OSM residential twin {int((~_a & rcx["twin_res"]).sum()):,} '
      f'and a residential noun in the type {int((~_a & ~rcx["twin_res"] & rcx["type_res"]).sum()):,}; stress-test fixes: a house twin on a '
      f'non-residential code {int(rcx["house_on_nonres"].sum()):,}, Kitas by type only {int(rcx["kita_by_type"].sum()):,} changed')

# --- 3.2b OSM-only buildings on step 04's placeholder height -------------------------------------------------------------------
# where OSM gives neither a height nor levels, step 04 wrote 1 x 2.0 m (OSM_GAP_DEFAULT_FLOORS x OSM_GAP_FLOOR_HEIGHT_M), a
# placeholder the height rule read as one storey. Such a building of a class that is storeyed by kind, below 2,500 m2 and under no
# hall-like OSM outline, takes the median storeys of the ALKIS buildings of its class and footprint band that have a LoD2 eaves
# height, no hall and no dwellings ("the class-and-size median", lib/gap_storeys.py, FLOOR_OSM_GAP_STOREYS); for these rows it
# replaces the mean height of the height rule. After the residential context (an estimate never makes dwellings), before the
# structure rules, which apply to it as to any building
_gap, gap_estimate = gap_storeys.estimate(bld, base, **FLOOR_OSM_GAP_STOREYS)
_gap_up = gap_estimate.index[gap_estimate['storeys'].to_numpy() > base.loc[gap_estimate.index, 'storeys'].to_numpy()]
base.loc[_gap_up, 'storeys'] = gap_estimate.loc[_gap_up, 'storeys']
base.loc[_gap_up, 'height_storeys'] = gap_estimate.loc[_gap_up, 'storeys']
base.loc[_gap_up, 'storeys_source'] = 'osm estimate'
print(f'  ok  OSM-only buildings without levels or a height (step 04\'s {FLOOR_OSM_GAP_STOREYS["placeholder_height_m"]} m): '
      f'{int(_gap.sum()):,} that are no hall, {len(gap_estimate):,} of them of a kind that takes an estimate, {len(_gap_up):,} above one '
      'storey: ' + ' · '.join(f'{int(k)} storeys {v:,}' for k, v in gap_estimate.loc[_gap_up, 'storeys'].value_counts().sort_index().items())
      + '; the median from ' + ' · '.join(f'{k} {v:,}' for k, v in gap_estimate.loc[_gap_up, 'source'].value_counts().items()))

# --- 3.3 the structure rules ---------------------------------------------------------------------------
tf = floor_rules.type_flags(bld['head'], bld['llm_type'], FLOOR_TYPE_PATTERNS, FLOOR_FULL_TYPE_KEYS)
own06 = own06.merge(poi_attrs[['poi_id', 'tags']], on='poi_id', how='left')
own06['levels'] = floor_rules.tag_value(own06['tags'], 'level').map(floor_rules.parse_levels)
_upper = set(own06.loc[own06['levels'].map(lambda l: l is not None and max(l) >= 1), 'building_id'])
_own_levels = pd.DataFrame({'building_id': own06['building_id'],
                            'level': pd.to_numeric(floor_rules.tag_value(own06['tags'], 'level').str.extract(r'^\s*(-?\d+)')[0], errors='coerce')})
N, _why, sf = floor_rules.structure_storeys(
    bld, tf, base, bld['RC'], _upper, codes=FLOOR_ALKIS_CODES, hall_classes=HALL_BOSSERHOF_CLASSES, craft_classes=FLOOR_CRAFT_CLASSES,
    industry_classes=FLOOR_INDUSTRY_CLASSES, sport_classes=FLOOR_SPORT_CLASSES, ikea_classes=FLOOR_IKEA_CLASSES,
    brand_unhall_classes=FLOOR_BRAND_UNHALL_CLASSES, furniture_class='furniture stores', box_classes=FLOOR_BOX_CLASSES,
    mall_class='shopping centers', hall_word=HALL_TYPE_PATTERN, hall_site=FLOOR_HALL_SITE_PATTERN, town_hall=FLOOR_TYPE_PATTERNS['town_hall'],
    location_split=FLOOR_LOCATION_SPLIT, canteen_max=FLOOR_CANTEEN_MAX_STOREYS, cinema_max=FLOOR_CINEMA_MAX_STOREYS,
    ikea_pattern=FLOOR_IKEA_PATTERN, ikea_exclude=FLOOR_IKEA_EXCLUDE_PATTERN, ikea_storeys=FLOOR_IKEA_STOREYS,
    furniture_min_eaves=FLOOR_FURNITURE_HOUSE_MIN_EAVES_M, furniture_min_m2=FLOOR_FURNITURE_HOUSE_MIN_FOOTPRINT_M2,
    furniture_max=FLOOR_FURNITURE_HOUSE_MAX_STOREYS, furniture_exclude_type=FLOOR_FURNITURE_HOUSE_EXCLUDE_TYPE,
    furniture_exclude_name=FLOOR_FURNITURE_HOUSE_EXCLUDE_NAME, box_min_m2=FLOOR_BOX_MIN_FOOTPRINT_M2, box_max_eaves=FLOOR_BOX_MAX_EAVES_M,
    box_type=FLOOR_BOX_TYPE_PATTERN, mall_box_max_eaves=FLOOR_MALL_BOX_MAX_EAVES_M, hyper=_hyper)

# --- 3.3b OSM levels on ALKIS buildings -----------------------------------------------------------------------------------------
# step 04 keeps building:levels on OSM rows only, so 3.1 reaches about 210 OSM-only buildings. The OSM outlines over an ALKIS
# building (storey_feats: its twin, outlines that cover it or lie in it; roof outlines left out) give its storeys where they
# cover at least min_cover of the footprint - their area-weighted levels, rounded - on buildings whose storeys come from the
# height alone (no hall, no structure rule) and whose implied storey height lies in the band (FLOOR_OSM_ALKIS_LEVELS); they
# then count as OSM levels, which no later one-storey rule overrides
_ol = FLOOR_OSM_ALKIS_LEVELS
_olf = storey_feats.reindex(bld['building_id'].to_numpy())
_olf.index = bld.index
_olv = np.floor(_olf['lvn_wmean'] + 0.5)
_olc = (bld['source'].eq('alkis') & ~base['osm'] & (_olf['lvn_cov'].fillna(0.0) >= _ol['min_cover'])
        & _olv.between(1, FLOOR_OSM_LEVELS_MAX)).fillna(False).astype(bool)
_olb = (base['height_used_m'] / _olv).between(_ol['storey_min_m'], _ol['storey_max_m']).fillna(False).astype(bool)
_olh = _olc & base['storeys_source'].eq('height')
_olm = _olh & _why.eq('') & _olb
_ol_out = {'a hall': int((_olc & base['storeys_source'].eq('hall')).sum()), 'a structure rule': int((_olh & _why.ne('')).sum()),
           'an implied storey outside the band': int((_olh & _why.eq('') & ~_olb).sum())}
_ol_n0 = N[_olm].copy()
N[_olm] = _olv[_olm]
base.loc[_olm, 'storeys'] = _olv[_olm]
base.loc[_olm, 'osm'] = True
base.loc[_olm, 'storeys_source'] = 'osm outline levels'
print(f'  ok  OSM levels on ALKIS buildings: {int(_olc.sum()):,} lie under outlines with levels over at least '
      f'{_ol["min_cover"]:.0%} of the footprint; {int(_olm.sum()):,} counted by the height alone with an implied storey of '
      f'{_ol["storey_min_m"]:g}-{_ol["storey_max_m"]:g} m take the OSM levels - {int((_olv[_olm] < _ol_n0).sum()):,} fewer storeys, '
      f'{int((_olv[_olm] > _ol_n0).sum()):,} more, {int((_olv[_olm] == _ol_n0).sum()):,} the same; left out: '
      + ' · '.join(f'{k} {v:,}' for k, v in _ol_out.items()))

# --- 3.3c tall culture halls -----------------------------------------------------------------------------------------------------
# a museum, theatre, event hall, library, castle ... (FLOOR_CULTURE_STOREY: its class and a culture word in its type or name)
# counted at min_storeys or more by the height alone takes storey_height_m per storey - never more storeys than before
_cs = FLOOR_CULTURE_STOREY
_cst = (bld['llm_type'].fillna('') + ' | ' + bld['name'].fillna('').astype(str)).str.contains(_cs['type_pattern'], case=False, regex=True)
_csn = np.maximum(1.0, np.floor(base['height_used_m'] / _cs['storey_height_m'] + 0.5))
_csm = (bld['bosserhof_class'].isin(_cs['classes']) & _cst & base['storeys_source'].eq('height') & _why.eq('') & ~base['osm']
        & (N >= _cs['min_storeys']) & (_csn < N)).fillna(False).astype(bool)
_cs_fa0 = float((bld.loc[_csm, 'area_m2'] * N[_csm]).sum())
_cs_n0 = N[_csm].copy()
N[_csm] = _csn[_csm]
_why[_csm] = 'culture_storey'
print(f'  ok  tall culture halls: {int(_csm.sum()):,} buildings of {", ".join(_cs["classes"])} with a culture word, '
      f'{_cs["min_storeys"]} storeys or more by the height alone, count {_cs["storey_height_m"]:g} m storeys - '
      f'{int((_cs_n0 - _csn[_csm]).sum()):,} storeys fewer, floor area {_cs_fa0 / 1e3:,.1f} -> '
      f'{(bld.loc[_csm, "area_m2"] * N[_csm]).sum() / 1e3:,.1f} thousand m2')

# --- 3.3d OSM levels on the halls of the VW Wolfsburg plant -------------------------------------------------------------------
# a hall in the listed zones under OSM outlines with levels over at least min_cover of its footprint takes their levels - at
# most max_levels, the implied storey in the band of 3.3b, no structure rule, no parking, no reclassified hypermarket, no power
# plant (FLOOR_OSM_HALL_LEVELS); it then counts as OSM levels
_hl = FLOOR_OSM_HALL_LEVELS
_hlx = bld['llm_type'].fillna('').str.contains(_hl['exclude_type'], case=False, regex=True)
_hlz = bld['zone_no'].isin(list(_hl['zones'])).fillna(False).astype(bool)
_hlm = (_hlz & bld['source'].eq('alkis') & ~base['osm'] & base['storeys_source'].eq('hall') & _why.eq('') & ~sf['parking'] & ~_hyper
        & ~_hlx & (_olf['lvn_cov'].fillna(0.0) >= _ol['min_cover']) & _olv.between(1, _hl['max_levels']) & _olb).fillna(False).astype(bool)
_hl_fa0 = float((bld.loc[_hlm, 'area_m2'] * N[_hlm]).sum())
_hl_n0 = N[_hlm].copy()
N[_hlm] = _olv[_hlm]
base.loc[_hlm, 'storeys'] = _olv[_hlm]
base.loc[_hlm, 'osm'] = True
base.loc[_hlm, 'storeys_source'] = 'osm outline levels (hall)'
_hlc = _hlz & base['storeys_source'].eq('hall') & (_olf['lvn_cov'].fillna(0.0) >= _ol['min_cover']) & (_olv >= 2)
print(f'  ok  OSM levels on the halls of zone(s) {", ".join(map(str, _hl["zones"]))}: {int(_hlm.sum()):,} halls take them - '
      f'{int((_olv[_hlm] > _hl_n0).sum()):,} more storeys, floor area {_hl_fa0 / 1e3:,.1f} -> {(bld.loc[_hlm, "area_m2"] * N[_hlm]).sum() / 1e3:,.1f} '
      f'thousand m2; left out with 2+ levels: {int(_hlc.fillna(False).sum()):,} (above {_hl["max_levels"]} levels, outside the band, '
      f'a structure rule or a power plant)')

# --- 3.4 one storey where only the height says more --------------------------------------------------------------------------
# shops and restaurants below 6.5 m or under OSM outlines that all say one level, shops with store evidence, low shop boxes,
# farm halls filed as shops or restaurants and station shops - each where no OSM levels, no structure rule, no dwellings and no
# POI upstairs say otherwise -, fuel-station canopies wherever OSM gives no levels; and the station shop within 15 m of each
# canopy (lib/storey_evidence.py, FLOOR_ONE_STOREY_RULES)
_R1 = FLOOR_ONE_STOREY_RULES
_own1 = own06[['building_id', 'poi_id', 'poi_use', 'name', 'tags']].merge(
    poi_attrs[['poi_id', 'poi_use_tag', 'amenity', 'shop', 'tourism', 'office', 'leisure', 'craft', 'healthcare']], on='poi_id', how='left')
_own1['group'] = floor_rules.poi_groups(_own1, POI_GROUPS, POI_KEY_FALLBACK)
_own1['brand'] = floor_rules.tag_value(_own1['tags'], 'brand')
_fuel_ids = set(_own1.loc[_own1['poi_use'].eq('fuel'), 'building_id'])
storey_canopy = storey_evidence.canopies(bld, _fuel_ids, storey_roof, station_function=_R1['station_function'],
                                         fuel_pattern=_R1['fuel_pattern'], wash_pattern=_R1['wash_pattern'],
                                         canopy_pattern=_R1['canopy_pattern'], roof_min_cover=_R1['canopy_roof_min_cover'])
station_pairs = storey_evidence.station_pairs(
    bld, storey_canopy, fuel_poi_ids=_fuel_ids,
    shop_poi_ids=set(_own1.loc[_own1['group'].isin(_R1['station_shop_groups']) | _own1['poi_use'].eq('fuel'), 'building_id']),
    max_dist=_R1['station_shop_max_dist_m'], station_function=_R1['station_function'], not_shop_head=_R1['station_not_shop_head'],
    fuel_pattern=_R1['fuel_pattern'], extra=_R1['station_extra_pairs'])
_one = storey_evidence.rules(bld, N, _why, base, _upper, _own1, storey_feats, storey_canopy, R=_R1)
# big low office and service boxes (FLOOR_OFFICE_BOX): an office or service class, a
# footprint over 5,000 m2 and eaves of at most 12 m (the mean height where none) - nearly all halls typed 'office' from a generic
# ALKIS function - count one storey where only the height gave more, unless the building's own data say office floors (OSM outlines
# with building:levels over at least half the footprint averaging 1.5 or more, an OSM building=office outline over at least half of
# it, or a type that names several floors); applied after the other one-storey rules, before the sport boxes
_obx = FLOOR_OFFICE_BOX
_obx_e = bld['height_eaves_m'].where(bld['height_eaves_m'].gt(0), bld['mean_height_m'])
_obx_f = storey_feats.reindex(bld['building_id'].to_numpy())
_obx_f.index = bld.index
_obx_floors = (((_obx_f['lvn_cov'].fillna(0.0) >= _obx['osm_min_cover']) & (_obx_f['lvn_wmean'].fillna(0.0) >= _obx['osm_min_levels']))
               | (_obx_f['cover_tag'].isin(_obx['office_tags']) & (_obx_f['cover_share'].fillna(0.0) >= _obx['osm_min_cover']))
               | bld['llm_type'].fillna('').str.contains(_obx['floors_pattern'], case=False, regex=True))
# one storey only with hall evidence - the building is on the list of halls the web and OSM check
# confirmed (FLOOR_OFFICE_BOX['hall_ids'], with the evidence) or its own OSM outline over at least half of the footprint is a hall
# type (FLOOR_OFFICE_BOX['hall_tags']); offices and unknown boxes keep the height storeys. The box is read on the model's class
# (class_06), so a listed hall keeps its one storey after its class fix (CLASS_FIX_BUILDINGS)
_obx_hall = (bld['building_id'].isin(list(_obx['hall_ids']))
             | (_obx_f['cover_tag'].isin(_obx['hall_tags']) & (_obx_f['cover_share'].fillna(0.0) >= _obx['osm_min_cover'])))
_one['office_box'] = (bld['class_06'].isin(_obx['classes']) & (bld['area_m2'] > _obx['min_footprint_m2'])
                      & (_obx_e <= _obx['max_eaves_m']) & base['storeys_source'].eq('height') & _why.eq('') & (N >= 2)
                      & ~bld['RC'].astype(bool) & ~bld['building_id'].isin(_upper) & ~_obx_floors & _obx_hall).fillna(False).astype(bool)
# sport halls and studios, corrected like the industry halls (FLOOR_SPORT_BOX, lib/storey_evidence.py): a sport building counts
# one storey where only
# the height gave more, unless its own data say floors - another non-sport occupant, a mixed-use, shop or office type, a school,
# hotel, care home or storeyed sport use, a type naming floors, OSM levels; clubhouses below 800 m2, boathouses, golf and stadium
# buildings keep theirs (like the industry rule's canteens), unless all their own OSM outlines say one level (osm_one)
_one['sport_box'] = storey_evidence.sport_boxes(bld, N, _why, base, _upper, _own1, storey_feats, R=FLOOR_SPORT_BOX)
_one_n, _done = {}, pd.Series(False, index=bld.index)
for _tag, _m in _one.items():                   # a building takes the first rule that hits it
    _m = _m & ~base['osm'] & (N >= 2) & ~_done
    N[_m] = 1.0
    _why[_m] = _why[_m].where(_why[_m] == '', _why[_m] + '+') + _tag
    _done |= _m
    _one_n[_tag] = int(_m.sum())
_cid = set(bld.loc[storey_canopy, 'building_id'])
if (N[_done] != 1).any() or (_done & base['osm']).any():
    raise AssertionError('a one-storey rule left a building above one storey or overrode OSM levels')
if (not set(station_pairs) <= _cid or set(station_pairs.values()) & _cid or not set(station_pairs.values()) <= set(bld['building_id'])
        or not set(_R1['canopy_no_unit_ids']) <= _cid):
    raise AssertionError('a station pair whose canopy is no canopy or whose shop is one or no building, or a listed canopy that is none')
print(f'  ok  one storey where only the height gave more (with the office and sport boxes): ' + ' · '.join(f'{k} {v:,}' for k, v in _one_n.items())
      + f' buildings; {len(_cid)} fuel-station canopies, {len(station_pairs)} of them with a station shop '
      f'({len(set(station_pairs) - set(_R1["station_extra_pairs"]))} within {_R1["station_shop_max_dist_m"]:.0f} m, '
      f'{len(_R1["station_extra_pairs"])} by the list)')
N, _why = floor_rules.retail_storeys(bld, base, N, _why, classes=FLOOR_RETAIL_TALL_CLASSES, storey_height=FLOOR_RETAIL_STOREY_HEIGHT_M)
N, _why = floor_rules.mall_storeys(bld, N, _why, bld['RC'], sf, _own_levels, mall_class='shopping centers',
                                   outlet_class='factory outlet centers', level_min=FLOOR_MALL_LEVELS[0], level_max=FLOOR_MALL_LEVELS[1],
                                   min_pois=FLOOR_MALL_MIN_LEVEL_POIS, mall_max=FLOOR_MALL_MAX_STOREYS,
                                   small_max=FLOOR_MALL_SMALL_MAX_STOREYS, small_m2=FLOOR_MALL_SMALL_FOOTPRINT_M2,
                                   outlet_max=FLOOR_OUTLET_MAX_STOREYS)
bld['storeys_08b'] = base['storeys'].astype(int)
bld['storeys'] = N.astype(int)
_hs = bld['building_id'].isin(list(FLOOR_OFFICE_BOX['hall_ids'])) & ~base['osm'] & bld['storeys'].ne(1)
if _hs.any():                                   # a checked hall that does not count one storey
    print(f"  !!  FLOOR_OFFICE_BOX hall_ids above one storey without OSM levels (re-check the list): {list(bld.loc[_hs, 'building_id'])}")
bld['storey_rule'] = base['storeys_source'] + np.where(_why != '', '+' + _why, '')
bld['floor_area_m2'] = bld['area_m2'] * bld['storeys']
bld['height_used_m'] = base['height_used_m']
for _c in ('worship', 'fire', 'sport_hall', 'parking'):
    bld[_c] = sf[_c]

# --- checks ------------------------------------------------------------------------------------------
_e7 = base['storeys_source'].eq('osm estimate')               # the storey estimate of 3.2b: OSM-only rows, no OSM levels, no hall
if (bld.loc[_e7, 'source'] != 'osm').any() or base.loc[_e7, 'osm'].any() or (base.loc[_e7, 'hall'] != '').any():
    raise AssertionError('a storey estimate on an ALKIS building, over OSM levels or on a hall')
if (N != np.floor(N)).any() or (bld['storeys'] < 1).any() or (bld['floor_area_m2'] < bld['area_m2'] - 1e-9).any():
    raise AssertionError('storeys that are no whole number of at least 1, or a floor area below the footprint')
_rule_tags = _why.str.split('+').explode()
_rule_tags = _rule_tags[_rule_tags != '']
_unhall = ('storeyed_company_use', 'canteen', 'outlet_not_hall', 'brand_not_hall', 'ikea', 'furniture_house', 'mall_mapped_levels')
_halls = (base['hall'] != '') & ~base['osm'] & ~_why.str.contains('|'.join(_unhall))
if (bld.loc[_halls, 'storeys'] != 1).any():
    raise AssertionError('a hall without OSM levels and without a rule that lifts it counts more than one storey')
if (bld.loc[bld['parking'], 'storeys'] != 1).any() or (bld.loc[_hyper & ~base['osm'], 'storeys'] != 1).any():
    raise AssertionError('a parking building or a reclassified hypermarket above one storey')
_mall = bld['bosserhof_class'].eq('shopping centers') & ~_why.str.contains('mall_mapped_levels')
if (bld.loc[_mall, 'storeys'] > FLOOR_MALL_MAX_STOREYS).any() or (bld.loc[bld['bosserhof_class'].eq('factory outlet centers'), 'storeys'] > FLOOR_OUTLET_MAX_STOREYS).any():
    raise AssertionError('a mall above its storey cap')
_fh = sf['furniture_house'] & ~base['osm']
if (bld.loc[_fh, 'storeys'] > FLOOR_FURNITURE_HOUSE_MAX_STOREYS).any():
    raise AssertionError('a furniture house above its cap')
print(f'  ok  {len(bld):,} buildings, {bld["floor_area_m2"].sum() / 1e6:,.2f} km2 of floor area after the structure rules '
      f'(height rule: {(bld["area_m2"] * bld["storeys_08b"]).sum() / 1e6:,.2f} km2); storeys are whole numbers of at least 1, halls one storey '
      'unless a rule lifts them, parking and the hypermarkets one storey, malls and outlets within their caps')
_t = pd.DataFrame({'buildings': _rule_tags.value_counts()})
_ch = (bld['storeys'] != bld['storeys_08b'])
_t['storeys changed'] = [int((_why.str.split('+').map(lambda l: k in l) & _ch).sum()) for k in _t.index]
_t['floor area height -> rules (1,000 m2)'] = [
    f'{(bld.loc[_m, "area_m2"] * bld.loc[_m, "storeys_08b"]).sum() / 1e3:,.1f} -> {bld.loc[_m, "floor_area_m2"].sum() / 1e3:,.1f}'
    for _m in (_why.str.split('+').map(lambda l, k=k: k in l) for k in _t.index)]
print('  ..  the structure rules (a building can carry several; a later rule overrides an earlier one):')
print(_t.to_string())
print(f'  ..  decisions: worship {int(sf["worship"].sum()):,} buildings, fire and rescue stations {int(sf["fire"].sum()):,}, sports halls and pools '
      f'{int(sf["sport_hall"].sum()):,}, single-level boxes {int(sf["box"].sum()):,} + malls {int(sf["mall_box"].sum()):,}, '
      f'furniture houses {int(sf["furniture_house"].sum()):,} ({int((sf["furniture_house"] & _ch).sum())} change), '
      f'malls {int(bld["bosserhof_class"].eq("shopping centers").sum()):,} ({int((bld["bosserhof_class"].eq("shopping centers") & _ch).sum())} change), '
      f'hypermarkets {int(_hyper.sum())}, outlets {int(_outlet.sum())}')
print('  ..  storeys after the rules:', bld['storeys'].clip(upper=8).value_counts().sort_index().rename(lambda s: f'{s}+' if s == 8 else str(s)).to_dict())
_x = bld.loc[_ch].assign(change=lambda t: t['floor_area_m2'] - t['area_m2'] * t['storeys_08b'])
_x = pd.concat([_x.loc[_x.groupby(_x['storey_rule'].str.split('+').str[-1])['change'].apply(lambda s: s.abs().idxmax())]])
print('  ..  the largest change per last rule:')
with pd.option_context('display.max_colwidth', 60):
    print(_x[['building_id', 'name', 'bosserhof_class', 'area_m2', 'storeys_08b', 'storeys', 'storey_rule', 'llm_type']]
          .sort_values('storey_rule').round({'area_m2': 0}).to_string(index=False))

  ok  the class fix (with the 23 halls among the office boxes): 27 buildings take the class of what they are - Yards, depots, storage areas, construction yards 10 · Industrial operations / Production 9 · highly productive industries / machine / material or space intensive 3 · car dealerships 2 · wholesale 2 · furniture stores 1 (from normal office 15 · Services 7 · retail (small-scale) 3 · restaurants / gastronomy 1 · business-oriented services 1)
  ok  reclassified: 8 shopping centres -> hypermarkets / superstores (factor 1.2), 8 -> factory outlet centers (factor 2.15); 43 buildings change class
  ok  the height storeys: height from eaves 21,533 · mean (eaves above it) 16,653 · mean (no eaves) 1,600; storeys from height 28,265 · hall 11,309 · osm levels 212; 48.79 km2 of floor area


  ok  residential context on 10,935 buildings: ALKIS residential code 11,322, of which taken back by an office-building head 374, a non-residential OSM twin 416, an unconfirmed low box 209; added by an OSM residential twin 759 and a residential noun in the type 37; stress-test fixes: a house twin on a non-residential code 295, Kitas by type only 67 changed
  ok  OSM-only buildings without levels or a height (step 04's 2.0 m): 671 that are no hall, 304 of them of a kind that takes an estimate, 206 above one storey: 2 storeys 191 · 3 storeys 15; the median from class and size 198 · size 8


  ok  OSM levels on ALKIS buildings: 6,003 lie under outlines with levels over at least 50% of the footprint; 4,373 counted by the height alone with an implied storey of 2.5-6 m take the OSM levels - 1,458 fewer storeys, 304 more, 2,611 the same; left out: a hall 762 · a structure rule 115 · an implied storey outside the band 753
  ok  tall culture halls: 132 buildings of entertainment, culture with a culture word, 3 storeys or more by the height alone, count 5 m storeys - 162 storeys fewer, floor area 460.6 -> 317.9 thousand m2
  ok  OSM levels on the halls of zone(s) 310399999: 55 halls take them - 44 more storeys, floor area 869.9 -> 2,847.5 thousand m2; left out with 2+ levels: 25 (above 4 levels, outside the band, a structure rule or a power plant)


  ok  one storey where only the height gave more (with the office and sport boxes): canopy 22 · osm_levels_1 45 · low_shop 407 · box_evidence 24 · box_low 3 · hall_tag_code 19 · station_shop 12 · office_box 20 · sport_box 106 buildings; 77 fuel-station canopies, 55 of them with a station shop (51 within 15 m, 4 by the list)
  ok  39,786 buildings, 47.84 km2 of floor area after the structure rules (height rule: 50.17 km2); storeys are whole numbers of at least 1, halls one storey unless a rule lifts them, parking and the hypermarkets one storey, malls and outlets within their caps


  ..  the structure rules (a building can carry several; a later rule overrides an earlier one):
                      buildings  storeys changed floor area height -> rules (1,000 m2)
worship                     767              767                        878.5 -> 278.6
low_shop                    407              407                        455.2 -> 227.6
hall_residential            340              273                        253.9 -> 107.1
fire_station                239              239                         199.5 -> 93.4
culture_storey              132              132                        460.6 -> 317.9
sport_box                   106              106                        253.8 -> 125.7
hall_before_office          103               58                         190.9 -> 83.8
single_level_box             64               64                        383.3 -> 185.7
sports_hall                  55               55                         211.7 -> 90.9
osm_levels_1                 45  

## 4. The units and their occupied floor area

| unit | which | occupied floor area O |
|---|---|---|
| **building** | a building without asked POIs, with its step-06 labels | `A × min(N, storeys of its class rule)` — refined by its only POI's group, level tag and chain |
| **POI** | an asked POI in a shared building, with its own labels | `min(s × FA, family storeys × A)`; GROUND units together ≤ A; ALL takes the rest |
| **rest** | a building label none of its asked POIs carries | `A × min(N, storeys of its class rule)`, split over the building's labels |
| **remainder** | the floor no unit occupies, in a building without dwellings (none in the 7 listed malls) | that floor, Workers only - factor 2.9 in shop and office buildings, else the building's own |

`s` is the POI's share re-split over the asked POIs once the landuse pairs are
removed; a vacant pair, a vacant POI, a holiday flat or a dormitory keeps its
floor as an empty slot (a vacant POI the floor it would occupy if active). Several POIs of one facility share the building's
sole-occupant floor. A POI mapped as a point is capped at the p90 of its
group's mapped polygons; the floor this frees goes to the ALL units that are
no point, else it stays free. **Small places stay small**: a building filed as
small-scale retail or a restaurant, its rest unit and the points of these
groups — also as one facility or main occupant — occupy at most the typical
size of their kind (748 / 438 m²); the cut floor carries nothing, food stores
keep their size; one whose own data name a big-box chain takes the big-box
cap. **The sizes of POI units** (4.4d / 4.5): a POI
mapped as a point counts at least its group's median polygon in its
building's shares (at most an equal share and the footprint; not a point that
repeats a polygon of its building by name or use), the share it gains coming
from the other occupants by their shares; tenant points outside the malls take
the point cap and are never the main occupant; in a mall the tenant points'
shares follow their group's typical size; a pharmacy occupies at most 307 m².
By format (`UNIT_SIZE_FORMATS`): the easyApotheke format
up to 400 m², a chain filed small at most its format's size, counters and
kiosks at least the mapped kiosk median, a mall tenant of a listed chain at
least its published size (in its building's shares; its weight in the malls
with a published floor). **The floor of the malls** (4.5b;
`MALL_RETAIL_FLOOR`): the retail units of 7 malls occupy at most their
published retail floor, one factor for all of them (so a unit can end below
the sizes above); a mall's own outline POI keeps only the floor its tenants
leave; the non-retail tenants keep theirs, the rest of the floor and its
remainder carry nothing.
A POI unit takes the Workers factor of its own
POI-level class; an industry POI that only repeats the building's OSM outline
takes the building's, and an office building filed in an industry class takes
the normal-office factor on its own units (4.0; not a building of the class fix). Non-occupants (amenity features asked as POIs, by tag) are
dropped and their host keeps the floor. Then the **rules on single
categories**: churches, volunteer fire stations, sports halls and clubhouses
lose their Workers (a volunteer fire station labelled only work loses its
unit) — also fitness and wellness halls, worship buildings by
the storey rules' worship flag and worship POIs in any building, city fire
stations whose own names or operators say volunteer (not a zone's only Workers
carrier), and parking garages and decks; the no-evidence residential buildings
keep one storey with Workers only — not those whose only sites are landuse
areas (`NO_EVIDENCE_LANDUSE`); holiday flats and dormitories give no unit, nor do the
buildings that no longer stand or the places known to be closed (4.2; `CLOSED_BUILDINGS`). The OSM signs of non-use count as the
tags say them (`VACANCY_OSM_SIGNS`: under works or gone, a lifecycle prefix without an active tag, closed opening hours, a past end date, a permanent note). Every such place is counted;
the vacant places are listed. **Who carries School, University and
Kindergarten**: real schools typed after a tenant get
the school label and vocational schools labelled university become School
before the units are built (4.4b); on the units, a place whose own data name
another use, a non-teaching part of a campus or a place without any evidence
loses the education label, keeping its `k_red` so that no other category
moves (4.6) — each with its evidence listed; the web-checked cases follow
`EDU_CARRIER_LISTS`. **The labels of steps 06 and 07
reconciled**: by its own OSM use and data, an eating
place carries leisure instead of shopping, a pharmacy retail_daily, a fuel
station retail_daily unless its data show no shop (then errands — once more
after 4.7, where a canopy passed retail_daily on), a beverage market, kiosk,
newsagent, tobacco or lottery shop retail_daily instead of retail_non_daily,
and a leisure venue labelled sports leisure too — before the units are built
(4.4c), so a label taken off a POI does not come back as a rest unit; on the
units, a service loses retail_non_daily and a rest label goes where an asked
POI of its building is its evident source and step 07 labelled that POI
otherwise (4.6b). Every change is listed with its rule; a dropped label's share
goes to the unit's other labels. **Fuel stations** (4.7): a canopy forms no remainder; a canopy with a station shop within
15 m (or on the list) gives the shop its labels — a shop without POIs then
counts as a place with evidence where the canopy had one — and keeps no unit
itself (a shop without a unit carries nothing); a canopy without a station
shop occupies at most the fuel cap, and a truck stop with diesel only counts as
without a shop. **Staff catering** (4.8): works canteens,
staff cafés and snack bars on company sites, Mensas and hospital cafeterias
lose daily and non-daily shopping and leisure, and the rest unit of their
building daily shopping and leisure; their guests are counted as Workers,
School or University (`lib/staff_catering.py`). Staff shops and kiosks in a
works building on a named company site lose these labels too
(`staff_shops`). Checked: every unit occupies a positive floor area.

In [5]:
def _labels(s):                                 # the labels of a unit, in the catalogue order
    have = set(s.split(';'))
    return tuple(l for l in LLM_ACTIVITY_LABELS if l in have)
def _ks(labels):                                # (k_red, k_all): its redistributed and all non-work labels
    return sum(l in RED_LABELS for l in labels), sum(l != 'work' for l in labels)

bld['labels'] = bld['mid_labels'].map(_labels)
pois['labels'] = pois['poi_mid_labels'].map(_labels)
if not (bld['labels'].map(lambda t: 'work' in t).all() and pois['labels'].map(lambda t: 'work' in t).all()):
    raise AssertionError('a building or POI without work - the work rule of step 06/07 was not applied')
_b = bld.set_index('building_id')

# --- 4.0 the Workers factor of office buildings in an industry class -----------------------------------------------------------
# a building of an industry, yard or transport class whose type head (before the first location word) names a whole office
# building - the residential context's office pattern - and no hall, works, production or storage word takes the normal-office
# factor on its own units (else the Salzgitter AG headquarters is weighted as a plant)
_of_pre = bld['head'].str.split(FLOOR_LOCATION_SPLIT, n=1, regex=True).str[0]
# a building of the class fix keeps the factor of its fixed class (its type may
# still name an 'office building')
_of_m = (bld['bosserhof_class'].isin(FLOOR_INDUSTRY_CLASSES) & ~bld['building_id'].isin(list(CLASS_FIX_BUILDINGS)) & _of_pre.str.contains(RC_OFFICE_BUILDING_PATTERN, case=False, regex=True)
         & ~_of_pre.str.contains(FLOOR_TYPE_PATTERNS['town_hall'], case=False, regex=True)
         & ~_of_pre.str.contains(OFFICE_IN_INDUSTRY_EXCLUDE_PATTERN, case=False, regex=True))
office_factor_ids = sorted(bld.loc[_of_m, 'building_id'])
bld.loc[_of_m, 'workers_factor'] = REDIST_REMAINDER_WORKERS_FACTOR
_b = bld.set_index('building_id')
print(f'  ok  office buildings in an industry class take the normal-office factor {REDIST_REMAINDER_WORKERS_FACTOR}: {len(office_factor_ids)} '
      '(' + ', '.join(f'{b_} {str(_b.at[b_, "name"]) if isinstance(_b.at[b_, "name"], str) else ""}'.strip() for b_ in office_factor_ids[:8])
      + (' ...' if len(office_factor_ids) > 8 else '') + ')')

# --- 4.1 POI groups, POI-level classes and factors ------------------------------------------------------
_pa = poi_attrs.set_index('poi_id')
for _df in (pois, own06):
    for _c in ('poi_use_tag', 'geom_kind', 'amenity', 'shop', 'tourism', 'office', 'leisure', 'craft', 'healthcare', 'tags'):
        _df[_c] = _df['poi_id'].map(_pa[_c])
    _df['poi_area_m2'] = _df['poi_id'].map(_pa['area_m2'])
    _df['group'] = floor_rules.poi_groups(_df, POI_GROUPS, POI_KEY_FALLBACK)
    _df['poi_class'] = floor_rules.poi_classes(_df, _df['group'], group_class=POI_GROUP_CLASS, class_by_use=POI_CLASS_BY_USE,
                                               food_class=POI_FOOD_STORE_CLASS, discount_class=POI_DISCOUNT_CLASS,
                                               discount_pattern=POI_DISCOUNT_PATTERN, hyper_class=POI_HYPER_CLASS,
                                               hyper_pattern=POI_HYPER_PATTERN)
    _df['levels'] = floor_rules.tag_value(_df['tags'], 'level').map(floor_rules.parse_levels)
    _ch = floor_rules.chain_names(_df['name'], MULTI_LEVEL_CHAINS)
    _df['chain'] = _ch.where(_ch != '', floor_rules.chain_names(floor_rules.tag_value(_df['tags'], 'brand'), MULTI_LEVEL_CHAINS))
# the POIs that stand for their building take its factor: an industry POI that only repeats the building's OSM
# outline (building=industrial / hall / warehouse ...), and the outline and works POIs of an office building of 4.0
_BF_POIS = (set(pois.loc[pois['group'].eq('industry') & pois['poi_use_tag'].eq('building'), 'poi_id'])
            | set(pois.loc[pois['building_id'].isin(office_factor_ids) & pois['group'].eq('industry')
                           & pois['poi_use_tag'].isin(['building', 'man_made']), 'poi_id']))
pois['information'] = floor_rules.tag_value(pois['tags'], 'information')
pois['workers_factor'] = [factor(c) if isinstance(c, str) and p_ not in _BF_POIS else fb
                          for p_, c, fb in zip(pois['poi_id'], pois['poi_class'], pois['building_id'].map(_b['workers_factor']))]
print(f'  ok  POI groups for {len(pois):,} asked POIs (' + ' · '.join(f'{k} {v:,}' for k, v in pois['group'].value_counts().head(8).items())
      + ' ...); a POI-level class, and so its own Workers factor, for '
      f'{int(pois["poi_class"].notna().sum()):,} of them, the building\'s factor for the {int(pois["poi_class"].isna().sum()):,} generic / unmapped')
_caps = floor_rules.point_size_caps(own06, _b['area_m2'], groups=list(POINT_SIZE_CAP_M2), quantile=POINT_SIZE_CAP_QUANTILE,
                                    outline_share=POINT_SIZE_CAP_OUTLINE_SHARE)
_dev = (_caps['cap'] - pd.Series(POINT_SIZE_CAP_M2)).abs()
if _dev.isna().any() or (_dev > 0.5).any():
    raise AssertionError(f'POINT_SIZE_CAP_M2 is not the p{POINT_SIZE_CAP_QUANTILE * 100:.0f} of the mapped polygons: {_caps.round(1).to_dict("index")}')
_all = floor_rules.point_size_caps(own06, _b['area_m2'], groups=sorted(set(own06['group']) - POI_NO_UNIT_GROUPS - {'vacant'}),
                                    quantile=POINT_SIZE_CAP_QUANTILE, outline_share=POINT_SIZE_CAP_OUTLINE_SHARE)
_miss = sorted(set(_all.index[_all['n'] >= POINT_SIZE_CAP_MIN_POLYGONS]) - set(POINT_SIZE_CAP_M2))
_n_all, _cap_all = floor_rules.point_size_cap_all(own06, _b['area_m2'], quantile=POINT_SIZE_CAP_QUANTILE, outline_share=POINT_SIZE_CAP_OUTLINE_SHARE)
if _miss or abs(_cap_all - POINT_SIZE_CAP_DEFAULT_M2) > 0.5:
    raise AssertionError(f'groups with at least {POINT_SIZE_CAP_MIN_POLYGONS} polygons but no cap: {_miss}; '
                         f'the default cap is {_cap_all:,.1f} m2 ({_n_all} polygons), not {POINT_SIZE_CAP_DEFAULT_M2}')
print(f'  ok  the point size caps are the p{POINT_SIZE_CAP_QUANTILE * 100:.0f} of the mapped polygons of each group: '
      + ' · '.join(f'{g} {r.cap:,.0f} m2 ({r.n:.0f})' for g, r in _caps.iterrows())
      + f'; every other group {POINT_SIZE_CAP_DEFAULT_M2:,.0f} m2 (all {_n_all:,} polygons)')

# --- 4.2 vacancy: a clear sign that nobody is there (lib/vacancy.py) ------------------------------------
bld['vacancy'], bld['vacancy_sign'], pois['vacancy'] = '', '', ''
_review = pd.DataFrame(columns=['building_id', 'cover', 'active_pois', 'signs'])
# the OSM signs of non-use as the tags say them: a building under works or gone,
# a lifecycle prefix without an active tag, opening_hours closed, a past end_date, a permanent note, shop=no
# (VACANCY_OSM_SIGNS); a building only these leave empty stays where it lies on the named site of an active occupant
if REDIST_EXCLUDE_VACANT:
    bld['vacancy'], bld['vacancy_sign'], pois['vacancy'], _review = vacancy.flag(
        bld, pois, own06, poi_attrs, polygons, uses=VACANCY_POI_USES, name_pattern=VACANCY_NAME_PATTERN,
        poi_tag_pattern=VACANCY_POI_TAG_PATTERN, text_pattern=VACANCY_TEXT_PATTERN,
        partial_pattern=VACANCY_PARTIAL_PATTERN, min_cover=VACANCY_POLYGON_MIN_COVER, texts=poi_osm_signs,
        sites=pairs06[pairs06['how'].eq('site')] if VACANCY_OSM_SIGNS['keep_named_site'] else None)
# buildings that no longer stand: ALKIS still holds them, while OSM
# shows a construction or brownfield site or other buildings there (GONE_BUILDINGS) - they give no unit, like a vacant place
_gone_m = bld['building_id'].isin(GONE_BUILDINGS) & bld['vacancy'].eq('')
bld.loc[_gone_m, 'vacancy'] = 'gone'
print(f'  ok  buildings that no longer stand: {int(_gone_m.sum())} of the {len(GONE_BUILDINGS)} listed give no unit '
      f'({bld.loc[_gone_m, "floor_area_m2"].sum() / 1e3:,.1f} thousand m2 of floor area)')
# places known to be closed: a building on CLOSED_BUILDINGS - closed on CLOSED_AS_OF by a dated source - gives no unit, like a vacant place, and so do the asked POIs in it, except those its entry keeps open ('keep_pois': an
# active place in an attached structure). A listed place an OSM sign or GONE_BUILDINGS already left out keeps that reason (the list
# is the backstop). The zone totals still hold: the zone's other carriers take the share, or the ring fallback where the place was
# the zone's last carrier
_cl_miss = sorted(set(CLOSED_BUILDINGS) - set(bld['building_id']))
if _cl_miss:
    raise AssertionError(f'CLOSED_BUILDINGS: not in the data {_cl_miss}')
_cl_kb = {p_: b_ for b_, v_ in CLOSED_BUILDINGS.items() for p_ in v_.get('keep_pois', ())}
_cl_kbad = [p_ for p_, b_ in _cl_kb.items() if not (pois['poi_id'].eq(p_) & pois['building_id'].eq(b_)).any()]
if _cl_kbad:
    raise AssertionError(f'CLOSED_BUILDINGS keep_pois that are no asked POI of their building: {_cl_kbad}')
_cl_m = bld['building_id'].isin(list(CLOSED_BUILDINGS)) & bld['vacancy'].eq('')
bld.loc[_cl_m, 'vacancy'] = 'closed'
bld.loc[_cl_m, 'vacancy_sign'] = [f"{CLOSED_BUILDINGS[b_]['what']} (closed since {CLOSED_BUILDINGS[b_]['since']}"
                                  + (f", reopening planned {CLOSED_BUILDINGS[b_]['until']})" if CLOSED_BUILDINGS[b_]['until'] else ')')
                                  for b_ in bld.loc[_cl_m, 'building_id']]
_cl_keep = {p_ for v_ in CLOSED_BUILDINGS.values() for p_ in v_.get('keep_pois', ())}
_cl_pm = pois['building_id'].isin(list(CLOSED_BUILDINGS)) & pois['vacancy'].eq('') & ~pois['poi_id'].isin(_cl_keep)
pois.loc[_cl_pm, 'vacancy'] = 'closed'
_cl_other = sorted(set(CLOSED_BUILDINGS) & set(bld.loc[bld['vacancy'].ne('') & ~_cl_m, 'building_id']))
_cl_due = sorted(b_ for b_, v_ in CLOSED_BUILDINGS.items() if v_.get('until') and str(v_['until'])[:4] <= CLOSED_AS_OF[:4])
print(f'  ok  places known to be closed on {CLOSED_AS_OF} (CLOSED_BUILDINGS): {int(_cl_m.sum())} of the {len(CLOSED_BUILDINGS)} listed '
      f'give no unit ({bld.loc[_cl_m, "floor_area_m2"].sum() / 1e3:,.1f} thousand m2 of floor area), and {int(_cl_pm.sum())} asked POI(s) '
      'in them' + (f'; kept open: {", ".join(sorted(_cl_keep))}' if _cl_keep else '')
      + (f'; left out already by another sign: {", ".join(_cl_other)}' if _cl_other else '')
      + (f'; the planned reopening year has come, re-check: {", ".join(_cl_due)}' if _cl_due else ''))
_vac_b = set(bld.loc[bld['vacancy'] != '', 'building_id'])
_vac_p = set(pois.loc[pois['vacancy'] != '', 'poi_id'])
if REDIST_EXCLUDE_VACANT:
    _vb = bld[bld['vacancy'] != '']
    print(f"  ..  vacant places left out: {len(_vb):,} buildings ({_vb['vacancy'].value_counts().to_dict()}; "
          f"{_vb['floor_area_m2'].sum():,.0f} m2 of floor) and {len(_vac_p):,} asked POI(s) "
          f"({pois.loc[pois['vacancy'] != '', 'vacancy'].value_counts().to_dict()}):")
    with pd.option_context('display.max_colwidth', 120):
        print(_vb[['building_id', 'vacancy', 'name', 'vacancy_sign']].sort_values(['vacancy', 'building_id']).to_string(index=False))
        print(pois.loc[pois['vacancy'] != '', ['poi_id', 'building_id', 'name', 'vacancy', 'poi_type']].to_string(index=False))
    if len(_review):
        _r = _review.merge(bld[['building_id', 'name', 'llm_type']], on='building_id', how='left')
        _kp = _r['active_pois'].eq(0)               # kept on a named site (flag(): those rows have no active POI)
        print(f'  ..  for review: {len(_r):,} buildings keep their values although a sign of non-use applies: {int((~_kp).sum()):,} lie '
              'at least half under an OSM polygon with such a sign but an active POI is inside; '
              f'{int(_kp.sum()):,} would be left empty only by a sign of non-use (on an OSM polygon or on all their own POIs) but '
              'lie on the named site of an active occupant:')
        with pd.option_context('display.max_colwidth', 70):
            print(_r.sort_values('cover', ascending=False).round({'cover': 2}).to_string(index=False))

# --- 4.3 the shares: landuse pairs re-split, vacant pairs kept as slots, non-occupants dropped --------------
_sign = vacancy.poi_signs(poi_attrs, VACANCY_POI_USES, VACANCY_NAME_PATTERN, VACANCY_POI_TAG_PATTERN, texts=poi_osm_signs)
_na = own06[~own06['poi_id'].isin(pois['poi_id']) & own06['building_id'].isin(pois['building_id'])]
_is_lu = _na['poi_use_tag'].eq('landuse') & _na['poi_use'].isin(REDIST_LANDUSE_USES)
_landuse = _na[_is_lu]
_slot_na = _na[~_is_lu]                         # vacant pairs (and any other non-asked pair): an empty slot each
_slot_shares = _slot_na.groupby('building_id')['share_in_building'].agg(list)
if (_slot_na['poi_id'].map(_sign).fillna('') == '').any():
    print(f"  ..  {int((_slot_na['poi_id'].map(_sign).fillna('') == '').sum())} non-asked pair(s) neither landuse nor vacant keep an empty slot too")
pois['non_occupant'] = False
if REDIST_DROP_NON_OCCUPANTS:
    pois['non_occupant'] = floor_rules.non_occupants(pois, tags=NON_OCCUPANT_TAGS, info_staffed=NON_OCCUPANT_INFO_STAFFED_PATTERN,
                                                     keep_keys=NON_OCCUPANT_KEEP_KEYS) & pois['vacancy'].eq('')
pois['slot'] = np.select([pois['vacancy'] != '', pois['group'].isin(POI_NO_UNIT_GROUPS), pois['poi_use'].isin(POI_DORM_USES)],
                         ['vacant', 'holiday flat', 'dormitory'], default='')
_kept = pois[~pois['non_occupant']]
_shared_b = set(_kept['building_id'])            # active POIs in a vacant building keep their units
_to_bu = set(pois['building_id']) - set(_kept['building_id']) - _vac_b
print(f'  ok  landuse pairs re-split away: {len(_landuse):,} pairs on {_landuse["building_id"].nunique():,} buildings held '
      f'{(_landuse["share_in_building"] * _landuse["building_id"].map(_b["floor_area_m2"])).sum() / 1e3:,.0f} thousand m2 of floor; '
      f'{len(_slot_na):,} vacant pairs on {_slot_na["building_id"].nunique():,} buildings keep an empty GROUND slot')
_no = pois[pois['non_occupant']]
_cand = pd.concat([pois[k].isin(v) for k, v in NON_OCCUPANT_TAGS.items()], axis=1).any(axis=1) | pois['tourism'].eq('information')
print(f'  ok  non-occupants dropped (by tag): {len(_no):,} asked POIs on {_no["building_id"].nunique():,} buildings ('
      + ' · '.join(f'{k} {v}' for k, v in _no['poi_use'].value_counts().items()) + f'); {len(_to_bu):,} buildings are left without an asked POI '
      f'and become building units; kept although tagged so: {int((_cand & ~pois["non_occupant"]).sum())} (a staffed tourist office, an occupant '
      'key such as shop=chemist, or vacant)')

# the no-evidence residential rule and landuse areas: a building whose only
# sites are commercial, retail or industrial landuse areas counts as without a site, as for the quarter weight - if it has
# dwellings by the residential context and is no plain home (NO_EVIDENCE_LANDUSE)
_ne_landuse = set()
if NO_EVIDENCE_LANDUSE['landuse_is_no_site']:
    _ne_site_ev = set(pairs06.loc[pairs06['how'].eq('site') & ~pairs06['poi_use'].isin(REDIST_LANDUSE_USES), 'building_id'])
    _ne_name0 = bld['name'].isna() | bld['name'].fillna('').astype(str).str.strip().isin(['', '-'])
    _ne_lu = (bld['function'].fillna('').str.startswith(NO_EVIDENCE_FUNCTION_PREFIX) & bld['bosserhof_class'].isin(NO_EVIDENCE_CLASSES)
              & bld['n_pois'].eq(0) & bld['n_sites'].gt(0) & ~bld['building_id'].isin(_ne_site_ev) & _ne_name0
              & bld['osm_twin_name'].fillna('').astype(str).str.strip().eq(''))
    if NO_EVIDENCE_LANDUSE['keep_plain_homes']:      # plain homes keep their one storey with their labels
        _ne_lu &= ~(tf['pure_home'] & ~tf['care'] & ~tf['pure_home_excl'] & bld['bosserhof_class'].isin(list(UNIT_HOME_STOREYS)))
    if NO_EVIDENCE_LANDUSE['needs_residential_context']:   # only with dwellings by the residential context (section 3.2)
        _ne_lu &= bld['RC'].astype(bool)
    _ne_landuse = set(bld.loc[_ne_lu, 'building_id'])
print(f'  ok  the no-evidence residential rule also takes {len(_ne_landuse):,} buildings whose only sites are landuse areas')

# --- 4.4 the building rules: the places the single-category rules apply to ----------------------------------
_sets = {
    'holiday_flat':   floor_rules.holiday_flat_ids(bld, pairs06, hotel_class=HOLIDAY_FLAT_CLASS, function_prefix=HOLIDAY_FLAT_FUNCTION_PREFIX,
                                                   uses=HOLIDAY_FLAT_USES),
    'dormitory':      set(bld.loc[tf['dorm'], 'building_id']),
    'no_evidence':    floor_rules.no_evidence_ids(bld, function_prefix=NO_EVIDENCE_FUNCTION_PREFIX, classes=NO_EVIDENCE_CLASSES)
                      | _ne_landuse,
    'church':         floor_rules.church_ids(bld, codes=CHURCH_ALKIS_CODES)      # and the storey rules' worship buildings
                      | set(bld.loc[bld['worship'] & ~bld['head'].str.contains(CHURCH_FLAG_EXCLUDE_PATTERN, case=False, regex=True),
                                    'building_id']),
    'volunteer_fire': floor_rules.volunteer_fire_ids(bld, station=VOLUNTEER_FIRE_PATTERN, exclude=VOLUNTEER_FIRE_EXCLUDE_PATTERN,
                                                     exclude_head=VOLUNTEER_FIRE_EXCLUDE_HEAD_PATTERN, volunteer=VOLUNTEER_FIRE_CITY_PATTERN,
                                                     city_ags=VOLUNTEER_FIRE_CITY_AGS),
    'sports_hall':    floor_rules.sports_hall_ids(bld, codes=SPORTS_HALL_ALKIS_CODES, type_pattern=SPORTS_HALL_TYPE_PATTERN,
                                                  classes=SPORTS_HALL_CLASSES, keep_pattern=SPORTS_HALL_KEEP_PATTERN, keep_ids=SPORTS_HALL_KEEP_IDS),
}
# the Workers-0 rules reach the places their own tests describe:
# a station in the three cities counts as volunteer where the volunteer word (or FF) stands in its name, full type, POI names,
# OSM twin name or its own POIs' names and operators, not only in the head - except a station that is its zone's only Workers
# carrier (BS Stiddien: the last-carrier guard would put its Workers on the church instead)
_vf_pa = poi_attrs.set_index('poi_id')
_vf_own = (own06.assign(_t=own06['name'].fillna('') + ' ' + own06['poi_id'].map(_vf_pa['operator']).fillna(''))
           .groupby('building_id')['_t'].agg(' | '.join))
_vf_txt = (bld['name'].fillna('') + ' | ' + bld['llm_type'].fillna('') + ' | ' + bld['poi_names'].fillna('').astype(str) + ' | '
           + bld['osm_twin_name'].fillna('') + ' | ' + bld['building_id'].map(_vf_own).fillna(''))
_vf_all = floor_rules.volunteer_fire_ids(bld, station=VOLUNTEER_FIRE_PATTERN, exclude=VOLUNTEER_FIRE_EXCLUDE_PATTERN,
                                         exclude_head=VOLUNTEER_FIRE_EXCLUDE_HEAD_PATTERN, volunteer=VOLUNTEER_FIRE_CITY_PATTERN, city_ags=())
_vf_city = bld['ags'].fillna('').astype(str).str[:5].isin(VOLUNTEER_FIRE_CITY_AGS)
_vf_new = set(bld.loc[bld['building_id'].isin(_vf_all) & _vf_city
                      & _vf_txt.str.contains(VOLUNTEER_FIRE_OWN_TEXT_PATTERN, case=False, regex=True), 'building_id']) - _sets['volunteer_fire']
_vf_w0 = _sets['church'] | _sets['volunteer_fire'] | _sets['sports_hall'] | _vf_new
_vf_others = bld[bld['vacancy'].eq('') & ~bld['building_id'].isin(_vf_w0)].groupby('zone_idx').size()
_vf_zone = bld.set_index('building_id')['zone_idx']
volunteer_fire_only_carrier = sorted(b_ for b_ in _vf_new if _vf_others.get(_vf_zone.get(b_), 0) == 0)
_sets['volunteer_fire'] = _sets['volunteer_fire'] | (_vf_new - set(volunteer_fire_only_carrier))
# a parking garage or deck: its head names one and no other use, and it holds no occupant POI (only outlines, non-occupants or
# parking) - Workers 0 on its own units
_pk_head = bld['head'].str.contains(FLOOR_TYPE_PATTERNS['parking'], case=False, regex=True)
_pk_other = bld['head'].str.contains(PARKING_OTHER_USE_PATTERN, case=False, regex=True)
_pk_occ = own06[~own06['group'].isin(['generic', 'unmapped', 'industry', 'transport', 'vacant'])
                & ~own06['amenity'].isin(NON_OCCUPANT_TAGS['amenity']) & ~own06['poi_use'].fillna('').str.contains('parking')]
_sets['parking'] = set(bld.loc[bld['parking'] & _pk_head & ~_pk_other & ~bld['building_id'].isin(_pk_occ['building_id']), 'building_id'])
print(f'  ok  city volunteer fire stations by their own text: {len(_vf_new):,} more, {len(volunteer_fire_only_carrier)} left out as '
      f'their zone\'s only Workers carrier ({", ".join(volunteer_fire_only_carrier)}); parking garages and decks: {len(_sets["parking"]):,}')
bld['rule'] = ''
for _name, _ids in _sets.items():                # one rule per building, in this order
    bld.loc[bld['rule'].eq('') & bld['building_id'].isin(_ids) & bld['vacancy'].eq(''), 'rule'] = _name
_b = bld.set_index('building_id')
_zoned = bld['zone_idx'].notna()
print('  ok  the single-category rules (vacant buildings left out; a building takes the first rule that applies, in this order):')
for _name, _ids in _sets.items():
    _m = bld['building_id'].isin(_ids)
    print(f'        {_name:<15} {int(_m.sum()):>6,} buildings ({int((_m & _zoned).sum()):,} in a zone); '
          f'the rule applies to {int(bld["rule"].eq(_name).sum()):,}')

# --- 4.4b who carries School, University and Kindergarten: the labels -----------------------------------------------
# real schools typed after a tenant (their Mensa, their Kita) or labelled lessons get the school label; vocational schools labelled
# university become School (vocational) - lib/edu_carriers.py, EDU_CARRIER_RULES; the units below follow the corrected labels
_edu = edu_carriers.Ctx(bld, pois, pairs06, own06, poi_attrs, edu_outlines, edu_geoms, EDU_CARRIER_RULES, head_split=HALL_TYPE_HEAD_SPLIT)
_ebl, _epl, edu_forced_types, edu_label_log = edu_carriers.label_fixes(_edu, bld, pois)
_efix_b = {b: set(_ebl[b]) - set(l) for b, l in zip(bld['building_id'], bld['labels']) if b in _ebl}
_efix_p = {p: set(_epl[p]) - set(l) for p, l in zip(pois['poi_id'], pois['labels']) if p in _epl}
bld['labels'] = pd.Series([_labels(';'.join(_ebl[b])) if b in _ebl else l for b, l in zip(bld['building_id'], bld['labels'])],
                          index=bld.index, dtype=object)
pois['labels'] = pd.Series([_labels(';'.join(_epl[p])) if p in _epl else l for p, l in zip(pois['poi_id'], pois['labels'])],
                           index=pois.index, dtype=object)
_kept = pois[~pois['non_occupant']]
_b = bld.set_index('building_id')
print('  ok  education labels from the evidence (R1 a school typed after a tenant gets school, R2 / R4 a vocational school labelled '
      'university becomes vocational School, R3 a school labelled lessons gets school): '
      + ' · '.join(f'{k} {v}' for k, v in edu_label_log['rule'].value_counts().sort_index().items()))
with pd.option_context('display.max_colwidth', 90):
    print(edu_label_log.assign(name=edu_label_log['building_id'].map(_b['name']))[['rule', 'building_id', 'poi_id', 'name', 'change', 'evidence']]
          .to_string(index=False))

# --- 4.4c the labels of steps 06 and 07 reconciled by OSM use ---------------------------------------------------------------------
# step 06 labelled each building whole (a one-POI building was not asked again), step 07 each asked POI on its own, with the same
# definitions - yet the same kinds of place got different labels. By its own OSM use and data (lib/label_rules.py, LABEL_RULES): a
# pharmacy carries retail_daily, a fuel station too unless its data show no shop (then errands); a beverage market, kiosk, newsagent,
# tobacco or lottery shop retail_daily instead of retail_non_daily; an eating place leisure instead of retail; a leisure venue
# labelled sports leisure too. A building unit follows its only POI, an asked POI its own use, a shared building's rest unit its
# own data; a label the rules take off a POI does not come back as a rest unit. Services and the rest units follow in 4.6b, the
# fuel stations without a shop once more after the canopy pass (4.7)
_lr_sole = label_rules.sole_pois(bld, own06)
_lr_unit_b = set(bld.loc[~bld['building_id'].isin(_shared_b) & bld['vacancy'].eq(''), 'building_id'])
_lr_act = set(pois.loc[~pois['non_occupant'] & pois['vacancy'].eq('') & pois['slot'].eq(''), 'poi_id'])
_lr_b0, _lr_p0 = dict(zip(bld['building_id'], bld['labels'])), dict(zip(pois['poi_id'], pois['labels']))
_lr_blank = lambda s: s.fillna('').astype(str).str.strip().eq('')    # a building with evidence: a POI, a name or a site (as in 4.5)
_lr_ev_b = set(bld.loc[bld['n_pois'].gt(0) | ~_lr_blank(bld['name']) | ~_lr_blank(bld['osm_twin_name'])
                       | bld['building_id'].isin(pairs06.loc[pairs06['how'].eq('site') & ~pairs06['poi_use'].isin(REDIST_LANDUSE_USES),
                                                             'building_id']), 'building_id'])
def _lr_cover():                                # what the asked POIs of each shared building carry (as for the rest units)
    return pois[~pois['non_occupant']].groupby('building_id')['labels'].agg(lambda s: set().union(*s)).to_dict()
def _lr_set(nb_, np_):
    bld['labels'] = pd.Series([nb_.get(b_, l_) for b_, l_ in zip(bld['building_id'], bld['labels'])], index=bld.index, dtype=object)
    pois['labels'] = pd.Series([np_.get(p_, l_) for p_, l_ in zip(pois['poi_id'], pois['labels'])], index=pois.index, dtype=object)
_lr_cov0, _lr_logs = _lr_cover(), []
_nb, _np, lr_no_shop_b, lr_no_shop_p, _lg = label_rules.pharmacies_fuel(
    bld, pois, _lr_sole, LABEL_RULES['pharmacies'], LABEL_RULES['fuel'], unit_b=_lr_unit_b, active_p=_lr_act,
    paired_canopies=set(station_pairs), head_split=HALL_TYPE_HEAD_SPLIT, wash_pattern=FLOOR_ONE_STOREY_RULES['wash_pattern'],
    catalogue=LLM_ACTIVITY_LABELS, head=floor_rules.head)
_lr_set(_nb, _np); _lr_logs.append(_lg)
_nb, _np, _lg = label_rules.daily_shops(bld, pois, _lr_sole, LABEL_RULES['daily_shops'], unit_b=_lr_unit_b, active_p=_lr_act,
                                        catalogue=LLM_ACTIVITY_LABELS)
_lr_set(_nb, _np); _lr_logs.append(_lg)
_nb, _np, _lg = label_rules.eating_places(bld, pois, _lr_sole, LABEL_RULES['eating_places'], unit_b=_lr_unit_b, active_p=_lr_act,
                                          staff_pattern=VISITOR_STAFF_ONLY_PATTERN + '|' + LABEL_RULES['eating_places']['staff_pattern'],
                                          staff_classes=VISITOR_STAFF_ONLY_CLASSES, catalogue=LLM_ACTIVITY_LABELS)
_lr_set(_nb, _np); _lr_logs.append(_lg)
_nb, _np, _lg = label_rules.venues(bld, pois, _lr_sole, poi_attrs, LABEL_RULES['venues'], unit_b=_lr_unit_b,
                                   shared_b=_shared_b - _vac_b, active_p=_lr_act, cover=_lr_cover(), evidence_b=_lr_ev_b,
                                   head_split=HALL_TYPE_HEAD_SPLIT, head=floor_rules.head, catalogue=LLM_ACTIVITY_LABELS)
_lr_set(_nb, _np); _lr_logs.append(_lg)
_nb, _lg = label_rules.no_reopen(bld, _lr_cov0, _lr_cover(), _shared_b, LLM_ACTIVITY_LABELS)
_lr_set(_nb, {}); _lr_logs.append(_lg)
label_log = pd.concat(_lr_logs, ignore_index=True)
_kept = pois[~pois['non_occupant']]
_b = bld.set_index('building_id')
_lc = label_log[label_log['labels_before'].map(tuple) != label_log['labels_after'].map(tuple)]
# checks: work stays everywhere; every change carries none of its rule's dropped labels and all its added ones; the labels now
# are those of each unit's last change
if not (bld['labels'].map(lambda t: 'work' in t).all() and pois['labels'].map(lambda t: 'work' in t).all()):
    raise AssertionError('a label rule took work off a building or a POI')
_lr_spec = {'eating_places': (LABEL_RULES['eating_places']['drop'], ()), 'pharmacies': ((), LABEL_RULES['pharmacies']['add']),
            'daily_shops': (LABEL_RULES['daily_shops']['drop'], LABEL_RULES['daily_shops']['add']), 'venues': ((), LABEL_RULES['venues']['add'])}
_bad = [r_.unit_id for r_ in _lc.itertuples(index=False) if r_.rule in _lr_spec
        and (set(_lr_spec[r_.rule][0]) & set(r_.labels_after) or not set(_lr_spec[r_.rule][1]) <= set(r_.labels_after))]
_lc = _lc.assign(is_poi=_lc['kind'].eq('poi'))                    # a POI and an OSM-only building can share an id
_last = _lc.drop_duplicates(['is_poi', 'unit_id'], keep='last')
_now = {False: dict(zip(bld['building_id'], bld['labels'])), True: dict(zip(pois['poi_id'], pois['labels']))}
_bad += [u_ for u_, p_, a_ in zip(_last['unit_id'], _last['is_poi'], _last['labels_after']) if tuple(_now[p_][u_]) != tuple(a_)]
if _bad:
    raise AssertionError(f'label rules: {len(_bad)} unit(s) do not carry what their rule says, e.g. {_bad[:5]}')
print('  ok  the labels of steps 06 and 07 reconciled by OSM use (LABEL_RULES) - units relabelled, and the '
      'labels they gain (+) and lose (-):')
for _r, _g in _lc.drop_duplicates(['rule', 'unit_id']).groupby('rule', sort=False):
    _add = pd.Series([l_ for b_, a_ in zip(_g['labels_before'], _g['labels_after']) for l_ in set(a_) - set(b_)], dtype=object).value_counts()
    _drop = pd.Series([l_ for b_, a_ in zip(_g['labels_before'], _g['labels_after']) for l_ in set(b_) - set(a_)], dtype=object).value_counts()
    _keep = label_log[label_log['rule'].eq(_r) & ~label_log.index.isin(_lc.index)]
    print(f'        {_r:<14} {len(_g):>4} ({" · ".join(f"{k_} {v_}" for k_, v_ in _g["kind"].value_counts().items())}): '
          + ' · '.join([f'+{k_} {v_}' for k_, v_ in _add.items()] + [f'-{k_} {v_}' for k_, v_ in _drop.items()])
          + (f'; left alone: ' + ' · '.join(f'{k_} {v_}' for k_, v_ in _keep['why'].str.extract(r'^(\w+)')[0].value_counts().items())
             if len(_keep) else ''))
print(f'        fuel stations whose data show no shop: {len(lr_no_shop_b)} one-POI buildings and {len(lr_no_shop_p)} asked POIs '
      '(errands instead of retail_daily)')

# --- 4.4d the typical size of a POI (UNIT_SIZE_RULES) --------------------------------------------------------------------------
# a POI mapped as a point counts about 1 m2 in the step-04.5 shares next to a polygon; the rules below read the statistic the
# point caps read: a quantile of the mapped polygons of each POI group (groups with at least POINT_SIZE_CAP_MIN_POLYGONS of
# them, else all groups' polygons)
_sz_grp = sorted(set(own06['group']) - POI_NO_UNIT_GROUPS - {'vacant'})
def _sz_typ(qq):
    t_ = floor_rules.point_size_caps(own06, _b['area_m2'], groups=_sz_grp, quantile=qq, outline_share=POINT_SIZE_CAP_OUTLINE_SHARE)
    n_, a_ = floor_rules.point_size_cap_all(own06, _b['area_m2'], quantile=qq, outline_share=POINT_SIZE_CAP_OUTLINE_SHARE)
    return {g_: float(r_.cap) for g_, r_ in t_.iterrows() if r_.n >= POINT_SIZE_CAP_MIN_POLYGONS}, float(a_)
size_log = []                                   # (building, POI, its floor before, after) of every size rule
size_point_min, size_point_min_all = _sz_typ(UNIT_SIZE_RULES['point_min_quantile'])
size_mall_w, size_mall_w_all = _sz_typ(UNIT_SIZE_RULES['mall_quantile'])
# a point that repeats a polygon of its building - the same name or the same use - is the same place mapped twice: not lifted
_sz_nn = lambda s_: s_.fillna('').astype(str).str.lower().str.replace(r'[^a-z0-9äöüß]', '', regex=True)
_sz_poly = own06[own06['geom_kind'].eq('area')]
_sz_dup = ({(b_, 'n:' + n_) for b_, n_ in zip(_sz_poly['building_id'], _sz_nn(_sz_poly['name'])) if n_}
           | {(b_, 'u:' + u_) for b_, u_ in zip(_sz_poly['building_id'], _sz_poly['poi_use'].fillna('').astype(str)) if u_})
size_dup_log = []
# a pharmacy occupies at most UNIT_SIZE_RULES['pharmacy_cap_m2'] (the p90 of the mapped pharmacy polygons): its point POIs and the
# building units whose only POI is a pharmacy
_sz_s1 = own06[own06['building_id'].isin(bld.loc[bld['n_pois'] == 1, 'building_id'])].drop_duplicates('building_id')
_sz_ph_b = set(_sz_s1.loc[_sz_s1['poi_use'].eq('pharmacy'), 'building_id'])
_SZ_PH = float(UNIT_SIZE_RULES['pharmacy_cap_m2'])
# the sizes of single tenants by format (UNIT_SIZE_FORMATS, lib/unit_sizes.py): the
# easyApotheke format keeps up to its own cap; a chain filed as a small shop at most its format's size; counters and kiosks the
# median of the mapped kiosk polygons; a mall tenant of a listed chain its published size - as its weight only in the malls whose
# retail floor is published (MALL_RETAIL_FLOOR, 4.5b), as its minimum in every mall; a format set to None or left out
# takes the general size rules
_SZF = {k_: UNIT_SIZE_FORMATS.get(k_) for k_ in ('pharmacy_large', 'chain_caps', 'counters', 'mall_chains')}
_sz_ph_bcap = unit_sizes.pharmacy_building_caps(_sz_s1, _sz_ph_b, _SZ_PH, _SZF['pharmacy_large'])
_sz_kiosk = unit_sizes.kiosk_median(own06, _SZF['counters'])
_sz_listed = {i_ for v_ in MALL_RETAIL_FLOOR.values() for i_ in v_['ids']}
_sz_ph_txt = f' (the easyApotheke format {_SZF["pharmacy_large"]["cap_m2"]:,.0f} m2)' if _SZF['pharmacy_large'] else ''
_sz_ki_txt = (f'counters and kiosks at least {_sz_kiosk:,.1f} m2 (the kiosk median) in their building\'s shares; ' if _sz_kiosk
              else '')

# --- 4.5 the units and their occupied floor area -------------------------------------------------------
bx = pd.DataFrame({'building_id': bld['building_id'], 'cls': bld['bosserhof_class'], 'RC': bld['RC'], 'N': bld['storeys'].astype(float),
                   'office_whole': rcx['office_whole'], 'public_office': tf['public_office'] & ~tf['care'], 'small_school': tf['small_school'],
                   'small_unit': tf['small_unit'], 'studio': tf['gym_studio'], 'sport_hall': sf['sport_hall'], 'parking': sf['parking'],
                   'alk_res': rcx['alk_res'], 'box_quirk': rcx['box_quirk'], 'pure_home': tf['pure_home'] & ~tf['care'] & ~tf['pure_home_excl'],
                   'n_pois': bld['n_pois'], 'small_attraction': tf['small_attraction'], 'worship': sf['worship']}).set_index('building_id', drop=False)
_BX = {r.building_id: r for r in bx.itertuples(index=False)}
_SOLE = dict(class_storeys=UNIT_CLASS_STOREYS, group_storeys=UNIT_GROUP_STOREYS, street_classes=UNIT_STREET_CLASSES,
             level_classes=UNIT_LEVEL_TAG_CLASSES, level_groups=UNIT_LEVEL_TAG_GROUPS, counter_all_uses=UNIT_COUNTER_ALL_USES,
             home_storeys=UNIT_HOME_STOREYS)
_FAM = dict(group_family=POI_GROUP_FAMILY, mall_classes=POI_MALL_CLASSES, mall_unit_groups=POI_MALL_UNIT_GROUPS,
            bigbox_classes=POI_BIGBOX_CLASSES, culture_all_uses=POI_CULTURE_ALL_USES, cinema_use='cinema', chain_groups=POI_CHAIN_GROUPS,
            anchor_groups=POI_ANCHOR_GROUPS)
_sole = own06[own06['building_id'].isin(bld.loc[bld['n_pois'] == 1, 'building_id'])].drop_duplicates('building_id').set_index('building_id')
_A, _N, _FA, _f = _b['area_m2'].to_dict(), _b['storeys'].to_dict(), _b['floor_area_m2'].to_dict(), _b['workers_factor'].to_dict()
_L = _b['labels'].to_dict()
EPS = 1e-9

def _rem_f(bid):                                # the remainder's factor: the office factor in shop and office buildings, else the building's own
    return REDIST_REMAINDER_WORKERS_FACTOR if _b.at[bid, 'bosserhof_class'] in REDIST_REMAINDER_OFFICE_CLASSES else _f[bid]

def _storeys_txt(n, N):
    return 'every storey' if n >= N else f'{int(n)} of {int(N)} storeys'

# small places stay small: a building filed as small-scale retail or a restaurant occupies at most the
# typical size of its kind, unless its type names a food store or its only POI is a food store or big box; the cut floor carries nothing
_xt = re.compile(SMALL_PLACE_EXEMPT_TYPE_PATTERN, re.IGNORECASE)
_sg = _sole['group'] if 'group' in _sole.columns else pd.Series(dtype=object)
_small_b = bld['bosserhof_class'].isin(list(SMALL_PLACE_CAP_GROUPS))
_exempt_b = _small_b & (bld['llm_type'].fillna('').str.contains(_xt) | bld['building_id'].map(_sg).isin(POI_ANCHOR_GROUPS))
_small_cap = dict(zip(bld.loc[_small_b & ~_exempt_b, 'building_id'],
                      bld.loc[_small_b & ~_exempt_b, 'bosserhof_class'].map(lambda c: float(POINT_SIZE_CAP_M2[SMALL_PLACE_CAP_GROUPS[c]]))))
_small_groups = set(SMALL_PLACE_CAP_GROUPS.values())
# a building filed as a small shop or restaurant whose own data name a big-box chain (the chain list of the one-storey rule)
# occupies at most the big-box cap instead (UNIT_SIZE_RULES chain_cap_group) - or its format's size
# (UNIT_SIZE_FORMATS chain_caps)
_sz_nm = (_own1.groupby('building_id')['name'].agg(lambda s_: ' | '.join(sorted({x_ for x_ in s_.dropna().astype(str) if x_})))
          .reindex(bld['building_id']).fillna('') + ' | '
          + _own1.groupby('building_id')['brand'].agg(lambda s_: ' | '.join(sorted({x_ for x_ in s_.dropna().astype(str) if x_})))
          .reindex(bld['building_id']).fillna('') + ' | ' + bld['name'].fillna('').astype(str).to_numpy() + ' | '
          + bld['poi_names'].fillna('').astype(str).to_numpy())
_sz_ch = (_sz_nm.str.contains(FLOOR_ONE_STOREY_RULES['box_chain_pattern'], case=False, regex=True)
          & ~_sz_nm.str.contains(FLOOR_ONE_STOREY_RULES['box_chain_exclude'], case=False, regex=True))
size_chain_ids = set(_sz_ch.index[_sz_ch])
_small_cap = {b_: (max(c_, unit_sizes.chain_cap(_sz_nm.get(b_, ''), FLOOR_ONE_STOREY_RULES['box_chain_pattern'], _SZF['chain_caps'],
                                                float(POINT_SIZE_CAP_M2[UNIT_SIZE_RULES['chain_cap_group']])))
                    if b_ in size_chain_ids else c_)
              for b_, c_ in _small_cap.items()}
_cut = {}                                       # unit -> the floor the cap cut

rows, slot_rows, shared_info = [], [], []
_used, _n_act = {}, {}                          # per shared building: the floor its POIs and slots occupy, its active POIs
# building units: the buildings without an asked POI (after the non-occupant drop)
for bid in bld.loc[~bld['building_id'].isin(_shared_b) & bld['vacancy'].eq(''), 'building_id']:
    r = _BX[bid]
    has = r.n_pois == 1 and bid in _sole.index
    g, u, lv, ch = ((_sole.at[bid, 'group'], _sole.at[bid, 'poi_use'], _sole.at[bid, 'levels'], _sole.at[bid, 'chain'])
                    if has else (None, None, None, ''))
    n, why = floor_rules.sole_storeys(r, g, u, lv, ch, **_SOLE)
    k = min(r.N, n)
    _O, _c = _A[bid] * k, min(_small_cap.get(bid, np.inf), _sz_ph_bcap.get(bid, np.inf))   # a pharmacy: at most its cap
    if _O > _c:
        _cut[bid] = _O - _c
    rows.append((bid, bid, 'building', min(_O, _c), _f[bid], _L[bid], _storeys_txt(n, r.N),
                 f'sole occupant, {why}: {_storeys_txt(n, r.N)}' + (f'; a small place: at most {_c:,.0f} m2' if _O > _c else '')))
    if not r.RC and k < r.N and why != 'plain home':
        rows.append((bid, bid + '#rem', 'remainder', _A[bid] * (r.N - k), _rem_f(bid), ('work',), 'remainder',
                     'the floors the sole occupant leaves, no dwellings'))
# POI units: the asked POIs of the shared buildings
for bid, q in _kept[_kept['building_id'].isin(_shared_b)].groupby('building_id', sort=False):
    r, A, N = _BX[bid], _A[bid], _N[bid]
    _vq = (q['slot'] == 'vacant').to_numpy()     # a vacant POI is sized as its active self, then left empty
    q = q.assign(slot=q['slot'].where(~_vq, ''))
    fw = [floor_rules.poi_family(x, r, **_FAM) for x in q.itertuples(index=False)]
    fams, whys = [f for f, _ in fw], [w for _, w in fw]
    s, slot = q['share_in_building'].to_numpy(dtype=float), (q['slot'] != '').to_numpy()
    # an anchor: a food store or big box in its own big-box building, or a point of family ALL whose POI-level class is the
    # building's class - the main occupant, not capped; every other point is capped and never the main occupant
    anchor = (((q['group'].isin(POI_ANCHOR_GROUPS)).to_numpy() & (r.cls in POI_BIGBOX_CLASSES))
              | ((np.asarray(fams[:len(q)], dtype=object) == 'ALL') & (q['poi_class'] == r.cls).to_numpy())) & (q['slot'] == '').to_numpy()
    point = ((q['geom_kind'] == 'point') & ((q['how'] != 'unit') | (r.cls not in POI_MALL_CLASSES))   # tenant points outside malls too
             & (q['slot'] == '')).to_numpy() & ~anchor
    point = point | (((q['poi_use'] == 'pharmacy') & (q['geom_kind'] == 'point') & (q['slot'] == '')).to_numpy() & ~anchor)
    cap = q['group'].map(POINT_SIZE_CAP_M2).fillna(POINT_SIZE_CAP_DEFAULT_M2).to_numpy(dtype=float)
    cap = np.where(q['poi_use'].eq('pharmacy').to_numpy(), np.minimum(cap, unit_sizes.pharmacy_caps(q, _SZ_PH, _SZF['pharmacy_large'])), cap)
    vl = _slot_shares.get(bid, [])
    vs = float(sum(vl))
    if vl:                                       # each vacant pair: an empty GROUND slot holding its share
        k = len(vl)
        s, slot, point, cap = np.append(s, vl), np.append(slot, [True] * k), np.append(point, [False] * k), np.append(cap, [np.inf] * k)
        fams.extend(['GROUND'] * k)
    s = s / s.sum()
    # in a mall, the tenant points' shares are weighted by their group's typical size; a listed chain by its
    # published size in the malls of MALL_RETAIL_FLOOR, a counter or kiosk by the kiosk median
    if r.cls in POI_MALL_CLASSES:
        _sz_u = np.append(((q['geom_kind'] == 'point') & (q['how'] == 'unit') & (q['slot'] == '')).to_numpy(), np.zeros(len(s) - len(q), bool))
        if _sz_u.sum() >= 2:
            _sz_w = np.append(unit_sizes.mall_weights(q, q['group'].map(size_mall_w).fillna(size_mall_w_all).to_numpy(dtype=float),
                                                     _SZF['mall_chains'] if bid in _sz_listed else None, _SZF['counters'], _sz_kiosk),
                              np.zeros(len(s) - len(q)))
            _s0 = s.copy()
            s = np.where(_sz_u, s[_sz_u].sum() * _sz_w / _sz_w[_sz_u].sum(), s)
            for _j in np.flatnonzero(_sz_u[:len(q)]):
                size_log.append((bid, q['poi_id'].iloc[_j], float(_s0[_j] * A * N), float(s[_j] * A * N)))
    # a point counts at least its group's median polygon (a counter or kiosk the kiosk median, a listed chain in a mall
    # its published size), at most an equal share and the footprint; the share it gains comes from
    # the building's other occupants by their shares
    _sz_pt = np.append(((q['geom_kind'] == 'point') & (q['slot'] == '')).to_numpy(), np.zeros(len(s) - len(q), bool))
    _sz_d = np.array([((bid, 'n:' + n_) in _sz_dup and n_ != '') or ((bid, 'u:' + u_) in _sz_dup and u_ != '')
                      for n_, u_ in zip(_sz_nn(q['name']), q['poi_use'].fillna('').astype(str))] + [False] * (len(s) - len(q)), dtype=bool)
    for _j in np.flatnonzero(_sz_pt & _sz_d):
        size_dup_log.append((bid, q['poi_id'].iloc[_j]))
    _sz_pt = _sz_pt & ~_sz_d
    if _sz_pt.any() and len(s) >= 2:
        _sz_m = np.append(unit_sizes.point_min(q, q['group'].map(size_point_min).fillna(size_point_min_all).to_numpy(dtype=float),
                                             _SZF['counters'], _sz_kiosk, _SZF['mall_chains'] if r.cls in POI_MALL_CLASSES else None),
                          np.zeros(len(s) - len(q)))
        _sz_lo = np.where(_sz_pt, np.minimum(np.minimum(_sz_m, A * N / len(s)), A) / (A * N), 0.0)
        _sz_L = _sz_lo > s * (1 + 1e-9)
        if _sz_L.any():
            _s0 = s.copy()
            for _it in range(100):
                if _sz_L.all():
                    s = _sz_lo / _sz_lo.sum()
                    break
                _new = np.where(_sz_L, _sz_lo, _s0 * (1.0 - _sz_lo[_sz_L].sum()) / max(_s0[~_sz_L].sum(), 1e-300))
                _add = ~_sz_L & (_sz_lo > _new * (1 + 1e-9))
                if not _add.any():
                    s = _new
                    break
                _sz_L = _sz_L | _add
            for _j in np.flatnonzero(_sz_L[:len(q)]):
                size_log.append((bid, q['poi_id'].iloc[_j], float(_s0[_j] * A * N), float(s[_j] * A * N)))
    act = ~slot
    single_cls = act.sum() == 1 and q.loc[act[:len(q)], 'poi_class'].iloc[0] == r.cls
    one = (not slot.any()) and q['group'].nunique() == 1 and (act.sum() >= 2 or single_cls)
    if one:
        g0, x0 = q['group'].iloc[0], q.iloc[0]
        n, why = floor_rules.sole_storeys(r, g0, x0['poi_use'], x0['levels'] if len(q) == 1 else None,
                                          x0['chain'] if len(q) == 1 else '', **_SOLE)
        occ = A * min(N, n) * s
        free = max(A * N - occ.sum(), 0.0)
        fams = ['SOLE ' + ('ALL' if n >= N else f'{int(n)}')] * len(q)
        whys = [f'one facility ({g0}), {why}'] * len(q)
        ground, capped, freed = np.nan, 0, 0.0
    else:
        occ0, _ = floor_rules.share_floor(A, N, s, fams, np.zeros(len(s), bool), slot, cap)
        occ, free = floor_rules.share_floor(A, N, s, fams, point, slot, cap)
        ground = floor_rules.ground_use(occ, fams)
        capped, freed = int((point & (occ0 > cap + 1e-9)).sum()), float(np.clip(occ0 - occ, 0, None)[point].sum())
    # a small place stays small: a point of a small-format group takes at most its cap also as one facility or main occupant
    _spt = ((q['geom_kind'] == 'point') & ((q['how'] != 'unit') | (r.cls not in POI_MALL_CLASSES)) & (q['slot'] == '')
            & q['group'].isin(_small_groups)).to_numpy()
    if _spt.any():
        occ = np.array(occ, dtype=float)
        _cj = q['group'].map(POINT_SIZE_CAP_M2).to_numpy(dtype=float)
        _cj = np.where(q['poi_use'].eq('pharmacy').to_numpy(), np.minimum(_cj, unit_sizes.pharmacy_caps(q, _SZ_PH, _SZF['pharmacy_large'])), _cj)
        for j in np.flatnonzero(_spt & (occ[:len(q)] > _cj + 1e-9)):
            _cut[q['poi_id'].iloc[j]] = occ[j] - _cj[j]
            occ[j] = _cj[j]
            whys[j] = f'{whys[j]}; a small place: at most {_cj[j]:,.0f} m2'
    _used[bid], _n_act[bid] = float(np.asarray(occ).sum()), int(act[:len(q)].sum())
    for j, x in enumerate(q.itertuples(index=False)):
        if slot[j] or _vq[j]:
            slot_rows.append((bid, x.poi_id, 'poi', occ[j], x.workers_factor, x.labels, fams[j], whys[j],
                              'vacant' if _vq[j] else x.slot))
        else:
            rows.append((bid, x.poi_id, 'poi', occ[j], x.workers_factor, x.labels, fams[j], whys[j]))
    if not r.RC and free > 1e-6 * A * N and bid not in _vac_b:
        rows.append((bid, bid + '#rem', 'remainder', free, _rem_f(bid), ('work',), 'remainder',
                     'the floor no unit occupies, no dwellings'))
    shared_info.append((bid, one, len(q), int((act[:len(q)] & ~_vq).sum()), ground, A, capped, freed, vs))
shared = pd.DataFrame(shared_info, columns=['building_id', 'one_facility', 'n_pois', 'n_active', 'ground_m2', 'area_m2', 'n_capped',
                                            'freed_m2', 'vacant_slot_share']).set_index('building_id')
# rest units: a building label none of its asked POIs carries, on the floor its POIs leave free - at
# most the floor its class rule gives a sole occupant; where the POIs fill that floor, one average occupant's share
_cover = _kept.groupby('building_id')['labels'].agg(lambda s: set().union(*s))
uncovered = {}
for bid in sorted(_shared_b - _vac_b):
    left = tuple(l for l in _L[bid] if l != 'work' and l not in _cover[bid])
    if left:
        uncovered[bid] = left
        if REDIST_KEEP_UNCOVERED_BUILDING_CATEGORIES:
            r = _BX[bid]
            n, why = floor_rules.sole_storeys(r, None, None, None, '', **_SOLE)
            _rule = min(_A[bid] * min(r.N, n), _small_cap.get(bid, np.inf))     # a small place stays small
            _free = max(_rule - _used.get(bid, 0.0), 0.0)
            _share = _rule / (_n_act.get(bid, 1) + 1)
            rows.append((bid, bid + '#rest', 'rest', max(_free, _share), _f[bid], left, _storeys_txt(n, r.N),
                         f'rest: the floor its POIs leave free of the class rule\'s {_storeys_txt(n, r.N)} ({why})' if _free >= _share else
                         f'rest: one average occupant\'s share of the class rule\'s {_storeys_txt(n, r.N)} - its POIs fill it ({why})'))

uo = pd.DataFrame(rows, columns=['building_id', 'unit_id', 'kind', 'O', 'f', 'labels', 'family', 'unit_rule'])

# --- 4.5b the floor of the malls (MALL_RETAIL_FLOOR) --------------------------------------------------------------------------
# the shops, restaurants and services of a listed mall occupy at most its published retail floor, one factor for all of them (never
# above 1); its non-retail tenants (MALL_NONRETAIL_GROUPS: the castle's library and museum, a Kita ...) keep their floor; a POI that
# is the mall's own outline (a generic POI tagged as the building: BraWoPark's) stands for the shops OSM does not map and keeps only
# the published floor its tenants leave; the rest of the floor - car parks, aisles, plant, delivery - carries nothing, a remainder
# unit included (as the small-place cap)
_mf_poi = pois.drop_duplicates('poi_id').set_index('poi_id')
_mf_grp = uo['unit_id'].map(_mf_poi['group']).where(uo['kind'].eq('poi'))
_mf_ret = uo['kind'].isin(['building', 'poi']) & ~_mf_grp.isin(MALL_NONRETAIL_GROUPS)
_mf_shell = (uo['kind'].eq('poi') & _mf_grp.isin(['generic', 'unmapped']) & uo['unit_id'].map(_mf_poi['poi_use_tag']).eq('building'))
mall_floor_log, _mrem_all = [], pd.Series(False, index=uo.index)
for _mk, _mv in MALL_RETAIL_FLOOR.items():
    _mb = uo['building_id'].isin(_mv['ids'])
    _mm, _ms = _mb & _mf_ret & ~_mf_shell, _mb & _mf_ret & _mf_shell
    _mO, _mS = float(uo.loc[_mm, 'O'].sum()), float(uo.loc[_ms, 'O'].sum())
    _mV = float(sum(r_[3] for r_ in slot_rows if r_[8] == 'vacant' and r_[0] in set(_mv['ids'])
                    and _mf_poi['group'].get(r_[1]) not in MALL_NONRETAIL_GROUPS))   # the mall's vacant shops
    _mc = min(1.0, float(_mv['m2']) / (_mO + _mV)) if _mO + _mV > 0 else 1.0
    _sc = min(1.0, (float(_mv['m2']) - (_mO + _mV) * _mc) / _mS) if _mS > 0 else 1.0
    if _sc <= 1e-9:
        raise AssertionError(f'{_mk}: its tenants fill its published retail floor, so its outline POI would occupy nothing')
    uo.loc[_mm, 'O'] = uo.loc[_mm, 'O'] * _mc
    uo.loc[_ms, 'O'] = uo.loc[_ms, 'O'] * _sc
    _mrem = uo['building_id'].isin(_mv['ids']) & uo['kind'].eq('remainder')
    mall_floor_log.append((_mk, int(bld['building_id'].isin(_mv['ids']).sum()), int((_mm | _ms).sum()), _mO + _mS,
                           _mO * _mc + _mS * _sc, float(_mv['m2']), _mS * _sc, int(_mrem.sum()), float(uo.loc[_mrem, 'O'].sum())))
    _mrem_all |= _mrem
uo = uo[~_mrem_all].copy()
mall_floor_log = pd.DataFrame(mall_floor_log, columns=['mall', 'buildings', 'units', 'retail_before_m2', 'retail_after_m2', 'published_m2',
                                                       'outline_m2', 'remainders_cut', 'remainder_m2_cut'])
if (mall_floor_log['retail_after_m2'] > mall_floor_log['published_m2'] + 1e-6).any():
    raise AssertionError('a listed mall\'s retail units occupy more than its published retail floor')
print(f'  ok  the floor of the malls: {len(mall_floor_log)} malls at most their published retail floor, '
      f'{mall_floor_log["retail_before_m2"].sum() / 1e3:,.1f} -> {mall_floor_log["retail_after_m2"].sum() / 1e3:,.1f} thousand m2 '
      f'({int(mall_floor_log["outline_m2"].gt(0).sum())} outline POI(s) keeping the {mall_floor_log["outline_m2"].sum():,.0f} m2 their '
      f'tenants leave); {int(mall_floor_log["remainders_cut"].sum())} remainder units ({mall_floor_log["remainder_m2_cut"].sum():,.0f} m2) '
      'of their floor beyond it carry nothing')
slots = pd.DataFrame(slot_rows, columns=['building_id', 'unit_id', 'kind', 'O', 'f', 'labels', 'family', 'unit_rule', 'slot'])
uo['rule'], uo['no_unit_reason'] = '', ''
_so = slots[slots['slot'].isin(['holiday flat', 'dormitory'])]   # dwellings: no unit, but subject to the last-carrier guard
uo = pd.concat([uo, _so.assign(rule=_so['slot'].map({'holiday flat': 'holiday_flat', 'dormitory': 'dormitory'}),
                               no_unit_reason=_so['slot'] + ' POI').drop(columns='slot')], ignore_index=True)
_kb = {bid: _ks(_L[bid]) for bid in uo['building_id'].unique()}
_kk = [(_kb[b] if k == 'rest' else _ks(l)) for b, k, l in zip(uo['building_id'], uo['kind'], uo['labels'])]
uo['k_red'], uo['k_all'] = [k[0] for k in _kk], [k[1] for k in _kk]
uo['O_pre'], uo['labels_pre'], uo['k_red_pre'], uo['k_all_pre'] = uo['O'], uo['labels'], uo['k_red'], uo['k_all']
# the units the rules of 4.4c changed (label_rule) keep their labels before the rules, for the last-carrier guard (section 5)
_lr_rb = _lc.groupby('building_id')['rule'].agg(lambda s: '+'.join(dict.fromkeys(s)))
_lr_ru = _lc.groupby(['is_poi', 'unit_id'])['rule'].agg(lambda s: '+'.join(dict.fromkeys(s)))
_lr_pre = []
for k_, b_, u_, l_ in zip(uo['kind'], uo['building_id'], uo['unit_id'], uo['labels']):
    o_ = (_lr_b0.get(b_) if k_ == 'building' else _lr_p0.get(u_) if k_ == 'poi'
          else tuple(x_ for x_ in _lr_b0.get(b_, ()) if x_ != 'work' and x_ not in _lr_cov0.get(b_, set())) if k_ == 'rest' else None)
    _lr_pre.append(o_ if o_ is not None and tuple(o_) != tuple(l_) else None)
_lm = pd.Series([x_ is not None for x_ in _lr_pre], index=uo.index)
uo['label_rule'] = ''
uo.loc[_lm, 'label_rule'] = [_lr_rb.get(b_, '') if k_ == 'rest' else _lr_ru.get((k_ == 'poi', u_), '') for k_, b_, u_ in
                             zip(uo.loc[_lm, 'kind'], uo.loc[_lm, 'building_id'], uo.loc[_lm, 'unit_id'])]
uo.loc[_lm, 'labels_pre'] = pd.Series([x_ for x_ in _lr_pre if x_ is not None], index=uo.index[_lm], dtype=object)
_lk = [_ks(_lr_b0[b_]) if k_ == 'rest' else _ks(t_) for k_, b_, t_ in zip(uo.loc[_lm, 'kind'], uo.loc[_lm, 'building_id'], uo.loc[_lm, 'labels_pre'])]
uo.loc[_lm, 'k_red_pre'], uo.loc[_lm, 'k_all_pre'] = [k_[0] for k_ in _lk], [k_[1] for k_ in _lk]
uo['zone_idx'] = uo['building_id'].map(_b['zone_idx'])
# places without evidence: a building unit with no POI of its own, no name (ALKIS or on its OSM outline)
# and no site other than a landuse area - its labels are the model's reading of the ALKIS function or a landuse area - counts
# REDIST_NO_POI_WEIGHT in REDIST_NO_POI_CATEGORIES (section 5)
_site_ev = set(pairs06.loc[pairs06['how'].eq('site') & ~pairs06['poi_use'].isin(REDIST_LANDUSE_USES), 'building_id'])
_blank = lambda s: s.fillna('').astype(str).str.strip().eq('')
uo['no_poi'] = (uo['kind'].eq('building') & uo['building_id'].map(_b['n_pois']).eq(0) & _blank(uo['building_id'].map(_b['name']))
                & _blank(uo['building_id'].map(_b['osm_twin_name'])) & ~uo['building_id'].isin(_site_ev))

# --- 4.6 the single-category rules on the units ----------------------------------------------------------
_brule = uo['building_id'].map(_b['rule'])
_puse = uo['unit_id'].map(pois.set_index('poi_id')['poi_use'])
uo = uo[~(uo['kind'].eq('remainder') & _brule.ne(''))].copy()          # no remainder in a building under a rule
_brule = uo['building_id'].map(_b['rule'])
_puse = uo['unit_id'].map(pois.set_index('poi_id')['poi_use'])
_m = _brule.eq('holiday_flat')
uo.loc[_m, 'rule'], uo.loc[_m & uo['no_unit_reason'].eq(''), 'no_unit_reason'] = 'holiday_flat', 'holiday flat'
_m = _brule.eq('dormitory') & uo['kind'].isin(['building', 'rest'])
uo.loc[_m, 'rule'], uo.loc[_m, 'no_unit_reason'] = 'dormitory', 'dormitory'
_m = _brule.eq('no_evidence') & uo['kind'].eq('building')
_keep = uo['building_id'].map(_b['osm_tag']).isin(NO_EVIDENCE_KEEP_TWINS)
uo.loc[_m, 'rule'] = 'no_evidence'
uo.loc[_m, 'O'] = np.minimum(uo.loc[_m, 'building_id'].map(_A), uo.loc[_m, 'building_id'].map(_small_cap).fillna(np.inf))   # a small place stays small
uo.loc[_m & ~_keep, 'labels'] = pd.Series([('work',)] * int((_m & ~_keep).sum()), index=uo.index[_m & ~_keep], dtype=object)
uo.loc[_m, 'unit_rule'] = np.where(_keep[_m], 'no-evidence residential: footprint x 1, labels kept (retail / commercial OSM twin)',
                                   'no-evidence residential: footprint x 1, Workers only')
uo.loc[_m, 'family'] = '1 storey'
# Workers 0: churches on their worship POI units, volunteer fire stations on their fire-station POI units, sports halls on
# the POI units that are the hall, its club or the school using it (unless its use, name or the head of its type is a pool,
# an ice rink, a play centre ... - the keep list, read as for the buildings)
_pq = pois.set_index('poi_id')
_pgrp = uo['unit_id'].map(_pq['group'])
_pkeep = uo['unit_id'].map((_pq['poi_use'].fillna('') + ' ' + _pq['name'].fillna('') + ' ' + floor_rules.head(_pq['poi_type'], HALL_TYPE_HEAD_SPLIT))
                           .str.contains(SPORTS_HALL_KEEP_PATTERN, case=False, regex=True)).fillna(False).astype(bool)
for _rname, _pm in (('church', _puse.isin(CHURCH_POI_USES) | _pgrp.isin(CHURCH_POI_GROUPS)), ('volunteer_fire', _puse.isin(VOLUNTEER_FIRE_POI_USES)),
                    ('sports_hall', _pgrp.isin(SPORTS_HALL_POI_GROUPS) & ~_pkeep),
                    ('parking', _pgrp.isin(['generic', 'unmapped', 'industry', 'transport']))):
    _m = (_brule.eq(_rname) & uo['no_unit_reason'].eq('')
          & (uo['kind'].isin(['building', 'rest']) | (uo['kind'].eq('poi') & _pm)))
    uo.loc[_m, 'rule'] = _rname
    uo.loc[_m, 'labels'] = uo.loc[_m, 'labels'].map(lambda t: tuple(l for l in t if l != 'work'))
_kk = [(_kb[b] if k == 'rest' else _ks(l)) for b, k, l in zip(uo['building_id'], uo['kind'], uo['labels'])]
uo['k_red'], uo['k_all'] = [k[0] for k in _kk], [k[1] for k in _kk]
# a worship POI unit gets Workers 0 in any building without a rule (a mosque in a shop building)
_m = uo['kind'].eq('poi') & _puse.isin(CHURCH_POI_USES) & uo['no_unit_reason'].eq('') & _brule.eq('') & uo['rule'].eq('')
uo.loc[_m, 'rule'] = 'church'
uo.loc[_m, 'labels'] = uo.loc[_m, 'labels'].map(lambda t: tuple(l for l in t if l != 'work'))
# a volunteer fire station labelled only work gets no unit; a church or sports hall labelled only work keeps its unit, which
# then carries nothing
_m = uo['no_unit_reason'].eq('') & uo['labels'].map(len).eq(0) & uo['rule'].eq('volunteer_fire')
uo.loc[_m, 'no_unit_reason'] = 'volunteer fire station labelled only work'
# who carries School, University and Kindergarten (lib/edu_carriers.py): a unit whose own data name another
# use, a non-teaching part of a campus or a place without any evidence loses the education label (a label the fixes of 4.4b gave
# stays) - the unit keeps its k_red, so the dropped share stays empty and no other category moves; the last-carrier guard
# (section 5) restores a zone's only carrier
_fx = {(u, l) for u, k, b in zip(uo['unit_id'], uo['kind'], uo['building_id'])
       for l in (_efix_p.get(u, ()) if k == 'poi' else _efix_b.get(b, ()) if k in ('building', 'rest') else ())}
edu_drops = edu_carriers.drops(_edu, uo[uo['no_unit_reason'].eq('')], _fx)
# the open education cases by the web check (EDU_CARRIER_LISTS): kept places lose no label, listed places
# lose theirs ('S-L')
edu_drops = edu_drops[~edu_drops['unit_id'].isin(EDU_CARRIER_LISTS['keep'])].reset_index(drop=True)
_add = []
for _u, _lab, _cat in EDU_CARRIER_LISTS['drop']:
    _r = uo[uo['unit_id'].eq(_u) & uo['no_unit_reason'].eq('')]
    if len(_r) and _lab in _r['labels'].iloc[0] and not ((edu_drops['unit_id'] == _u) & (edu_drops['label'] == _lab)).any():
        _add.append((_u, _r['building_id'].iloc[0], _r['kind'].iloc[0], _lab, _cat, 'S-L', 'list (2026-10-09)'))
if _add:
    edu_drops = pd.concat([edu_drops, pd.DataFrame(_add, columns=edu_drops.columns)], ignore_index=True)
_dl = edu_drops.groupby('unit_id')['label'].agg(set)
uo['edu_rule'] = uo['unit_id'].map(edu_drops.groupby('unit_id')['rule'].agg(lambda s: '+'.join(dict.fromkeys(s)))).fillna('')
_m = uo['unit_id'].isin(_dl.index)
uo.loc[_m, 'labels'] = pd.Series([tuple(l for l in L if l not in _dl[u]) for u, L in zip(uo.loc[_m, 'unit_id'], uo.loc[_m, 'labels'])],
                                 index=uo.index[_m], dtype=object)
# a School place in a zone with a School total of 0, known only from the cadastre or a bare OSM school outline, with no school
# POI or site nearby, joins no neighbour and takes no ring fallback (section 6)
edu_nojoin = edu_carriers.no_join(_edu, uo[uo['no_unit_reason'].eq('') & uo['zone_idx'].notna()],
                                  set(zones.index[zones[col('target', 'School')].eq(0)]))
uo['edu_nojoin'] = uo['unit_id'].isin(edu_nojoin['unit_id'])
# the Kita weights (section 5): the Kita part of a school, a Kita polygon and a squeezed Kita POI weigh their own size
_FS = {'GROUND': 1.0, 'ONE': 1.0, 'LOW2': 2.0, 'CAP2': 2.0, 'CAP3': 3.0}
def _fam_storeys(fam, bid):
    m_ = re.match(r'SOLE (\d+)', str(fam))
    return float(m_.group(1)) if m_ else _FS.get(fam, float(_N[bid]))
edu_kita = edu_carriers.kita_weights(_edu, uo[uo['no_unit_reason'].eq('')], fam_storeys=_fam_storeys,
                                     kita_cap_m2=float(POINT_SIZE_CAP_M2['childcare']), n_active=shared['n_active'].to_dict(),
                                     even_split=REDIST_MULTI_CATEGORY_SPLIT != 'full')
_ek = edu_kita.set_index('unit_id')
uo['kita_w'] = uo['unit_id'].map(_ek['kita_weight'])                   # NaN: O / k_red as usual
_m = uo['unit_id'].isin(EDU_CARRIER_LISTS['kita_cap']) & uo['labels'].map(lambda t: 'childcare' in t)   # a listed Kita: at most the cap
uo.loc[_m, 'kita_w'] = np.minimum(uo.loc[_m, 'O'] / uo.loc[_m, 'k_red'].clip(lower=1), float(POINT_SIZE_CAP_M2['childcare']))
uo['school_w_add'] = uo['unit_id'].map(_ek['to_school']).fillna(0.0)
uo['kita_rule'] = uo['unit_id'].map(_ek['rule']).fillna('')

# --- 4.6b services and the rest units: a service carries no retail_non_daily
# (phone shops, post counters and the listed shops that also sell goods aside; also the rest unit of a building whose asked POIs are
# all services); a Retail or Leisure rest label goes where an active asked POI of the building is its evident source and step 07
# labelled that POI otherwise ("the POI's answer wins"). A dropped label's share goes to the unit's other labels (the counts move)
_lr_sv = label_rules.services(uo, pois, _sole, _b['n_pois'], LABEL_RULES['services'], active_p=_lr_act)
uo = label_rules.drop_on_units(uo, _lr_sv, LABEL_RULES['services']['drop'], 'services', RED_LABELS, LLM_ACTIVITY_LABELS)
lr_rest_drops, lr_rest_table, lr_source_by_use, lr_source_by_group = label_rules.rest_sources(
    uo, bld, _sole, pois, pairs06, LABEL_RULES['rest_sources'], active_p=_lr_act, non_occupant_p=set(pois.loc[pois['non_occupant'], 'poi_id']))
_lr_rd = {}
for u_, l_ in lr_rest_drops:
    _lr_rd.setdefault(u_, set()).add(l_)
uo = label_rules.drop_on_units(uo, uo['unit_id'].isin(_lr_rd), None, 'rest_sources', RED_LABELS, LLM_ACTIVITY_LABELS, per_unit=_lr_rd,
                               reason='rest: its POIs answer for its labels')
_rt = lr_rest_table
_rs = LABEL_RULES['rest_sources']
_svk, _svp, _svr = LABEL_RULES['services'], pois.set_index('poi_id'), []      # the service units that keep it, by reason
for k_, b_, u_ in uo.loc[uo['no_unit_reason'].eq('') & uo['labels'].map(lambda t: 'retail_non_daily' in t),
                         ['kind', 'building_id', 'unit_id']].itertuples(index=False):
    if k_ == 'poi':
        g_, us_, p_ = _svp.at[u_, 'group'], _svp.at[u_, 'poi_use'], u_
    elif k_ == 'building' and _b.at[b_, 'n_pois'] == 1 and b_ in _sole.index:
        g_, us_, p_ = _sole.at[b_, 'group'], _sole.at[b_, 'poi_use'], _sole.at[b_, 'poi_id']
    else:
        continue
    if g_ in _svk['groups'] and (us_ in _svk['keep_uses'] or p_ in _svk['keep_poi_ids']):
        _svr.append('listed shops' if p_ in _svk['keep_poi_ids'] else us_)
print(f'  ok  services carry no Retail_Non-Daily: {int(_lr_sv.sum())} units lose it (' + ' · '.join(f'{k_} {v_}' for k_, v_ in uo.loc[_lr_sv, 'kind']
      .value_counts().items()) + f'); {len(_svr)} keep it: ' + ' · '.join(f'{k_} {v_}' for k_, v_ in pd.Series(_svr, dtype=object).value_counts().items()))
print(f'  ok  the POI\'s answer wins: {len(_rt):,} Retail / Leisure rest labels on {_rt["unit_id"].nunique():,} rest units - '
      f'{int(_rt["kind"].eq("A").sum())} with an evident source that step 07 labelled otherwise go ('
      + ' · '.join(f'{k_} {v_}' for k_, v_ in _rt.loc[_rt['kind'].eq('A'), 'label'].value_counts().items()) + '); kept: '
      f'{int(_rt["kind"].eq("B").sum())} carried by a non-occupant POI, {int(_rt["kind"].eq("C").sum())} with the building\'s own evidence, '
      f'{int(_rt["kind"].eq("D").sum())} without a source (keep all); {int(uo["no_unit_reason"].eq("rest: its POIs answer for its labels").sum())} '
      f'rest units keep no label. Evident sources (step 06 gives the label to at least {_rs["min_rate"]:.0%} of the one-POI buildings of the use, '
      f'of the group below {_rs["min_n"]} such buildings): '
      + ' · '.join(f'{l_} {int(((lr_source_by_use["n"] >= _rs["min_n"]) & (lr_source_by_use[l_] >= _rs["min_rate"])).sum())} uses' for l_ in _rs['labels']))

# --- 4.7 fuel stations: a canopy forms no remainder; a canopy with a station shop gives the shop its
# labels and keeps no unit (lib/storey_evidence.py)
uo, canopy_log = storey_evidence.canopy_units(uo, bld, set(bld.loc[storey_canopy, 'building_id']), station_pairs,
                                              FLOOR_ONE_STOREY_RULES['canopy_no_unit_ids'], RED_LABELS)
# a fuel station whose data show no shop carries errands instead of retail_daily, also where a canopy passed it on (4.4c)
# a canopy without a station shop (it keeps its unit) occupies at most the fuel group's cap
_sz_can = (uo['no_unit_reason'].eq('') & uo['building_id'].isin(set(bld.loc[storey_canopy, 'building_id']))
           & uo['kind'].isin(['building', 'rest', 'poi']))
size_canopy_log = list(zip(uo.loc[_sz_can, 'unit_id'], uo.loc[_sz_can, 'O']))
_SZ_CAN = float(POINT_SIZE_CAP_M2[UNIT_SIZE_RULES['canopy_cap_group']])
uo.loc[_sz_can, 'O'] = np.minimum(uo.loc[_sz_can, 'O'], _SZ_CAN)
uo.loc[_sz_can, 'O_pre'] = np.minimum(uo.loc[_sz_can, 'O_pre'], _SZ_CAN)
_lr_fs = label_rules.fuel_no_shop(uo, lr_no_shop_b, lr_no_shop_p)
uo = label_rules.drop_on_units(uo, _lr_fs, ('retail_daily',), 'fuel', RED_LABELS, LLM_ACTIVITY_LABELS, add=LABEL_RULES['fuel']['no_shop_add'])
print(f'  ok  fuel stations without a shop: {int(_lr_fs.sum())} units carry errands instead of retail_daily')

# --- 4.8 staff catering ------------------------------------------------------------------------------------------------------
# works canteens, staff cafes and snack bars on company sites, school and university Mensas and hospital cafeterias carry none of
# retail_daily / retail_non_daily / leisure: their guests are the staff, pupils and students counted as Workers, School or
# University (lib/staff_catering.py, STAFF_CATERING_RULES); the rest unit of such a building loses retail_daily and leisure too.
# A dropped label's share goes to the unit's other labels; the last-carrier guard keeps a zone's only carrier
_sc_store = set(LABEL_RULES['eating_places']['store_classes'])
staff_catering_units = staff_catering.staff_catering(uo, bld, pois, _sole, poi_attrs, STAFF_CATERING_RULES,
                                                     eating_uses=set(LABEL_RULES['eating_places']['uses']), store_classes=_sc_store,
                                                     head_split=HALL_TYPE_HEAD_SPLIT, pairs06=pairs06)
# staff shops (STAFF_CATERING_RULES['shops']): a shop or kiosk unit in a building filed
# in a works class on a named company site, not public by its own words, serves the plant's staff like the canteen - it joins staff
# catering, and its visitor values count 0 like the staff-only places' (section 8)
staff_shop_units = staff_catering.staff_shops(uo, bld, pois, _sole, STAFF_CATERING_RULES, store_classes=_sc_store,
                                              head_split=HALL_TYPE_HEAD_SPLIT, pairs06=pairs06)
staff_shop_units = staff_shop_units[~(staff_shop_units['kind'] + '|' + staff_shop_units['unit_id'])
                                    .isin(set(staff_catering_units['kind'] + '|' + staff_catering_units['unit_id']))].reset_index(drop=True)
staff_shop_keys = (set(zip(staff_shop_units['kind'], staff_shop_units['unit_id'])) if STAFF_CATERING_RULES['shops']['visitors_zero']
                   else set())
staff_catering_units = pd.concat([staff_catering_units, staff_shop_units], ignore_index=True)
staff_catering_rest = staff_catering.rest_catering(uo, bld, staff_catering_units, STAFF_CATERING_RULES, store_classes=_sc_store,
                                                   head_split=HALL_TYPE_HEAD_SPLIT)
_sck = uo['kind'] + '|' + uo['unit_id']
_scd = tuple(STAFF_CATERING_RULES['drop'])
_m = (uo['no_unit_reason'].eq('') & _sck.isin(set(staff_catering_units['kind'] + '|' + staff_catering_units['unit_id']))
      & uo['labels'].map(lambda t: bool(set(t) & set(_scd))))
uo = label_rules.drop_on_units(uo, _m, _scd, 'staff_catering', RED_LABELS, LLM_ACTIVITY_LABELS)
_m2 = (uo['no_unit_reason'].eq('') & _sck.isin(set('rest|' + staff_catering_rest['unit_id']))
       & uo['labels'].map(lambda t: bool(set(t) & set(STAFF_CATERING_RULES['rest_labels']))))
uo = label_rules.drop_on_units(uo, _m2, tuple(STAFF_CATERING_RULES['rest_labels']), 'staff_catering_rest', RED_LABELS, LLM_ACTIVITY_LABELS,
                               reason='rest: its staff catering answers for its labels')
print(f'  ok  staff catering: {len(staff_catering_units):,} units (' + ' · '.join(f'{k_} {v_}' for k_, v_ in staff_catering_units['where']
      .value_counts().items()) + f'; by their words {int(staff_catering_units["how"].eq("words").sum())}, on a company site '
      f'{int(staff_catering_units["how"].eq("site").sum())}, staff shops inside a plant '
      f'{int(staff_catering_units["how"].eq("shop").sum())}); {int(_m.sum())} lose retail / leisure labels, {int(_m2.sum())} rest units of '
      'staff-catering buildings lose retail_daily / leisure')

units_ok = uo['no_unit_reason'].eq('')
_bad = units_ok & ~(uo['O'] > 0)                 # NaN fails too
if _bad.any():
    raise AssertionError(f'{int(_bad.sum())} unit(s) without a positive occupied floor area, e.g. '
                         f'{uo.loc[_bad, ["unit_id", "kind", "O", "family"]].head(5).to_dict("records")}')

# --- what this gives -------------------------------------------------------------------------------------
_carried = set(uo.loc[uo['no_unit_reason'].eq('canopy: the station shop carries it'), 'building_id'])
_can_left = sorted(c for c in set(station_pairs) - _carried if (uo['building_id'].eq(c) & uo['no_unit_reason'].eq('')).any())
print(f'  ok  fuel stations: {int(uo["no_unit_reason"].eq("canopy: no remainder").sum())} canopy remainder(s) left out; '
      f'{len(_carried)} canopies keep no unit, their {len({station_pairs[c] for c in _carried})} station shops carry them '
      f'({len(canopy_log)} shops gain labels or evidence, {int(canopy_log["now_with_evidence"].sum())} of them now a place with evidence); '
      + (f'{len(_can_left)} paired canopies keep their unit, their shop has none ({", ".join(_can_left[:5])}); ' if _can_left else '')
      + f'{int(uo.loc[uo["rule"].eq("canopy_no_unit"), "building_id"].nunique())} listed canopy without a unit')
print(f'  ok  {int(units_ok.sum()):,} units: ' + ' · '.join(f'{k} {v:,}' for k, v in uo.loc[units_ok, 'kind'].value_counts().items())
      + f'; {(uo.loc[units_ok & uo["kind"].ne("rest"), "O"].sum()) / 1e6:,.2f} km2 occupied of {bld.loc[bld["vacancy"].eq(""), "floor_area_m2"].sum() / 1e6:,.2f} '
      'km2 of floor area (rest units lie on the floor their building\'s POIs leave free, or take an average occupant\'s share)')
_szl = pd.DataFrame(size_log, columns=['building_id', 'poi_id', 'before_m2', 'after_m2']).drop_duplicates('poi_id', keep='last')
print(f'  ok  the sizes of POI units: {len(_szl):,} POI units resized by the typical size of their group or format '
      f'({_szl["before_m2"].sum() / 1e3:,.1f} -> {_szl["after_m2"].sum() / 1e3:,.1f} thousand m2; {len(size_dup_log):,} points that repeat a polygon '
      f'of their building left alone); pharmacies at most {_SZ_PH:,.0f} m2{_sz_ph_txt}; '
      f'{len(size_chain_ids & set(_small_cap)):,} buildings filed as small shops or restaurants that name a big-box chain up to '
      f'their format\'s size (else {POINT_SIZE_CAP_M2[UNIT_SIZE_RULES["chain_cap_group"]]:,.0f} m2); {_sz_ki_txt}chain tenants '
      f'weigh their published size in the {len(MALL_RETAIL_FLOOR)} listed malls and count at least it in every mall; {len(size_canopy_log):,} canopies without a station shop at most {_SZ_CAN:,.0f} m2')
_p = uo[units_ok & uo['kind'].eq('poi')]
print(f'  ..  POI units by floor family: ' + ' · '.join(f'{k} {v:,}' for k, v in _p['family'].str.split(' ').str[0].value_counts().items()))
print(f'  ..  shared buildings: {len(shared):,}; one facility (all asked POIs one group, sharing the sole-occupant floor) '
      f'{int(shared["one_facility"].sum()):,}; points capped at their size {int(shared["n_capped"].sum()):,} '
      f'({shared["freed_m2"].sum() / 1e3:,.1f} thousand m2 freed)')
_rem = uo[units_ok & uo['kind'].eq('remainder')]
_remo = _rem['f'].eq(REDIST_REMAINDER_WORKERS_FACTOR) & _rem['building_id'].map(_b['bosserhof_class']).isin(REDIST_REMAINDER_OFFICE_CLASSES)
print(f'  ..  remainder units (Workers only): {len(_rem):,}, {_rem["O"].sum() / 1e6:,.3f} km2 - at the office factor '
      f'{REDIST_REMAINDER_WORKERS_FACTOR} in {int(_remo.sum()):,} shop and office buildings ({_rem.loc[_remo, "O"].sum() / 1e6:,.3f} km2), '
      f'at the building\'s own factor in {int((~_remo).sum()):,} others ({_rem.loc[~_remo, "O"].sum() / 1e6:,.3f} km2)')
_rl = uo.loc[units_ok & uo['kind'].eq('rest'), 'labels']     # the labels the rest units carry now (after 4.6b and 4.7)
print(f'  ..  rest units: {len(_rl):,} carry {sum(map(len, _rl)):,} building labels no asked POI carries ('
      + ' · '.join(f'{k} {v}' for k, v in pd.Series([l for L in _rl for l in L], dtype=object).value_counts().items()) + ')')
_nu = uo[~units_ok]
print('  ..  no unit: ' + ' · '.join(f'{k} {v:,}' for k, v in _nu['no_unit_reason'].value_counts().items())
      + f'; vacant {len(_vac_b):,} buildings and {len(_vac_p):,} POIs')
_wz = uo[uo['rule'].isin(['church', 'volunteer_fire', 'sports_hall', 'parking']) & uo['labels_pre'].map(lambda t: 'work' in t)]
_wzp = _wz.loc[_wz['kind'].eq('poi'), 'rule'].value_counts()
_kp = int((_brule.eq('sports_hall') & uo['kind'].eq('poi') & _pgrp.isin(SPORTS_HALL_POI_GROUPS) & _pkeep).sum())
print(f'  ..  Workers 0: ' + ' · '.join(f'{k} {v:,} units ({int(_wzp.get(k, 0)):,} of them POI units)' for k, v in _wz['rule'].value_counts().items())
      + f'; POI units of a sports hall that keep their Workers by the keep list: {_kp}'
      + f'; no-evidence residential: {int(uo["rule"].eq("no_evidence").sum()):,} units on footprint x 1 '
      f'({int((uo["rule"].eq("no_evidence") & uo["labels"].map(len).gt(1)).sum())} keep their labels)')
_cb = [u for u in _cut if u in _b.index]
print(f'  ..  small places stay small: {len(_cb):,} building units capped (small-scale retail or restaurants at '
      + ' / '.join(f'{POINT_SIZE_CAP_M2[g]:,.0f} m2' for g in SMALL_PLACE_CAP_GROUPS.values())
      + '; a pharmacy at its cap, a chain at its format\'s size or the big-box cap)' + f', {len(_cut) - len(_cb):,} point POIs of '
      f'these groups as one facility or main occupant; {sum(_cut.values()) / 1e3:,.0f} thousand m2 cut, carrying nothing; '
      f'{int(_exempt_b.sum()):,} food-store buildings of these classes keep their size')
print(f'  ..  places without evidence: {int((units_ok & uo["no_poi"]).sum()):,} building units without a POI, a name or a site other than landuse count '
      f'{REDIST_NO_POI_WEIGHT} in {", ".join(REDIST_NO_POI_CATEGORIES)}')
print('  ..  who carries School, University and Kindergarten - the label dropped: '
      + ' · '.join(f'{c} {int((edu_drops["category"] == c).sum())} units ('
                   + ', '.join(f'{r} {n}' for r, n in edu_drops.loc[edu_drops['category'] == c, 'rule'].value_counts().sort_index().items()) + ')'
                   for c in ('School', 'University', 'Kindergarten'))
      + f'; {int(uo["edu_nojoin"].sum())} School place(s) in zones with a School target of 0 join no neighbour (only the cadastre or a bare '
      f'OSM school outline, no school POI or site within {EDU_CARRIER_RULES["near_school_m"]:,.0f} m); the Kita weights: '
      + ' · '.join(f'{r} {n}' for r, n in edu_kita['rule'].value_counts().sort_index().items()))
print('        S-C1 former school · S-C2 village house · S-C3 adult / out-of-school education · S-C4 Kita building · S-C5 no evidence at all '
      '· S-C6 another occupant of a school site · S-N a non-teaching part (Mensa, boarding house, office, toilets) · K1 / K1s social '
      'facility · K2 not a Kita by name (home, youth club, elderly day care ...) · K3 a Kita lot on another use · K4 no evidence · K6 an '
      'all-day school programme · U2 continuing education · U4 no evidence · U5 a rest unit without a university building · U6 a '
      'non-teaching part (Mensa, administration, computing, events ...) · U9 a residential building on a campus site, and the list')
with pd.option_context('display.max_colwidth', 100):
    print(edu_drops.assign(name=edu_drops['building_id'].map(_b['name']))[['category', 'rule', 'unit_id', 'name', 'evidence']]
          .sort_values(['category', 'rule', 'unit_id']).to_string(index=False))
    if len(edu_nojoin):
        print(edu_nojoin.to_string(index=False))
    if len(edu_kita):
        print(edu_kita.to_string(index=False))

  ok  office buildings in an industry class take the normal-office factor 2.9: 21 (DENIAL0100000T1s, DENIAL0100001ene, DENIAL0100003vwD, DENIAL0100004ZUf, DENIAL01000050Fa, DENIAL01000050In, DENIAL01000050T7, DENIAL01000050pC Siemens AG ...)


  ok  POI groups for 7,864 asked POIs (small_retail 1,853 · gastronomy 983 · health_practice 604 · office 525 · personal_service 478 · counter 421 · school 384 · sport 292 ...); a POI-level class, and so its own Workers factor, for 7,594 of them, the building's factor for the 270 generic / unmapped
  ok  the point size caps are the p90 of the mapped polygons of each group: gastronomy 438 m2 (137) · small_retail 748 m2 (298) · personal_service 187 m2 (38) · counter 388 m2 (33) · health_practice 319 m2 (30) · office 1,540 m2 (52) · community 779 m2 (37) · small_school 692 m2 (5) · big_box 4,774 m2 (35) · care_home 1,510 m2 (25) · childcare 454 m2 (29) · craft 278 m2 (21) · culture 1,112 m2 (30) · fire_rescue 444 m2 (24) · food_store 1,771 m2 (25) · fuel 181 m2 (12) · generic 2,460 m2 (62) · hospital 2,053 m2 (20) · hotel 466 m2 (42) · industry 12,074 m2 (168) · public_office 959 m2 (39) · school 2,209 m2 (145) · sport 1,531 m2 (122) · unmapped 1,267 m2 (34) · vehicle 492 m2 (30) · worshi

  ok  buildings that no longer stand: 78 of the 78 listed give no unit (134.2 thousand m2 of floor area)
  ok  places known to be closed on 2026-10-09 (CLOSED_BUILDINGS): 20 of the 23 listed give no unit (77.3 thousand m2 of floor area), and 1 asked POI(s) in them; kept open: way/243689994; left out already by another sign: DENIAL01000051AX, DENIAL0100006qPi, DENIAL9100006RDd; the planned reopening year has come, re-check: DENIAL9100006RDd
  ..  vacant places left out: 420 buildings ({'polygon': 133, 'osm+model': 96, 'gone': 78, 'osm': 24, 'model': 23, 'osm+polygon': 21, 'closed': 20, 'osm+polygon+model': 17, 'polygon+model': 8}; 592,785 m2 of floor) and 20 asked POI(s) ({'tag': 18, 'closed': 1, 'name+model': 1}):
      building_id           vacancy                                           name                                                                                                                                                                                                  

  ok  landuse pairs re-split away: 180 pairs on 179 buildings held 292 thousand m2 of floor; 43 vacant pairs on 39 buildings keep an empty GROUND slot
  ok  non-occupants dropped (by tag): 73 asked POIs on 48 buildings (shower 32 · public_bookcase 10 · photo_booth 7 · luggage_locker 6 · information 3 · reception_desk 3 · bleachers 3 · washing_machine 2 · dressing_room 2 · give_box 2 · kitchen 1 · clothes_dryer 1 · vending_machine 1); 7 buildings are left without an asked POI and become building units; kept although tagged so: 32 (a staffed tourist office, an occupant key such as shop=chemist, or vacant)
  ok  the no-evidence residential rule also takes 0 buildings whose only sites are landuse areas


  ok  city volunteer fire stations by their own text: 31 more, 1 left out as their zone's only Workers carrier (DENIAL01000036uK); parking garages and decks: 37
  ok  the single-category rules (vacant buildings left out; a building takes the first rule that applies, in this order):
        holiday_flat       384 buildings (384 in a zone); the rule applies to 384
        dormitory           21 buildings (21 in a zone); the rule applies to 21
        no_evidence      1,762 buildings (1,752 in a zone); the rule applies to 1,743
        church           1,031 buildings (1,026 in a zone); the rule applies to 1,027
        volunteer_fire     553 buildings (551 in a zone); the rule applies to 551
        sports_hall      1,092 buildings (1,080 in a zone); the rule applies to 1,090
        parking             37 buildings (37 in a zone); the rule applies to 36


...

  ok  the labels of steps 06 and 07 reconciled by OSM use (LABEL_RULES) - units relabelled, and the labels they gain (+) and lose (-):
        fuel             90 (building 79 · poi 11): +retail_daily 89 · +errands 1 · -retail_daily 1; left alone: kept 1
        pharmacies       46 (poi 46): +retail_daily 46
        daily_shops      63 (poi 44 · building 19): +retail_daily 61 · -retail_non_daily 63; left alone: kept 11
        eating_places   265 (building 250 · poi 15): +leisure 189 · -retail_daily 156 · -retail_non_daily 110; left alone: kept 42
        venues           37 (building 31 · poi 5 · rest 1): +leisure 37; left alone: excluded 19 · skipped 4
        no_reopen        21 (rest 21): -retail_non_daily 16 · -retail_daily 5
        fuel stations whose data show no shop: 37 one-POI buildings and 17 asked POIs (errands instead of retail_daily)


  ok  the floor of the malls: 7 malls at most their published retail floor, 285.3 -> 148.6 thousand m2 (1 outline POI(s) keeping the 13,238 m2 their tenants leave); 8 remainder units (4,074 m2) of their floor beyond it carry nothing


  ok  services carry no Retail_Non-Daily: 125 units lose it (building 80 · poi 38 · rest 7); 32 keep it: telecommunication 20 · post_office 6 · listed shops 6
  ok  the POI's answer wins: 192 Retail / Leisure rest labels on 188 rest units - 125 with an evident source that step 07 labelled otherwise go (retail_daily 57 · retail_non_daily 53 · leisure 15); kept: 3 carried by a non-occupant POI, 6 with the building's own evidence, 58 without a source (keep all); 110 rest units keep no label. Evident sources (step 06 gives the label to at least 25% of the one-POI buildings of the use, of the group below 5 such buildings): retail_daily 16 uses · retail_non_daily 68 uses · leisure 50 uses


  ok  fuel stations without a shop: 33 units carry errands instead of retail_daily


  ok  staff catering: 93 units (works 58 · school 21 · university 12 · hospital 2; by their words 69, on a company site 16, staff shops inside a plant 8); 77 lose retail / leisure labels, 6 rest units of staff-catering buildings lose retail_daily / leisure
  ok  fuel stations: 1 canopy remainder(s) left out; 55 canopies keep no unit, their 53 station shops carry them (36 shops gain labels or evidence, 26 of them now a place with evidence); 1 listed canopy without a unit
  ok  44,679 units: building 35,552 · poi 7,663 · remainder 1,143 · rest 321; 43.07 km2 occupied of 47.25 km2 of floor area (rest units lie on the floor their building's POIs leave free, or take an average occupant's share)
  ok  the sizes of POI units: 1,135 POI units resized by the typical size of their group or format (237.6 -> 459.5 thousand m2; 182 points that repeat a polygon of their building left alone); pharmacies at most 307 m2 (the easyApotheke format 400 m2); 207 buildings filed as small shops or restaurants

## 5. The weights

One row per **unit and redistributed category**: Workers `O × f`, each other
category `O / k_red` (`k_red` the unit's redistributed non-work labels; a
rest unit the building's); a building without evidence — no POI of its own,
no name, no site other than a landuse area — weighs a quarter of that in
Retail_Daily, Retail_Non-Daily and Leisure. **Kita weights**: the Kita part of a school weighs at most the Kita's own polygons
(else the childcare point cap) and gives the rest to School on the same unit
where it carries School; a Kita polygon weighs at most its area × its storeys
(split over its labels like any weight) where the building's Kita polygons
cover less than 80 % of it, and a Kita point next to such a polygon
is the same facility; a squeezed Kita POI weighs its own extent. The **school
types** also follow the label fixes, the OSM isced / grades tags, the
Oskar-Kämmer sites and the vocational campuses. The five
labels without a zone total give an
**area** per unit, `O / k_all` (`k_all` all its non-work labels), and feed no
category. A label the label rules take off leaves both counts, so its share goes
to the unit's other labels; an education drop keeps them. The **last-carrier guard**: where a rule on a single category, a
label rule (or a no-unit rule) leaves a zone with a target but no carrier of a
category, the
units the rule removed there keep that category at their weight before the
rule — not where only the label mapping left the zone without a carrier.
Checked: a positive floor area and weight on every unit (a missing value
fails too); a building's units and slots within its floor area, its
ground-floor units within one footprint; no unit for a vacant place, a holiday
flat or a dormitory; no Workers on a Workers-0 unit the guard did not restore
(parking garages included); every weight from O, f and k; every POI factor from
its class (the building's for an outline POI); the five area
labels in no category.

In [6]:
def _rows(df, pre=False):
    """One row per unit and redistributed category, with its weight (O x f, or O / k_red)."""
    sfx = '_pre' if pre else ''
    O, k = df['O' + sfx].to_numpy(dtype=float), df['k_red' + sfx].to_numpy(dtype=float)
    f = df['f'].to_numpy(dtype=float) if REDIST_WORKERS_WEIGHT == 'floor_x_bosserhof' else np.ones(len(df))
    g = np.where(df['no_poi'].to_numpy(dtype=bool), REDIST_NO_POI_WEIGHT, 1.0)     # places without evidence
    kw = np.full(len(df), np.nan) if pre else df['kita_w'].to_numpy(dtype=float)      # the Kita weights
    sa = np.zeros(len(df)) if pre else df['school_w_add'].to_numpy(dtype=float)
    out = []
    for i, (bid, uid, kind, labels) in enumerate(zip(df['building_id'], df['unit_id'], df['kind'], df['labels' + sfx])):
        if not O[i] > 0:
            continue
        for l in labels:
            if l == 'work':
                out.append((bid, uid, kind, l, 'Workers', O[i] * f[i]))
            elif l in MID_LABEL_TO_ACTIVITY:
                c_ = MID_LABEL_TO_ACTIVITY[l]
                w_ = (O[i] if REDIST_MULTI_CATEGORY_SPLIT == 'full' else O[i] / k[i]) * (g[i] if c_ in REDIST_NO_POI_CATEGORIES else 1.0)
                if c_ == 'Kindergarten' and not np.isnan(kw[i]):
                    w_ = kw[i]                          # the Kita part of a school, a Kita polygon, a squeezed Kita POI
                elif c_ == 'School':
                    w_ += sa[i]                         # the floor a Kita part of a school gives up
                if w_ > 0:
                    out.append((bid, uid, kind, l, c_, w_))
    out = pd.DataFrame(out, columns=['building_id', 'unit_id', 'kind', 'label', 'category', 'weight'])
    out['zone_idx'] = out['building_id'].map(_b['zone_idx'])
    return out

units = _rows(uo[units_ok])
units['restored'] = False
# the areas of the five labels without a zone total: O / k_all, never redistributed
_ar = [(b_, u_, k_, l, o / ka) for b_, u_, k_, ls, o, ka in uo.loc[units_ok, ['building_id', 'unit_id', 'kind', 'labels', 'O', 'k_all']].itertuples(index=False)
       for l in ls if l in AREA_LABELS]
areas = pd.DataFrame(_ar, columns=['building_id', 'unit_id', 'kind', 'label', 'area_m2'])

# --- the last-carrier guard -----------------------------------------------------------------------------
_pre = _rows(uo[uo['rule'].ne('') | uo['edu_rule'].ne('') | uo['kita_rule'].ne('') | uo['label_rule'].ne('')], pre=True)   # the education rules and the label rules too
_cand = _pre[~(_pre['unit_id'] + '|' + _pre['category']).isin(units['unit_id'] + '|' + units['category'])]   # rows a rule removed
_restore = []
if REDIST_LAST_CARRIER_GUARD:
    for c in CATS:
        _tgt = zones[col('target', c)]
        _have = set(units.loc[(units['category'] == c) & units['zone_idx'].notna(), 'zone_idx'].astype(int))
        _cc = _cand[(_cand['category'] == c) & _cand['zone_idx'].notna()]
        _lost = [z for z in _cc['zone_idx'].astype(int).unique() if _tgt[z] > 0 and z not in _have]
        _restore.append(_cc[_cc['zone_idx'].astype(int).isin(_lost)])
_restore = pd.concat(_restore, ignore_index=True) if _restore else _cand.iloc[0:0]
# a label a label rule took off and the guard gives back counts again in its unit's split: the unit's counts take it
# back in, so its other rows and areas shrink to their share with that label, and the restored row takes the same split
_edl = set(zip(edu_drops['unit_id'], edu_drops['label'])) if len(edu_drops) else set()      # education drops keep their share
_gb = _restore[_restore['category'].ne('Workers') & _restore['unit_id'].isin(uo.loc[uo['label_rule'].ne(''), 'unit_id'])
               & pd.Series([(u_, l_) not in _edl for u_, l_ in zip(_restore['unit_id'], _restore['label'])], index=_restore.index, dtype=bool)]
lr_guard_back = _gb.groupby('unit_id')['label'].agg(set).to_dict()
lr_guard_back_rows = set(zip(_gb['unit_id'], _gb['label']))
if lr_guard_back:
    _gi = uo.index[uo['unit_id'].isin(lr_guard_back)]
    uo.loc[_gi, 'k_red'] = [k_ + sum(l_ in RED_LABELS for l_ in lr_guard_back[u_]) for u_, k_ in zip(uo.loc[_gi, 'unit_id'], uo.loc[_gi, 'k_red'])]
    uo.loc[_gi, 'k_all'] = [k_ + len(lr_guard_back[u_]) for u_, k_ in zip(uo.loc[_gi, 'unit_id'], uo.loc[_gi, 'k_all'])]
    _gm = units_ok & uo['unit_id'].isin(lr_guard_back)
    units = pd.concat([units[~units['unit_id'].isin(lr_guard_back)], _rows(uo[_gm]).assign(restored=False)], ignore_index=True)
    areas = pd.concat([areas[~areas['unit_id'].isin(lr_guard_back)],
                       pd.DataFrame([(b_, u_, k_, l, o / ka) for b_, u_, k_, ls, o, ka in uo.loc[_gm, ['building_id', 'unit_id', 'kind', 'labels', 'O', 'k_all']]
                                     .itertuples(index=False) for l in ls if l in AREA_LABELS], columns=areas.columns)], ignore_index=True)
    _gt = uo.loc[_gi].copy()
    _gt['labels_pre'] = [tuple(l_ for l_ in LLM_ACTIVITY_LABELS if l_ in lr_guard_back[u_]) for u_ in _gt['unit_id']]
    _gt['k_red_pre'] = _gt['k_red']
    _gr = _rows(_gt, pre=True)
    _restore = pd.concat([_restore[~pd.Series([(u_, l_) in lr_guard_back_rows for u_, l_ in zip(_restore['unit_id'], _restore['label'])],
                                              index=_restore.index, dtype=bool)], _gr], ignore_index=True)
units = pd.concat([units, _restore.assign(restored=True)], ignore_index=True)
uo['restored'] = uo['unit_id'].isin(_restore['unit_id'])
units['assigned'] = 0.0
if len(_restore):
    _ru = uo.set_index('unit_id')                # the rule that removed the row: one on a single category or a who-carries rule
    _rr = _ru['rule'].where(_ru['rule'].ne(''), _ru['edu_rule'].where(_ru['edu_rule'].ne(''),
                                                         _ru['kita_rule'].where(_ru['kita_rule'].ne(''), _ru['label_rule'])))
    _rz = _restore.assign(rule=_restore['unit_id'].map(_rr),
                          zone=_restore['zone_idx'].astype(int).map(zones[ZONE_ID_COL]))
    print(f'  ok  last-carrier guard: {len(_restore)} row(s) restored at their weight before the rule, where a rule left a zone with a '
          f'target but no carrier ({len(lr_guard_back_rows)} label(s) a label rule took off count again in the split of their unit):')
    print(_rz[['zone', 'category', 'rule', 'unit_id', 'kind', 'weight']].round({'weight': 1}).to_string(index=False))
else:
    print('  ok  last-carrier guard: no rule leaves a zone with a target but no carrier')

# --- checks -----------------------------------------------------------------------------------------------
if uo['unit_id'].duplicated().any():
    raise AssertionError(f'two units with one id, e.g. {uo.loc[uo["unit_id"].duplicated(), "unit_id"].head(3).tolist()}')
if not (units['weight'] > 0).all() or not (uo.loc[units_ok, 'O'] > 0).all():          # NaN fails too
    raise AssertionError('a unit without a positive occupied floor area or weight')
_occ = uo[units_ok & uo['kind'].ne('rest')].groupby('building_id')['O'].sum().add(slots.groupby('building_id')['O'].sum(), fill_value=0.0)
if (_occ > _occ.index.map(_FA) * (1 + 1e-6)).any():
    raise AssertionError(f'the units (and slots) of {int((_occ > _occ.index.map(_FA) * (1 + 1e-6)).sum())} buildings occupy more than the floor area')
if (uo.loc[units_ok & uo['kind'].eq('rest'), 'O'] > uo.loc[units_ok & uo['kind'].eq('rest'), 'building_id'].map(_FA) * (1 + 1e-6)).any():
    raise AssertionError('a rest unit above its building\'s floor area')
if (shared['ground_m2'] > shared['area_m2'] * (1 + 1e-6)).any():
    raise AssertionError(f'the ground-floor units of {int((shared["ground_m2"] > shared["area_m2"] * (1 + 1e-6)).sum())} buildings use more than one footprint')
_post = uo[uo['no_unit_reason'].eq('')]
_dw = ((_post['building_id'].isin(_vac_b) & _post['kind'].ne('poi')) | _post['unit_id'].isin(_vac_p)
       | _post['building_id'].map(_b['rule']).eq('holiday_flat')
       | (_post['building_id'].map(_b['rule']).eq('dormitory') & _post['kind'].isin(['building', 'rest', 'remainder']))
       | _post['unit_id'].isin(pois.loc[pois['slot'] != '', 'poi_id']))
if _dw.any():
    raise AssertionError(f'{int(_dw.sum())} unit(s) of a vacant place, a holiday flat or a dormitory')
_wr = units[(units['category'] == 'Workers') & units['unit_id'].isin(uo.loc[uo['rule'].isin(['church', 'volunteer_fire', 'sports_hall', 'parking']), 'unit_id'])]
if (~_wr['restored']).any():
    raise AssertionError('Workers on a unit under a Workers-0 rule that the guard did not restore')
_u = units.merge(uo[['unit_id', 'O', 'f', 'k_red', 'O_pre', 'k_red_pre', 'no_poi', 'kita_w', 'school_w_add']], on='unit_id', how='left')
_O = np.where(_u['restored'], _u['O_pre'], _u['O'])
_back = np.array([(u_, l_) in lr_guard_back_rows for u_, l_ in zip(_u['unit_id'], _u['label'])], dtype=bool)
_kr = np.where(_u['restored'] & ~_back, _u['k_red_pre'], _u['k_red']).astype(float)   # a label given back: its unit's split
_fw = _u['f'] if REDIST_WORKERS_WEIGHT == 'floor_x_bosserhof' else 1.0
_gw = np.where(_u['no_poi'].astype(bool) & _u['category'].isin(REDIST_NO_POI_CATEGORIES), REDIST_NO_POI_WEIGHT, 1.0)
_expect = np.where(_u['category'] == 'Workers', _O * _fw,
                   _O / (1.0 if REDIST_MULTI_CATEGORY_SPLIT == 'full' else np.maximum(_kr, 1.0)) * _gw)
_live = ~_u['restored'].astype(bool)
_expect = np.where(_live & _u['category'].eq('Kindergarten') & _u['kita_w'].notna(), _u['kita_w'], _expect)
_expect = np.where(_live & _u['category'].eq('School'), _expect + _u['school_w_add'].fillna(0.0), _expect)
if not np.allclose(_u['weight'], _expect, rtol=1e-12, atol=0):
    raise AssertionError('a unit weight differs from O x f (Workers) or O / k_red (the other categories; x REDIST_NO_POI_WEIGHT without evidence; '
                         'the Kita weights and what a Kita part gives School)')
_pu = uo[uo['kind'].eq('poi')]
_fexp = [factor(c) if isinstance(c, str) and u_ not in _BF_POIS else _f[b_] for u_, c, b_ in zip(_pu['unit_id'], _pu['unit_id'].map(pois.set_index('poi_id')['poi_class']), _pu['building_id'])]
_ru = uo[uo['kind'].eq('remainder')]
_rexp = np.where(_ru['building_id'].map(_b['bosserhof_class']).isin(REDIST_REMAINDER_OFFICE_CLASSES), REDIST_REMAINDER_WORKERS_FACTOR,
                 _ru['building_id'].map(_f))
if not np.allclose(_pu['f'], _fexp) or not np.allclose(_ru['f'], _rexp):
    raise AssertionError('a POI or remainder unit whose Workers factor is not the one of its POI-level class / the remainder factor')
if not units['label'].isin(list(MID_LABEL_TO_ACTIVITY)).all() or units['label'].isin(AREA_LABELS).any() or not areas['label'].isin(AREA_LABELS).all():
    raise AssertionError('an area label feeds a category, or a redistributed label has an area')
if units.loc[units['kind'] == 'building', 'building_id'].isin(_shared_b).any():
    raise AssertionError('a building with asked POIs is a unit itself')
_r = units[units['kind'] == 'rest']
if (_r['category'] == 'Workers').any() or any(l in _cover[b_] for b_, l in zip(_r['building_id'], _r['label'])):
    raise AssertionError('a rest unit carries Workers or a label one of its POIs carries')
print('  ok  every unit occupies a positive floor area; a building\'s units and slots occupy at most its floor area, its ground-floor '
      'units at most one footprint; no vacant place, holiday flat or dormitory has a unit; no Workers on a Workers-0 unit the guard did not '
      'restore; every weight recomputes from O, f and k (the weight of places without evidence, the Kita weights and what a Kita part '
      'of a school gives School included); every POI factor is its class\'s; '
      'the five area labels feed no category')
print(f'  ok  {len(units):,} unit-category rows on {units["unit_id"].nunique():,} units: '
      + ' · '.join(f'{k} {v:,}' for k, v in units['kind'].value_counts().items()))
_t = units.pivot_table(index='category', columns='kind', values='weight', aggfunc=['size', 'sum'], fill_value=0)
_t = pd.concat({'units': _t['size'], 'weight (M)': (_t['sum'] / 1e6).round(3)}, axis=1).reindex(CATS)
print(_t.to_string())
print('  ..  the areas of the labels without a zone total (thousand m2): '
      + ' · '.join(f'{l} {areas.loc[areas["label"] == l, "area_m2"].sum() / 1e3:,.0f} on {int((areas["label"] == l).sum()):,} units' for l in AREA_LABELS))

# --- the school types --------------------------------------------------------------------------------------------
# primary / secondary / vocational: from a POI's own name and step-07 type, else from its building's name, POI names, site
# names and step-05 type, else from the ALKIS function, else primary and secondary (SCHOOL_TYPE_PATTERNS, SCHOOL_TYPE_DEFAULT)
_sites = pairs06[pairs06['how'] == 'site'].groupby('building_id')['name'].agg(lambda s: list(dict.fromkeys(s.dropna())))
_pqs = pois.set_index('poi_id')
_sch = uo[uo['labels'].map(lambda t: 'school' in t) | uo['labels_pre'].map(lambda t: 'school' in t)]
_st = {}
def _tfix(t):                                   # the model's English type words (EDU_CARRIER_RULES['type_text_fixes'])
    if isinstance(t, str):
        for a_, b_ in EDU_CARRIER_RULES['type_text_fixes']:
            t = re.sub(a_, b_, t, flags=re.IGNORECASE)
    return t
for _uid, _bid, _kind in _sch[['unit_id', 'building_id', 'kind']].itertuples(index=False):
    _own = ([_pqs.at[_uid, 'name']], _tfix(_pqs.at[_uid, 'poi_type'])) if _kind == 'poi' else ([], None)
    _st[_uid] = floor_rules.school_types(_own[0], _own[1], [_b.at[_bid, 'name'], _b.at[_bid, 'poi_names']] + _sites.get(_bid, []),
                                         _tfix(_b.at[_bid, 'llm_type']), _b.at[_bid, 'function'], rx=_RX_SCHOOL,
                                         alkis_types=SCHOOL_TYPE_ALKIS, all_types=SCHOOL_TYPES, default_types=SCHOOL_TYPE_DEFAULT)
# who carries School: a vocational school that was labelled university is vocational; then the types the OSM
# isced / grades tags name, the Oskar-Kaemmer sites (vocational) and the vocational campuses (lib/edu_carriers.py)
for _uid, _bid, _kind in _sch[['unit_id', 'building_id', 'kind']].itertuples(index=False):
    _f = edu_forced_types.get(_uid) if _kind == 'poi' else (edu_forced_types.get(_bid) if _kind in ('building', 'rest') else None)
    if _f:
        _st[_uid] = (tuple(_f), 'label fix')
edu_type_fixes = edu_carriers.type_fixes(_edu, _sch[['unit_id', 'building_id', 'kind']], _st,
                                         school_type_rx=lambda n: bool(floor_rules._school_hits([n], None, _RX_SCHOOL)))
for _uid, (_t, _r, _e) in edu_type_fixes.items():
    _st[_uid] = (tuple(_t), _r)
uo['school_types'] = uo['unit_id'].map(lambda u: _st[u][0] if u in _st else ())
uo['school_type_from'] = uo['unit_id'].map(lambda u: _st[u][1] if u in _st else '')
units['school_types'] = [_st[u][0] if c == 'School' else () for u, c in zip(units['unit_id'], units['category'])]
units['no_join'] = (units['unit_id'].map(uo.set_index('unit_id')['edu_nojoin']).fillna(False).astype(bool)
                    & units['category'].eq('School') & ~units['restored'].astype(bool))      # section 6
_ss = units[units['category'] == 'School']
if _ss['school_types'].map(len).eq(0).any():
    raise AssertionError('a School unit without a school type')
_sx = pd.crosstab(_ss['unit_id'].map(uo.set_index('unit_id')['school_type_from']), _ss['school_types'].map(' + '.join))
print(f'  ok  the school types of the {len(_ss):,} School units (rows: where the type comes from):')
print(_sx.reindex(['name', 'building', 'label fix', 'T-isced', 'T-list', 'T-campus', 'ALKIS', 'none']).dropna(how='all').fillna(0).astype(int)
      .to_string())
if edu_type_fixes:
    _tf = pd.DataFrame([(u, '+'.join(t), r, e) for u, (t, r, e) in edu_type_fixes.items()], columns=['unit_id', 'type', 'rule', 'evidence'])
    print(f'  ..  school types from the evidence: ' + ' · '.join(f'{k} {v}' for k, v in _tf['rule'].value_counts().items()))
    with pd.option_context('display.max_colwidth', 90):
        print(_tf[['unit_id', 'type', 'rule', 'evidence']].to_string(index=False))

  ok  last-carrier guard: 49 row(s) restored at their weight before the rule, where a rule left a zone with a target but no carrier (39 label(s) a label rule took off count again in the split of their unit):
     zone         category           rule               unit_id     kind  weight
315700615          Workers         church      DENIAL6100002uCG building   152.6
315700615          Workers volunteer_fire      DENIAL6100002uCZ building   253.1
310118902           School        S-C5+U4      DENIAL0100006qh3 building    61.7
315803726           School      dormitory      DENIAL0600001A6S building  6820.5
310126904       University             U4      DENIAL0100003uMy building  3570.0
310102902       University             U4      DENIAL0100005xNO building  1253.8
310211901 Retail_Non-Daily    no_evidence      DENIAL0500009ta6 building    71.4
315700501 Retail_Non-Daily    no_evidence      DENIAL6100003W0i building    73.6
315402802          Leisure   holiday_flat      DENIAL040000aTiG

  ok  every unit occupies a positive floor area; a building's units and slots occupy at most its floor area, its ground-floor units at most one footprint; no vacant place, holiday flat or dormitory has a unit; no Workers on a Workers-0 unit the guard did not restore; every weight recomputes from O, f and k (the weight of places without evidence, the Kita weights and what a Kita part of a school gives School included); every POI factor is its class's; the five area labels feed no category
  ok  58,122 unit-category rows on 42,569 units: building 44,966 · poi 11,936 · remainder 1,143 · rest 77
                    units                      weight (M)                         
kind             building   poi remainder rest   building     poi remainder   rest
category                                                                          
Workers             33443  7507      1143    0     54.767  15.482     3.149  0.000
School               1220   173         0    6      2.575   0.533    

  ok  the school types of the 1,399 School units (rows: where the type comes from):
school_types  primary  primary + secondary  secondary  secondary + vocational  vocational
unit_id                                                                                  
name               46                    0         48                       1          11
building          361                   60        367                       0         149
label fix           0                    0          0                       0           6
T-isced             4                    0          9                       0           0
T-list              0                    0          0                       0          12
T-campus            0                    0          0                       0           6
ALKIS               0                  206          0                       0           0
none                0                  113          0                       0           0
  ..  school typ

## 6. The allocation

For every category and every zone with a target above 0, the target goes to
the zone's units of that category in proportion to their weight:
`assigned = weight / Σ weight × target`. A zone without a carrier pools the
units of its ring 1, else of ring 2, and so on until a ring has carriers
(`REDIST_FALLBACK_MAX_RINGS`); what no ring reaches is logged as unassigned.
Buildings in no zone are in no pool. The cell keeps, per category, the ring
that was used and what each zone sent away and received this way, and lists the
moves; the zones layer keeps the target and what was placed in the zone. A zone
with a target of 0
whose places carry Workers shares the Workers targets of the nearest ring of
zones that have one (`REDIST_SHARE_ZERO_TARGET_CATEGORIES`): each of its units
joins those zones, its weight split by their targets — the Autostadt's places
share the VW plant zone's Workers, among which VISUM counts their staff. University, Kindergarten and School places do the same, but
only with the zones that touch them (`REDIST_SHARE_ZERO_TARGET_MAX_RINGS`); a school place that joins two zones takes each zone's pupils by
type as one of its school places. A place with nothing it could take there —
no touching zone holds a total of the category, for a school place one of its
own types — joins the nearest ring up to ring 3 with zones that hold such a
total and places of their own, its weight split by those totals
(`REDIST_SHARE_ZERO_TARGET_FAR_RINGS`; without it Schule Stöckheim,
BBS VII, the Welfenakademie and ten Kitas would get nothing). A Workers total
above 0 but under 1 % of the zone's own Bosserhof expectation (100 expected
Workers or more) counts as 0 for this sharing, with the touching zones only:
the zone keeps its small total and takes no joiners, so a zone with a target
of 0 next to it looks past it (`REDIST_SHARE_NEAR_ZERO_TARGET`:
34 residential VISUM zones have totals of 107 Workers for 19,224
expected, with the Klinikum Wolfenbüttel and the MobileLifeCampus in them). A
place joins only zones with a target and
places of their own: a zone with a target but no place takes no joiners and
keeps the ring fallback, and a school place splits its weight by the zones'
targets of its own types, by their School targets where none of them has its
types. A School place in a zone with a School total of 0,
known only from the cadastre or a bare OSM school outline, with no school POI or
site within 100 m, joins no neighbour and takes no ring fallback. Checks confirm that every unit of a zone with a
target of 0 (or a near-zero total) is split in full over the nearest ring
(within its limit) of zones with a target that is not near zero and places of
their own, by their targets (a school place: of
its own types) — or over the nearest ring up to the far limit with a total it
could take, where the touching zones hold none —, and that inside every pool
each place takes the target by weight (School: by type); the near-zero zones
are recounted from the units, and no zone joins one. A check confirms
that every zone's own carriers share its target exactly in proportion to their
weight. School goes by type: the primary, secondary and vocational targets each
go to the zone's (or the ring's) school places of that type, and to all of
them where none is of that type, so every zone keeps its School total; the
cell lists the zones that gave a type to all their school places.

In [7]:
t0 = time.perf_counter()
ztab = pd.DataFrame(index=zones.index)
_moves = []
# School by type: the type targets, the value per unit and type, and what went to all school places
_ST = zones[[scol('target', t) for t in SCHOOL_TYPES]].set_axis(SCHOOL_TYPES, axis=1)
for t in SCHOOL_TYPES:
    units[scol('assigned', t)] = 0.0
_s_all = {t: 0.0 for t in SCHOOL_TYPES}          # placed on all school places of a zone (or ring) without a place of the type
_s_lost = {t: 0.0 for t in SCHOOL_TYPES}
_s_to_all = {}                                    # zone -> the types it gave to all its school places
far_joins = {}                                    # category -> zone -> its places that joined a ring beyond the touching zones
near_zero = {}                                    # category -> the zones whose small total counts as 0 for the sharing
# a School place in a zone with a School total of 0, known only from the cadastre or a bare OSM school outline, joins no neighbour and
# takes no ring fallback
_NJ = units['no_join'] if 'no_join' in units.columns else pd.Series(False, index=units.index)
for c in CATS:
    tgt = zones[col('target', c)]
    uc = units.loc[(units['category'] == c) & units['zone_idx'].notna(), ['zone_idx', 'weight']].astype({'zone_idx': int})
    # a total above 0 but below max_share of the zone's own Bosserhof expectation (its weights / 100), with min_expected or more
    # expected, counts as 0 for the sharing (REDIST_SHARE_NEAR_ZERO_TARGET): the zone's units join the zones within max_ring that
    # have a total which is not near zero, it is joined by none and keeps its own small total
    _nzr, _tg = REDIST_SHARE_NEAR_ZERO_TARGET.get(c), tgt           # _tg: the totals the sharing reads
    if _nzr is not None:
        _E = uc.groupby('zone_idx')['weight'].sum() / 100.0
        _T0 = tgt.reindex(_E.index)
        near_zero[c] = sorted(int(z_) for z_ in _E.index[(_T0 > 0) & (_T0 < _nzr['max_share'] * _E) & (_E >= _nzr['min_expected'])])
        _tg = tgt.where(~tgt.index.isin(near_zero[c]), 0.0)
    _NZ = set(near_zero.get(c, []))
    # a zone with a target of 0 whose places carry the category shares the targets of the nearest ring of zones that have one
    # and places of their own (REDIST_SHARE_ZERO_TARGET_CATEGORIES; School,
    # University and Kindergarten only with the zones that touch it: REDIST_SHARE_ZERO_TARGET_MAX_RINGS): each of
    # its units joins those zones, its weight split by their targets (a school place: by their targets of its own types), and
    # takes its part of their targets by weight. A zone with a target but no place of its own takes no joiners: it keeps the
    # ring fallback below. A place with nothing it could take next door -
    # no touching zone holds a total of the category (a school place: of its own types), with places or without - joins the
    # nearest ring up to REDIST_SHARE_ZERO_TARGET_FAR_RINGS with zones that hold one and places of their own, its weight split
    # by those totals
    _att, share_ring = [], pd.Series(pd.NA, index=zones.index, dtype='Int64')
    _home = set(uc['zone_idx'])                   # the zones with places of their own
    _ELG = ~_NJ.reindex(uc.index, fill_value=False)   # the places that may take a neighbour's pupils
    if c in REDIST_SHARE_ZERO_TARGET_CATEGORIES:
        _mr = REDIST_SHARE_ZERO_TARGET_MAX_RINGS[c]
        _fr = REDIST_SHARE_ZERO_TARGET_FAR_RINGS.get(c)
        far_joins[c] = {}

        def _can(j, ur):                          # what zone j holds that the place could take
            return float(sum(_ST.at[j, t] for t in units.at[ur, 'school_types'])) if c == 'School' else float(tgt[j])
        for zi in sorted(_home - set(_tg.index[_tg > 0])):
            _u = uc[(uc['zone_idx'] == zi) & ~_NJ.reindex(uc.index, fill_value=False)]
            if not len(_u):
                continue                              # every place of the zone stays out
            seen, frontier, r, nb = {zi}, {zi}, 0, []
            while not nb:
                r += 1
                if (_mr is not None and r > _mr) or (zi in _NZ and _nzr['max_ring'] is not None and r > _nzr['max_ring']):
                    break
                frontier = set().union(*(NEIGHBOURS[j] for j in frontier)) - seen
                if not frontier:
                    break
                seen |= frontier
                nb = [j for j in sorted(frontier) if _tg[j] > 0 and j in _home]
            if _fr is not None:
                _rg, _sn, _fz = [], {zi}, {zi}                # the rings around the zone up to _fr
                for _ in range(_fr):
                    _fz = set().union(*(NEIGHBOURS[j] for j in _fz)) - _sn
                    _sn |= _fz
                    _rg.append(sorted(_fz))
                for ur in _u.index:
                    if any(_can(j, ur) > 0 for j in _rg[0] if tgt[j] > 0):
                        continue                          # something to take next door: the touching zones
                    for rf, _ring in enumerate(_rg[1:], 2):
                        _hz = [j for j in _ring if tgt[j] > 0 and j in _home and _can(j, ur) > 0]
                        if _hz:
                            _v = np.array([_can(j, ur) for j in _hz])
                            _att += [_u.loc[[ur]].assign(pool=j, weight=_u.at[ur, 'weight'] * v / _v.sum()) for j, v in zip(_hz, _v)]
                            share_ring[zi] = rf if pd.isna(share_ring[zi]) else min(int(share_ring[zi]), rf)
                            far_joins[c][zi] = far_joins[c].get(zi, []) + [ur]
                            break
                _u = _u.drop(index=far_joins[c].get(zi, []))
            if nb and len(_u):
                if c == 'School':
                    # by the zones' targets of the place's own types, by their School targets where none of them has its types
                    _sw = pd.DataFrame([[float(sum(_ST.at[j, t] for t in s)) for j in nb] for s in units.loc[_u.index, 'school_types']],
                                       index=_u.index, columns=nb)
                    _no = _sw.sum(axis=1).eq(0)
                    _sw.loc[_no] = np.tile(tgt[nb].to_numpy(dtype=float), (int(_no.sum()), 1))
                    _sw = _sw.div(_sw.sum(axis=1), axis=0)
                    _att += [_u[_sw[j] > 0].assign(pool=j, weight=_u['weight'] * _sw[j]) for j in nb if (_sw[j] > 0).any()]
                else:
                    _att += [_u.assign(pool=j, weight=_u['weight'] * _tg[j] / _tg[nb].sum()) for j in nb]
                share_ring[zi] = r
    pooled = pd.concat([uc.assign(pool=uc['zone_idx'])] + _att).rename_axis('unit_row').reset_index()
    wz = pooled.groupby('pool')['weight'].sum()
    # the zone's own carriers, and the units that joined it
    _own = pooled['pool'].map(tgt).gt(0)
    pooled['got'] = np.where(_own, pooled['weight'] / pooled['pool'].map(wz) * pooled['pool'].map(tgt), 0.0)
    if c == 'School':
        # by type: each type's target over the pool's school places of that type, by weight; a pool without a place of a
        # type gives it to all its school places, so the pool keeps its School total. A place of a zone with a target of 0 can
        # join two pools: its values per type are summed over them
        _sg, _sa = floor_rules.school_split(pooled['weight'], pooled['unit_row'].map(units['school_types']), pooled['pool'], _ST,
                                            SCHOOL_TYPES)
        pooled['got'] = _sg.sum(axis=1).to_numpy()
        _sgu = _sg.groupby(pooled['unit_row'].to_numpy()).sum()
        for t in SCHOOL_TYPES:
            units.loc[_sgu.index, scol('assigned', t)] += _sgu[t].to_numpy()
            _s_all[t] += float(_sg.loc[_sa[t], t].sum())
            for p in pooled.loc[_sa[t].to_numpy(), 'pool'].unique():
                _s_to_all.setdefault(int(p), []).append(t)
    _got = pooled.groupby('unit_row')['got'].sum()
    units.loc[_got.index, 'assigned'] += _got.to_numpy()
    _sh = pooled[pooled['pool'] != pooled['zone_idx']]
    shared_out = _sh.groupby('pool')['got'].sum().reindex(zones.index, fill_value=0.0)
    shared_in = _sh.groupby('zone_idx')['got'].sum().reindex(zones.index, fill_value=0.0)
    if c in REDIST_SHARE_ZERO_TARGET_CATEGORIES:
        # the joined pools: every unit of a zone with a target of 0 that shares is split in full over the nearest
        # ring (within the category's limit) of zones with a target and places of their own, by their targets (a school place:
        # by their targets of its own types, by their School totals where none of them has its types) - or, with nothing it
        # could take in the touching zones, over the nearest ring up to the far limit with zones that hold a total it could
        # take and places of their own, by those totals -, and nothing else joins; inside every pool each place
        # takes the target by weight (School: each type over the places of that type, or all of them); a zone with a near-zero
        # total counts as 0 within its own ring limit, and no zone joins it. Re-derived per unit from each zone's
        # distances to the zones around it, from the targets and the near-zero zones themselves
        _nzv = set(near_zero.get(c, []))
        _tz, _ex, _rgx = {j for j in _home if tgt[j] > 0 and j not in _nzv}, [], {}
        for zi in sorted(_home - _tz):
            _d, _fz, r = {zi: 0}, {zi}, 0
            while _fz and (_mr is None or r < max(_mr, _fr or 0)):
                r += 1
                _fz = {k for j in _fz for k in NEIGHBOURS[j] if k not in _d}
                _d.update(dict.fromkeys(_fz, r))
                if _mr is None and _fz & _tz:
                    break
            _r1 = [j for j, k in _d.items() if k == 1 and tgt[j] > 0 and j not in _nzv]
            _nr = [j for j, k in _d.items() if j in _tz and k >= 1 and (_mr is None or k <= _mr)
                   and not (zi in _nzv and _nzr['max_ring'] is not None and k > _nzr['max_ring'])]
            _r0 = min((_d[j] for j in _nr), default=None)
            for _ur, _w in uc.loc[(uc['zone_idx'] == zi) & ~_NJ.reindex(uc.index, fill_value=False), 'weight'].items():
                _cv = ((lambda j, s=units.at[_ur, 'school_types']: float(sum(_ST.at[j, t] for t in s))) if c == 'School'
                       else (lambda j: float(tgt[j])))
                _rf = (min((_d[j] for j in _tz & set(_d) if 2 <= _d[j] <= _fr and _cv(j) > 0), default=None)
                       if _fr is not None and not any(_cv(j) > 0 for j in _r1) else None)
                if _rf is not None:
                    _hz = sorted(j for j in _tz & set(_d) if _d[j] == _rf and _cv(j) > 0)
                    _v, _rr = np.array([_cv(j) for j in _hz]), _rf
                elif _r0 is not None:
                    _hz = sorted(j for j in _nr if _d[j] == _r0)
                    _v, _rr = (np.array([_cv(j) for j in _hz]) if c == 'School' else tgt[_hz].to_numpy(dtype=float)), _r0
                    if not _v.sum() > 0:
                        _v = tgt[_hz].to_numpy(dtype=float)
                else:
                    continue
                _rgx[zi] = min(_rgx.get(zi, _rr), _rr)
                _ex += [(_ur, j, _w * v / _v.sum()) for j, v in zip(_hz, _v) if v > 0]
        if {int(k): int(v) for k, v in share_ring.dropna().items()} != _rgx:
            raise AssertionError(f'{c}: a zone with a target of 0 (or a near-zero total) does not join the ring the sharing rule gives '
                                 'it (the nearest with a target that is not near zero and places of its own, or the far ring)')
        _ex = pd.DataFrame(_ex, columns=['unit_row', 'pool', 'weight']).set_index(['unit_row', 'pool'])['weight'].sort_index()
        _ac = _sh.set_index(['unit_row', 'pool'])['weight'].sort_index()
        if not (list(_ex.index) == list(_ac.index) and np.allclose(_ex.to_numpy(), _ac.to_numpy(), rtol=1e-9, atol=1e-9)):
            raise AssertionError(f'{c}: a unit of a zone with a target of 0 (or a near-zero total) is not split in full over its ring by '
                                 'the targets (School: of its own types)')
        if _NJ.reindex(_sh['unit_row'], fill_value=False).any():
            raise AssertionError(f'{c}: a place that joins no neighbour joined a zone')
        if _nzr is not None:                      # the near-zero zones, recounted from the units, and no zone joins one
            _Eu = units.loc[units['category'].eq(c) & units['zone_idx'].notna()].groupby('zone_idx')['weight'].sum() / 100.0
            _Eu.index = _Eu.index.astype(int)
            _tu = tgt.reindex(_Eu.index)
            if (sorted(int(z_) for z_ in _Eu.index[(_tu > 0) & (_tu < _nzr['max_share'] * _Eu) & (_Eu >= _nzr['min_expected'])])
                    != near_zero[c] or _sh['pool'].isin(near_zero[c]).any()):
                raise AssertionError(f'{c}: the near-zero zones differ from REDIST_SHARE_NEAR_ZERO_TARGET, or a zone joined one')
        if c == 'School':
            _ty = pooled['unit_row'].map(units['school_types'])
            for t in SCHOOL_TYPES:
                _e = _ty.map(lambda s, t=t: t in s)
                _el = _e | ~_e.groupby(pooled['pool']).transform('any')
                _T = pooled['pool'].map(_ST[t])
                _ws = pooled['weight'].where(_el, 0.0).groupby(pooled['pool']).transform('sum')
                if not np.allclose(_sg[t].to_numpy(), np.where(_el & (_T > 0), pooled['weight'] / _ws.where(_ws > 0) * _T, 0.0),
                                   rtol=1e-9, atol=1e-9):
                    raise AssertionError(f'School {t}: a pool does not give the type to its places of that type by weight')
        else:
            _rt = (pooled['got'] / pooled['weight'].where(pooled['weight'] > 0)).groupby(pooled['pool'])
            _pt = pooled.groupby('pool')['got'].sum()
            if not (np.allclose((_rt.max() - _rt.min()).fillna(0.0), 0.0, atol=1e-9)
                    and np.allclose(_pt, tgt.reindex(_pt.index), rtol=1e-9, atol=1e-6)):
                raise AssertionError(f'{c}: inside a pool the places do not take the target by weight')
    ring = pd.Series(pd.NA, index=zones.index, dtype='Int64')
    ring[(tgt > 0) & tgt.index.isin(list(_home))] = 0
    sent = pd.Series(0.0, index=zones.index)
    received = pd.Series(0.0, index=zones.index)
    lost = pd.Series(0.0, index=zones.index)
    # the zones without one: ring by ring, over the places located in the ring
    for zi in tgt.index[(tgt > 0) & ~tgt.index.isin(list(_home))]:
        seen, frontier, r, pool = {zi}, {zi}, 0, None
        while pool is None:
            r += 1
            frontier = set().union(*(NEIGHBOURS[j] for j in frontier)) - seen
            if not frontier or (REDIST_FALLBACK_MAX_RINGS is not None and r > REDIST_FALLBACK_MAX_RINGS):
                break
            seen |= frontier
            if uc.loc[uc['zone_idx'].isin(frontier) & _ELG, 'weight'].sum() > 0:
                pool = frontier
        if pool is None:
            lost[zi], ring[zi] = tgt[zi], -1
            if c == 'School':
                for t in SCHOOL_TYPES:
                    _s_lost[t] += float(_ST.at[zi, t])
            continue
        _in = uc['zone_idx'].isin(pool) & _ELG
        _part = uc.loc[_in, 'weight'] / uc.loc[_in, 'weight'].sum() * tgt[zi]
        if c == 'School':                         # by type here too, over the ring's school places
            _sg, _sa = floor_rules.school_split(uc.loc[_in, 'weight'], units.loc[uc.index[_in], 'school_types'],
                                                pd.Series(zi, index=uc.index[_in]), _ST, SCHOOL_TYPES)
            _part = _sg.sum(axis=1)
            for t in SCHOOL_TYPES:
                units.loc[_sg.index, scol('assigned', t)] += _sg[t].to_numpy()
                _s_all[t] += float(_sg.loc[_sa[t], t].sum())
                if _sa[t].any():
                    _s_to_all.setdefault(int(zi), []).append(t)
        units.loc[_part.index, 'assigned'] += _part.to_numpy()
        ring[zi], sent[zi] = r, tgt[zi]
        received = received.add(_part.groupby(uc.loc[_in, 'zone_idx']).sum(), fill_value=0.0)
        _moves.append((c, zones.at[zi, 'NAME'], r, tgt[zi], len(pool), int(_in.sum())))
    ztab[col('target', c)] = tgt
    ztab[col('fallback_ring', c)] = ring
    ztab[col('fallback_out', c)] = sent
    ztab[col('fallback_in', c)] = received.reindex(zones.index, fill_value=0.0)
    ztab[col('unassigned', c)] = lost
    ztab[col('shared_ring', c)] = share_ring
    ztab[col('shared_out', c)] = shared_out
    ztab[col('shared_in', c)] = shared_in
    ztab[col('near_zero', c)] = ztab.index.isin(near_zero.get(c, []))
_sl = (units.dropna(subset=['zone_idx']).astype({'zone_idx': int})
       .groupby('zone_idx')[[scol('assigned', t) for t in SCHOOL_TYPES]].sum().reindex(zones.index, fill_value=0.0))
for t in SCHOOL_TYPES:
    ztab[scol('target', t)] = _ST[t]
    ztab[scol('assigned', t)] = _sl[scol('assigned', t)]
ztab['school_types_to_all'] = pd.Series({z_: ';'.join(t for t in SCHOOL_TYPES if t in ts) for z_, ts in _s_to_all.items()},
                                        dtype=object).reindex(zones.index).fillna('')
print(f'  ok  allocated in {time.perf_counter() - t0:,.1f}s')

# --- conservation: per category, and per zone by location ----------------------------------------
_loc = (units.dropna(subset=['zone_idx']).astype({'zone_idx': int})
        .pivot_table(index='zone_idx', columns='category', values='assigned', aggfunc='sum', fill_value=0.0)
        .reindex(index=zones.index, columns=CATS, fill_value=0.0))
for c in CATS:
    ztab[col('assigned', c)] = _loc[c]
    _placed, _total, _un = units.loc[units['category'] == c, 'assigned'].sum(), ztab[col('target', c)].sum(), ztab[col('unassigned', c)].sum()
    if not np.isclose(_placed + _un, _total, rtol=1e-9, atol=1e-6):
        raise AssertionError(f'{c}: placed {_placed:,.3f} + unassigned {_un:,.3f} != target {_total:,.3f}')
    _expect = (ztab[col('target', c)] - ztab[col('unassigned', c)] - ztab[col('fallback_out', c)] + ztab[col('fallback_in', c)]
               - ztab[col('shared_out', c)] + ztab[col('shared_in', c)])
    if not np.allclose(_loc[c], _expect, rtol=1e-9, atol=1e-6):
        raise AssertionError(f'{c}: what lies in a zone differs from target - unassigned - sent + received - shared out + shared in')
    if c not in REDIST_SHARE_ZERO_TARGET_CATEGORIES and (ztab[col('shared_in', c)].any() or ztab[col('shared_out', c)].any()):
        raise AssertionError(f'{c}: shared with zones of target 0 although it is not one of REDIST_SHARE_ZERO_TARGET_CATEGORIES')
if (units['assigned'] < 0).any() or not np.isfinite(units['assigned']).all():
    raise AssertionError('a negative or non-finite assigned value')
if units.loc[units['zone_idx'].isna(), 'assigned'].any():
    raise AssertionError('a unit in no zone received a value')
if units.loc[_NJ.reindex(units.index, fill_value=False), 'assigned'].any():
    raise AssertionError('a School place that joins no neighbour received a value')
for c in [c for c in CATS if c != 'School']:   # a zone placed on its own carriers alone: assigned / weight = target / its weight
    _zs = ztab.index[ztab[col('fallback_ring', c)].eq(0).fillna(False) & ztab[col('fallback_in', c)].eq(0)
                     & ztab[col('shared_out', c)].eq(0) & ztab[col('shared_in', c)].eq(0)]
    _u = units[(units['category'] == c) & units['zone_idx'].isin(_zs)].astype({'zone_idx': int})
    _rate = _u['zone_idx'].map(ztab[col('target', c)]) / _u['zone_idx'].map(_u.groupby('zone_idx')['weight'].sum())
    if not np.allclose(_u['assigned'] / _u['weight'], _rate, rtol=1e-9, atol=0):
        raise AssertionError(f'{c}: a zone does not share its target in proportion to the weights')
# School by type: the types add up to School on every unit, each type's total holds, and a zone on its own school places
# gives each type to its places of that type by weight (to all of them where it has none)
_sc = units['category'].eq('School')
_sty = [scol('assigned', t) for t in SCHOOL_TYPES]
if not np.allclose(units.loc[_sc, _sty].sum(axis=1), units.loc[_sc, 'assigned'], rtol=1e-9, atol=1e-9) or units.loc[~_sc, _sty].to_numpy().any():
    raise AssertionError('the school types do not add up to School on a unit, or a unit of another category holds a school type')
for t in SCHOOL_TYPES:
    if not np.isclose(units[scol('assigned', t)].sum() + _s_lost[t], _ST[t].sum(), rtol=1e-9, atol=1e-6):
        raise AssertionError(f'School {t}: placed + unassigned != the target')
_zs = ztab.index[ztab[col('fallback_ring', 'School')].eq(0).fillna(False) & ztab[col('fallback_in', 'School')].eq(0)
                 & ztab[col('shared_out', 'School')].eq(0) & ztab[col('shared_in', 'School')].eq(0)]
_u = units[_sc & units['zone_idx'].isin(_zs)].astype({'zone_idx': int})
for t in SCHOOL_TYPES:
    _e = _u['school_types'].map(lambda s: t in s)
    _el = _e | ~_e.groupby(_u['zone_idx']).transform('any')
    _T = _u['zone_idx'].map(_ST[t])
    _ws = _u['weight'].where(_el, 0.0).groupby(_u['zone_idx']).transform('sum')
    if not np.allclose(_u[scol('assigned', t)], np.where(_el & (_T > 0), _u['weight'] / _ws * _T, 0.0), rtol=1e-9, atol=1e-9):
        raise AssertionError(f'School {t}: a zone does not give the type to its places of that type by weight')
print('  ok  every category: placed + unassigned = the zone total; every zone: what lies in it = target - unassigned - sent + received '
      '- shared out + shared in; a zone on its own carriers shares its target exactly by weight (School: each type over its places '
      'of that type, or all of them where it has none; the types add up to School on every unit); a zone with a target of 0 (or a '
      'near-zero total, within its own ring limit) splits each unit in full over the nearest ring of zones with a target that is not '
      'near zero and places of their own, by their targets (School: of its '
      'own types) - a unit with nothing it could take next door over the nearest ring up to the far limit with a total it '
      'could take -, and every pool shares by weight (School: by type); the near-zero zones are recounted from the units, and no '
      'zone joins one')
for c in REDIST_SHARE_ZERO_TARGET_CATEGORIES:
    _nzc = ztab[col('near_zero', c)]
    _zr = ztab[col('shared_ring', c)].notna() & ~_nzc
    _top = ztab.loc[_zr, col('shared_in', c)].sort_values(ascending=False).head(6)
    _mr = REDIST_SHARE_ZERO_TARGET_MAX_RINGS[c]
    _fz = ztab.loc[ztab[col('target', c)].eq(0), col('fallback_in', c)].sum()
    _zn = int((_zr & ztab[col('shared_in', c)].le(0)).sum())
    _fr = REDIST_SHARE_ZERO_TARGET_FAR_RINGS.get(c)
    _fj = far_joins.get(c, {})
    _fu = [u_ for v_ in _fj.values() for u_ in v_]
    print(f'  ..  {c}: {int(_zr.sum())} zone(s) with a target of 0 share the targets of their ring ({int(ztab.loc[_zr, col("shared_ring", c)].eq(1).sum())} '
          f'of them ring 1; {"any ring" if _mr is None else f"at most ring {_mr}"}'
          + (f', up to ring {_fr} for a place with nothing it could take next door: {len(_fu)} place(s) on '
             f'{units.loc[_fu, "building_id"].nunique()} building(s) in {len(_fj)} zone(s)' if _fr is not None else '')
          + (f'; {_zn} of them take nothing from it - no zone they join has a target their places can take' if _zn else '')
          + f'): {ztab.loc[~_nzc, col("shared_in", c)].sum():,.0f} '
          f'({100 * ztab.loc[~_nzc, col("shared_in", c)].sum() / ztab[col("target", c)].sum():.2f} %) '
          'of the regional total is shared onto their places'
          + (f' (and {_fz:,.0f} reaches zones with a target of 0 through the ring fallback of zones without a place)' if _fz > 0 else '')
          + ', e.g. ' + ' · '.join(f'{zones.at[i, "NAME"]} {v:,.0f}' for i, v in _top.items()))
    if _nzc.any():                                # near-zero totals
        _nzr = REDIST_SHARE_NEAR_ZERO_TARGET[c]
        _si = ztab.loc[_nzc, col('shared_in', c)]
        _rgt = 'any ring' if _nzr['max_ring'] is None else f'at most ring {_nzr["max_ring"]}'
        print(f'  ..  {c}: {int(_nzc.sum())} zone(s) with a near-zero total (above 0 but under {100 * _nzr["max_share"]:g} % of their own '
              f'expectation, with {_nzr["min_expected"]:,.0f} or more expected) count as 0 for the sharing - '
              f'{int(ztab.loc[_nzc, col("shared_ring", c)].notna().sum())} of them join ({_rgt}): {_si.sum():,.0f} '
              f'({100 * _si.sum() / ztab[col("target", c)].sum():.2f} %) of the regional total is shared onto their places, e.g. '
              + ' · '.join(f'{zones.at[i, "NAME"]} {v:,.0f}' for i, v in _si.sort_values(ascending=False).head(6).items()))

_s = pd.DataFrame({c: {'zones with a target': int((ztab[col('target', c)] > 0).sum()),
                       'own carriers': int((ztab[col('fallback_ring', c)] == 0).sum()),
                       'ring 1': int((ztab[col('fallback_ring', c)] == 1).sum()),
                       'ring 2+': int((ztab[col('fallback_ring', c)] >= 2).sum()),
                       'none reached': int((ztab[col('fallback_ring', c)] == -1).sum()),
                       'moved': round(ztab[col('fallback_out', c)].sum(), 1),
                       'moved %': round(100 * ztab[col('fallback_out', c)].sum() / ztab[col('target', c)].sum(), 2),
                       'unassigned': round(ztab[col('unassigned', c)].sum(), 1)} for c in CATS}).T
_s = _s.astype({k: int for k in ('zones with a target', 'own carriers', 'ring 1', 'ring 2+', 'none reached')})
print(_s.to_string())
_far = [m for m in _moves if m[2] >= 2]
if _far:
    print('  ..  placed beyond ring 1: ' + '; '.join(f'{c} {t:,.0f} of {n} (ring {r})' for c, n, r, t, _, _ in _far))
_amt = {z_: sum(_ST.at[z_, t] for t in ts) for z_, ts in _s_to_all.items()}
print('  ..  School by type: ' + ' · '.join(
    f'{t} {units[scol("assigned", t)].sum():,.0f}, {100 * _s_all[t] / _ST[t].sum():.1f} % of it on all school places of a zone or ring '
    f'without a {t} place' for t in SCHOOL_TYPES))
print(f'  ..  {len(_s_to_all)} zone(s) gave a type to all their school places, the largest: '
      + ' · '.join(f'{zones.at[z_, "NAME"]} {ztab.at[z_, "school_types_to_all"]} {v:,.0f}'
                   for z_, v in sorted(_amt.items(), key=lambda kv: -kv[1])[:8]))

  ok  allocated in 2.1s
  ok  every category: placed + unassigned = the zone total; every zone: what lies in it = target - unassigned - sent + received - shared out + shared in; a zone on its own carriers shares its target exactly by weight (School: each type over its places of that type, or all of them where it has none; the types add up to School on every unit); a zone with a target of 0 (or a near-zero total, within its own ring limit) splits each unit in full over the nearest ring of zones with a target that is not near zero and places of their own, by their targets (School: of its own types) - a unit with nothing it could take next door over the nearest ring up to the far limit with a total it could take -, and every pool shares by weight (School: by type); the near-zero zones are recounted from the units, and no zone joins one
  ..  Workers: 87 zone(s) with a target of 0 share the targets of their ring (82 of them ring 1; any ring): 17,425 (2.94 %) of the regional total is shared

## 7. Workers: realistic headcounts

VISUM's Workers are a real headcount of jobs, and the allocation of section 6
gives every unit of a zone the same multiple of its Bosserhof value: where a
zone's total is many times what its buildings show, a theatre, a supermarket and
a Kita are inflated alike (the Peiner Festsäle would hold 1,095 Workers in
Peine 22, 9.8 times their Bosserhof value, for an operator with 34 staff for two
venues). Rule R works on top of the allocation, for Workers only (config block
"STEP 08 — Workers: realistic headcounts", `lib/workers_headcount.py`):

* every Workers unit gets an **activity type** (60) from its own data — an asked
  POI by its OSM use, else its step-07 type, name and class; a building unit by
  the head of its step-05 type, its name, a specific class, its full type, its
  class — and a **sub-type** with the published band of its kind (typical / high
  / max per establishment; a size rule where the size can be known: sales area,
  rooms, places, pupils, seats, screens);
* its **realistic headcount** is its Bosserhof value, never above the band's high
  or its size-rule value;
* its **upper limit**: an establishment at most 2 × its realistic headcount and
  never above the published maximum of its type; an open-ended employer (offices,
  administration, industry, hospitals, universities …) at most 3 × its Bosserhof
  value, industry, logistics and transport 5 ×; the remainder unit of an
  establishment's building at most 1 ×; a technical structure (below) at most its
  realistic headcount; the 39 named institutions with a published staff total at
  most that total over all their units; no lower limits;
* in every zone one level fills the units up to their upper limits, so that the
  zone keeps what its units hold after section 6; what an over-full zone cannot
  hold goes to the units of the zones around it — the touching zones first, then
  rings 2 and 3 (the rings of section 2) — that hold less than their realistic
  headcount, the emptiest first and never above it; the regional total stays
  exact. A zone whose Workers total is 0 or near 0 takes part like any other: it
  sends what the sharing of section 6 gave it beyond what its units can hold, and
  it receives spill (the "takes no joiners" of section 6 is about that sharing
  only);
* the spill passes over two kinds of units, which keep what their own zone's
  fill gives them (`WORKERS_NO_SPILL`): the remainder unit of an establishment's building — its
  unmapped upper floors, often storeys that a single-storey shop hall has only
  by its height (without this rule 215 of them take 1,724 Workers of spill) —
  and the technical structures, units whose own texts read as a sewage plant, a
  biogas or composting plant, a cistern, a bunker, a greenhouse, a silo or an
  animal shelter;
* a technical structure also holds at most its realistic headcount in its own
  zone's fill, the way the remainder rule caps the remainders (`technical_k`;
  without it the Klärwerk Gifhorn holds 292 Workers as industry, 5 × its Bosserhof value, in the
  over-full Gifhorn 07) — what its zone can then no longer hold goes to the
  zones around it like any other surplus. Its words count in the head of its
  type, in the whole type where the head names nothing, and in its name only
  where the type carries a word of the same group too or the head names
  nothing (the Fitesa plant in Peine, ALKIS name "Silos", is a factory); a
  digester or fermenter outside a sewage plant is a biogas plant, a greenhouse
  whose type names a garden centre or a shop is the garden centre or the shop,
  and the fitness word "aerobic" does not read "anaerobic" digesters as
  fitness studios;
* a conference or seminar centre is an event venue with the band 12 / 25 / 45,
  also in an office class (read as business services, the Tagungszentrum Haus
  Hessenkopf in Goslar would hold 244 Workers); only the buildings of the two company campuses on VW sites, the
  MobileLifeCampus and the technology conference and trade fair centre, stay
  business services.

Since Workers are real headcounts, the food stores — supermarkets, discounters, hypermarkets —
take the published size rule of their type as their realistic headcount in both
directions, also above their Bosserhof value (`size_rule_both_ways`: 241
discounters rise from about 10 to about 18); EDEKA Center and REWE Center count
as hypermarkets (`POI_HYPER_PATTERN`). And the named institutions are raised to
their published staff after the spill (`institution_floor_k` = 1): the deficit
comes from the units of their own zones, then of rings 1–3, that hold more than
their realistic headcount, the fullest first and never below it; an institution
is never taken from, and each institution unit holds its published share of the
faces. All 20 under-full institutions reach their staff (the PTB 98 → 1,700,
Klinikum Peine 443 → 784) with 4,659 Workers from 1,617 units.

The exact zone totals do not
hold for Workers where a zone holds more than its buildings realistically
can: the zones layer shows what lies in each zone after the rule, its target
stays VISUM's. The other six categories and the visitor values are not touched.
The cell checks the rule's invariants and prints what moved, the zones that
changed, the units that take no spill, the units above their limits before and
after, the named institutions against their published staff, the largest
gainers and flows, and the multipliers.

In [8]:
t0 = time.perf_counter()
# Workers: realistic headcounts on top of the allocation (rule R; lib/workers_headcount.py). The
# Workers rows of `units` take the rule's values; every other category keeps what section 6 gave it. The units of
# WORKERS_NO_SPILL (the remainders the remainder rule caps, the technical structures) take no spill;
# a technical structure holds at most its realistic headcount (technical_k)
_wa = dict(groups=WORKERS_ACTIVITY_GROUPS, keywords=WORKERS_ACTIVITY_KEYWORDS, poi_uses=WORKERS_ACTIVITY_POI_USES,
           shop_uses=WORKERS_ACTIVITY_SHOP_USES, classes=WORKERS_ACTIVITY_CLASSES, rules=WORKERS_ACTIVITY_RULES)
workers_rule = workers_headcount.apply(units, uo, bld, pois, zones, NEIGHBOURS, zone_id_col=ZONE_ID_COL, knobs=WORKERS_HEADCOUNT,
                                       activity=_wa, bands=WORKERS_BANDS, subtypes=WORKERS_SUBTYPES, sizes=WORKERS_SIZES,
                                       institutions=WORKERS_INSTITUTIONS, no_spill=WORKERS_NO_SPILL)
_wok = workers_headcount.verify(workers_rule)           # the rule's invariants (raises on a violation)
_wW = workers_rule['W']
_keep = units.loc[units['category'] != 'Workers', ['assigned'] + [scol('assigned', t) for t in SCHOOL_TYPES]].copy()
units.loc[_wW.index, 'assigned'] = _wW['after'].to_numpy()
# the zones: what lies in each zone after the rule, by the location of its units as in section 6; target_Workers stays VISUM's
_wz = units[units['category'].eq('Workers') & units['zone_idx'].notna()]
ztab[col('assigned', 'Workers')] = _wz.groupby(_wz['zone_idx'].astype(int))['assigned'].sum().reindex(zones.index, fill_value=0.0)
if not units.loc[units['category'] != 'Workers', _keep.columns].equals(_keep):
    raise AssertionError('the Workers rule touched another category')
if not (np.isclose(ztab[col('assigned', 'Workers')].sum(), ztab[col('target', 'Workers')].sum() - ztab[col('unassigned', 'Workers')].sum(),
                   rtol=0, atol=1e-6)
        and np.isclose(units.loc[units['category'].eq('Workers'), 'assigned'].sum(), _wW['before'].sum(), rtol=0, atol=1e-6)):
    raise AssertionError('Workers: the units or the zones no longer add up to the regional total')
print(f'  ok  rule R in {time.perf_counter() - t0:,.1f}s on {len(_wW):,} Workers units ({int(_wW["zone_idx"].notna().sum()):,} in a zone): '
      f'{_wok}; the other categories are untouched; the zones hold what lies in them after the rule, their targets stay VISUM\'s')
_wr = workers_headcount.report(workers_rule, zones)
_n = _wr['total_before']
_wg = _wW[_wW['zone_idx'].notna()]
_we = _wg['agroup_eff'].eq('fixed')
print(f"  ..  {_wg['atype'].nunique()} activity types: {int(_we.sum()):,} units with an establishment band hold "
      f"{_wr['establishments_W_before']:,.0f} -> {_wr['establishments_W_after']:,.0f} Workers, {int((~_we).sum()):,} open-ended units the rest; "
      f"the realistic headcounts sum to {_wg['face'].sum():,.0f} against a Bosserhof value of {_wg['exp'].sum():,.0f} (VISUM: {_n:,.0f})")
print(f"  ..  moved: {_wr['moved_units']:,.0f} Workers ({100 * _wr['moved_units'] / _n:.1f} %) between units, {_wr['moved_buildings']:,.0f} "
      f"({100 * _wr['moved_buildings'] / _n:.1f} %) between buildings ({_wr['buildings_changed']:,} buildings change by more than 0.5), "
      f"{_wr['moved_zones']:,.0f} ({100 * _wr['moved_zones'] / _n:.1f} %) between zones: {_wr['over_full_zones']} zone(s) could not hold "
      f"their total and sent {_wr['surplus']:,.0f} to the zones around them (ring 1: the touching zones) - "
      + ', '.join(f'ring {int(k)} {v:,.0f}' for k, v in sorted(_wr['ring_split'].items()))
      + (f"; {_wr['forced']:,.0f} found no place within ring {WORKERS_HEADCOUNT['max_ring']} and stay on the open-ended units of "
         f"{', '.join(_wr['forced_zones'])}" if _wr['forced'] else '; none is forced back'))
print(f"  ..  zones: {_wr['zones_changed']} of the {_wr['zones_with_units']} zones with Workers units change their total (by more than 0.5), "
      f"{_wr['senders']} send and {_wr['receivers']} receive; {_wr['zones_doubled']} at least double and {_wr['zones_halved']} lose half or "
      'more (by more than 5 Workers)')
_ns = _wr['no_spill']                       # who takes no spill, the technical structures' cap, the conference centres
_nt, _nr = _ns[_ns.index.str.startswith('technical')], _ns[~_ns.index.str.startswith('technical')]
_wc = _wg['atype_rule'].str.contains('conference / seminar centre|company campus', regex=True)
print(f"  ..  no spill into {int(_ns['units'].sum()):,} units, which keep their zone's fill: {int(_nr['units'].sum()):,} remainder units of "
      f"establishment buildings ({_nr['before'].sum():,.0f} -> {_nr['after'].sum():,.0f} Workers) and {int(_nt['units'].sum()):,} technical "
      f"structures ({_nt['before'].sum():,.0f} -> {_nt['after'].sum():,.0f}): " + ', '.join(f"{k[11:]} {int(v)}" for k, v in _nt['units'].items())
      + f"; technical structures above their realistic headcount: {_wr['technical_above_face_before']:,} "
      f"({_wr['W_technical_above_face_before']:,.0f} Workers above it) -> {_wr['technical_above_face_after']:,}"
      + f"; {int((_wc & _wg['atype'].eq('event venue / city hall')).sum())} conference or seminar centre(s) of an office class stay event "
      f"venues, {int((_wc & _wg['atype'].eq('business services')).sum())} building(s) of the two company campuses business services")
print(f"  ..  units above the published maximum of their type: {_wr['above_max_before']:,} ({_wr['W_above_max_before']:,.0f} Workers above "
      f"it) -> {_wr['above_max_after']:,} ({_wr['W_above_max_after']:,.0f}); establishments above {WORKERS_HEADCOUNT['fixed_k']:g} x their "
      f"realistic headcount: {_wr['est_above_kf_before']:,} ({_wr['W_est_above_kf_before']:,.0f} Workers above it) -> "
      f"{_wr['est_above_kf_after']:,} ({_wr['W_est_above_kf_after']:,.0f})")
_mb, _ma = _wr['mult_before'], _wr['mult_after']
print('  ..  Workers / Bosserhof value of a unit, weighted by its Workers: '
      + ' · '.join(f'q{int(round(100 * q))} {_mb[q]:.2f} -> {_ma[q]:.2f}' for q in _mb)
      + f" · max {_wr['mult_max_before']:.1f} -> {_wr['mult_max_after']:.1f}")
_wi = _wr['institutions'].assign(x_before=lambda d: d['before'] / d['published'], x_after=lambda d: d['after'] / d['published'])
print(f"  ..  the {len(_wi)} named institutions against their published staff (the upper limit and the lower one, "
      f"institution_floor_k {WORKERS_HEADCOUNT.get('institution_floor_k')}): "
      f"{int(_wi['x_before'].between(0.5, 2).sum())} -> {int(_wi['x_after'].between(0.5, 2).sum())} within half and twice of it, "
      f"{int((_wi['x_before'] > 1 + 1e-9).sum())} -> {int((_wi['x_after'] > 1 + 1e-9).sum())} above it")
print(_wi.assign(institution=_wi['institution'].str[:64]).round({'before': 1, 'after': 1, 'x_before': 2, 'x_after': 2})
      .to_string(index=False))
if WORKERS_HEADCOUNT.get('institution_floor_k'):   # the raise of the under-full institutions
    _wrr = _wr['raised']
    print(f"  ..  {len(_wrr)} named institution(s) raised to {WORKERS_HEADCOUNT['institution_floor_k']:g} x their published "
          f"staff after the spill: {_wr['raise_moved']:,.0f} Workers from {_wr['raise_donors']:,} units above their realistic headcount "
          f"in {_wr['raise_zones']} zone(s) (ring 0 = the institution's own zones), none taken below it; "
          f"{int((_wrr['short'] > 0.5).sum())} short of their published staff")
    print(_wrr.assign(institution=_wrr['institution'].str[:64]).round({'before': 1, 'after': 1, 'found': 1, 'short': 1})
          .to_string(index=False))
_wbw = _wg['how'].str.endswith('(both ways)')
_wbt = _wg[_wbw].groupby('atype').agg(units=('face', 'size'), raised=('face', lambda s_: int((s_ > _wg.loc[s_.index, 'exp'] + 0.5).sum())),
                                       Bosserhof=('exp', 'sum'), realistic=('face', 'sum'), after=('after', 'sum'))
print(f"  ..  {int(_wbw.sum()):,} food stores take the size rule of their type both ways "
      f"({', '.join(WORKERS_HEADCOUNT.get('size_rule_both_ways', ()))}): realistic headcount {_wg.loc[_wbw, 'face'].sum():,.0f} "
      f"against a Bosserhof value of {_wg.loc[_wbw, 'exp'].sum():,.0f}")
print(_wbt.round(0).to_string())
print('  ..  the 15 largest gainers (face = the realistic headcount):')
print(_wr['gainers'].head(15).round(1).to_string(index=False))
print('  ..  the largest zone-to-zone flows (ring: how far from the sending zone):')
print(_wr['flows'].head(15).round(0).to_string(index=False))
print('  ..  by activity type, the largest losses and gains:')
_wt = _wr['types']
print(pd.concat([_wt.head(10), _wt.tail(6)]).round(0).to_string())

  ok  rule R in 9.8s on 42,093 Workers units (41,885 in a zone): the Workers total holds (591,700.19); every zone holds min(its total, the sum of its upper limits) after the fill, at one level per zone; the spill only lifts units below their realistic headcount, never above it, and none of the 1,493 units of WORKERS_NO_SPILL; the 425 technical structures hold at most their realistic headcount; what an over-full zone sends equals its surplus; no unit is above its upper limit; no named institution above its published staff; the raise lifted 20 institution(s) to 1 x their published staff (0 short), taking only from units above their realistic headcount and never below it; the other categories are untouched; the zones hold what lies in them after the rule, their targets stay VISUM's
  ..  60 activity types: 24,763 units with an establishment band hold 225,484 -> 194,517 Workers, 17,122 open-ended units the rest; the realistic headcounts sum to 675,925 against a Bosserhof value of 727,950 (

## 8. The checks, the diagnostics and the file

A building's value per category is the sum of its units, a POI's its own
unit's. Checked: the region totals once more on the buildings, and every
building as the sum of its building, POI, rest and remainder units. Each
building gets its storeys and the rule that set them, its floor area, its
residential context, the floor its units occupy (`occupied_m2`: the units left
after the rules, the units the last-carrier guard restored included at their
floor before the rule; a rest unit lies on floor its building's POI units
already occupy, so it counts only where no other unit of the building is left;
checked against the floor area), its remainder, its unit and Workers rule, why
it has no unit, and the areas of the five labels without a zone total; each
POI its group, class, factor, floor family, occupied floor (0 for a POI
without a unit — a vacant, holiday-flat or dormitory slot keeps its floor
empty) and areas. The diagnostics: where the demand per m² of
occupied floor peaks, the zones whose target is 0 although they hold carriers
(for Workers, School, University and Kindergarten: how many share the targets
of their ring, take nothing from it, keep 0 or receive through a ring
fallback; only reported for the other categories), the Workers zones with a
near-zero total and what they hold, Workers per class, how much
Retail_Daily and Leisure go to the ring fallback, and our Workers per
sector and district next to VISUM's — a check only. The visitor values follow
the rates as meant (`VISITOR_RULES`); an audience
venue's meetup counts only beyond its own Leisure share, a building whose only
POI is a heritage feature takes its class rate unless it is filed as a culture
or leisure facility, and staff shops count no visitors; a note says
that Salzgitter's errands and sports values are probably too low, and Peine's
nearly as much, OSM mapping few practices and small shops there. Each building
and POI gets the labels its units carry after all the rules (`labels_used`,
`poi_labels_used`: a label of the seven where a unit in a zone carries it with
a weight above 0, one of the five where its visitor value is above 0; a carrier
in a zone whose total of that category is 0, or a school place whose type gets
no pupils, shows the label with 0 placed on it), checked against the numbers.

The file keeps only the necessary columns: per building
its id, zone, name, ALKIS function, class, the model's labels and the labels used, storeys, floor area,
occupied floor, the seven values, the five visitor values and `value_assumed`;
per POI its id, building, zone, name, the model's labels and the labels used, class, occupied floor and the
same values; per zone its target and placed value per category. The rules,
vacancy, areas and bookkeeping stay in this notebook.

In [9]:
# --- per building, per POI ------------------------------------------------------------------------
_pb = (units.pivot_table(index='building_id', columns='category', values='assigned', aggfunc='sum', fill_value=0.0)
       .reindex(index=bld['building_id'], columns=CATS, fill_value=0.0))
_pp = (units[units['kind'] == 'poi'].pivot_table(index='unit_id', columns='category', values='assigned', aggfunc='sum', fill_value=0.0)
       .reindex(index=pois['poi_id'], columns=CATS, fill_value=0.0))
for c in CATS:
    bld[col('assigned', c)] = _pb[c].to_numpy()
    pois[col('assigned', c)] = _pp[c].to_numpy()
pois['zone_no'] = pois['building_id'].map(bld.set_index('building_id')['zone_no']).astype('Int64')
# School by type per building and POI - for the check below; the file keeps School only
_sty = [scol('assigned', t) for t in SCHOOL_TYPES]
_sb = units.groupby('building_id')[_sty].sum().reindex(bld['building_id'], fill_value=0.0)
_sp = units[units['kind'] == 'poi'].groupby('unit_id')[_sty].sum().reindex(pois['poi_id'], fill_value=0.0)
for _c in _sty:
    bld[_c], pois[_c] = _sb[_c].to_numpy(), _sp[_c].to_numpy()

for c in CATS:
    if not np.isclose(bld[col('assigned', c)].sum() + ztab[col('unassigned', c)].sum(), ztab[col('target', c)].sum(), rtol=1e-9, atol=1e-6):
        raise AssertionError(f'{c}: the buildings do not add up to the zone total')
_by_kind = units.pivot_table(index=['building_id', 'kind'], columns='category', values='assigned', aggfunc='sum', fill_value=0.0).reindex(columns=CATS, fill_value=0.0)
_parts = _by_kind.groupby(level='building_id').sum().reindex(bld['building_id'], fill_value=0.0)
if not np.allclose(_parts.to_numpy(), bld[[col('assigned', c) for c in CATS]].to_numpy(), rtol=1e-9, atol=1e-6):
    raise AssertionError('a building is not the sum of its building, POI, rest and remainder units')
_sh = pois.groupby('building_id')[[col('assigned', c) for c in CATS]].sum()
_pk = _by_kind.xs('poi', level='kind') if 'poi' in _by_kind.index.get_level_values('kind') else _sh.iloc[0:0]
if not np.allclose(_sh.reindex(_pk.index).to_numpy(), _pk.to_numpy(), rtol=1e-9, atol=1e-6):
    raise AssertionError('the POI layer does not hold its units\' values')
if not (np.allclose(bld[_sty].sum(axis=1), bld[col('assigned', 'School')], rtol=1e-9, atol=1e-6)
        and np.allclose(pois[_sty].sum(axis=1), pois[col('assigned', 'School')], rtol=1e-9, atol=1e-6)):
    raise AssertionError('a building or POI whose school types do not add up to its School')
print('  ok  the buildings add up to the zone totals; a building is the sum of its building, POI, rest and remainder units; '
      'the POI layer holds the values of the POI units; the school types add up to School on every building and POI')

# --- the occupied floor, the areas and the rules per building and POI ------------------------------------
# occupied_m2: the units after the rules, the guard-restored ones included (at their floor before the rule); a rest unit lies on
# floor its building's POI units already occupy and counts only where no other unit of the building is left
_uok = uo[uo['no_unit_reason'].eq('') | uo['restored']]
_Oe = pd.Series(np.where(_uok['no_unit_reason'].eq(''), _uok['O'], _uok['O_pre']), index=_uok.index)
_isrest = _uok['kind'].eq('rest')
_floor = _Oe[~_isrest].groupby(_uok.loc[~_isrest, 'building_id']).sum()
_rest_only = _Oe[_isrest].groupby(_uok.loc[_isrest, 'building_id']).sum()
_rest_only = _rest_only[~_rest_only.index.isin(_floor.index)]
bld['occupied_m2'] = bld['building_id'].map(pd.concat([_floor, _rest_only])).fillna(0.0)
if (bld['occupied_m2'] > bld['floor_area_m2'] * (1 + 1e-6)).any():
    raise AssertionError(f'{int((bld["occupied_m2"] > bld["floor_area_m2"] * (1 + 1e-6)).sum())} buildings occupy more than their floor area')
bld['remainder_m2'] = bld['building_id'].map(_uok[_uok['kind'] == 'remainder'].groupby('building_id')['O'].sum()).fillna(0.0)
bld['residential_context'] = bld['RC'].map({True: 'dwellings', False: ''})
_bu = uo[uo['kind'] == 'building'].set_index('building_id')['unit_rule']
# a shared building: its POI units after the rules, its slots, the asked POIs a rule left without a unit
_pr = uo[uo['kind'].eq('poi')]
_pu_n = _pr[_pr['no_unit_reason'].eq('') | _pr['restored']].groupby('building_id').size()
_pl = _pr[_pr['no_unit_reason'].ne('') & ~_pr['restored'] & ~_pr['unit_id'].isin(pois.loc[pois['slot'] != '', 'poi_id'])]
_pl = _pl.groupby('building_id')['no_unit_reason'].agg(lambda s: f'{len(s)} asked POI(s) without a unit ({", ".join(sorted(set(s)))})')
_rest_b = set(_uok.loc[_isrest, 'building_id'])
def _shared_txt(b_, r_):
    n_u = int(_pu_n.get(b_, 0))
    return (f'shared by {n_u} POI unit(s)' + (f' and {int(r_.n_pois - r_.n_active)} slot(s)' if r_.n_pois > r_.n_active else '')
            + (f'; {_pl[b_]}' if b_ in _pl.index else '')
            + (', one facility on the sole-occupant floor' if r_.one_facility else '')
            + (f', {int(r_.n_capped)} point(s) capped' if r_.n_capped else '') + (', an empty slot per vacant pair' if r_.vacant_slot_share > 0 else '')
            + ('; only its rest unit carries labels' if n_u == 0 and b_ in _rest_b else ''))
_sr = pd.Series({b_: _shared_txt(b_, r_) for b_, r_ in shared.iterrows()}, dtype=object)
_lu = set(_landuse['building_id'])
bld['unit_rule'] = (bld['building_id'].map(_bu).fillna(bld['building_id'].map(_sr)).fillna('')
                    + np.where(bld['building_id'].isin(_lu) & bld['building_id'].isin(_shared_b), '; landuse share re-split', '')
                    + np.where(bld['building_id'].isin(_to_bu), '; its asked POIs were non-occupants', '')
                    + np.where(bld['class_06'] != bld['bosserhof_class'], '; reclassified from ' + bld['class_06'], ''))
_kept_lab = bld['building_id'].isin(uo.loc[uo['rule'].eq('no_evidence') & uo['labels'].map(len).gt(1), 'building_id'])
_wrule = {'church': 'church: Workers 0', 'volunteer_fire': 'volunteer fire station: Workers 0', 'sports_hall': 'sports hall / clubhouse: Workers 0',
          'no_evidence': 'no-evidence residential: one storey, Workers only', 'holiday_flat': 'holiday flat: no unit', 'dormitory': 'dormitory: no unit'}
_rnote = bld['building_id'].map(_restore.groupby('building_id')['category'].agg(lambda s: ', '.join(sorted(set(s)))))
bld['workers_rule'] = (bld['rule'].map(_wrule).fillna('').where(~_kept_lab, 'no-evidence residential: one storey, labels kept (retail / commercial twin)')
                       .where(~bld['rule'].isin(['holiday_flat', 'dormitory']) | _rnote.notna(), '')
                       + np.where(_rnote.notna(), ' (the last-carrier guard restored ' + _rnote.fillna('') + ')', ''))
_has_unit = set(_uok['building_id'])
bld['no_unit_reason'] = np.select(
    [bld['building_id'].isin(_has_unit), bld['vacancy'] != ''],     # a building with a unit has no reason (a kept POI)
    ['', bld['vacancy'].where(bld['vacancy'].isin(['gone', 'closed']), 'vacant').to_numpy(dtype=object)], default=bld['building_id'].map(uo.groupby('building_id')['no_unit_reason'].agg(lambda s: '; '.join(sorted(set(x for x in s if x))))).fillna(''))
_vp_b = (~bld['building_id'].isin(_has_unit) & bld['vacancy'].eq('')     # emptied by its vacant asked POIs
         & bld['building_id'].isin(set(pois.loc[pois['vacancy'].ne(''), 'building_id'])))
bld.loc[_vp_b, 'no_unit_reason'] = ('vacant POI; ' + bld.loc[_vp_b, 'no_unit_reason']).str.rstrip('; ')
_puo = uo[uo['kind'] == 'poi'].set_index('unit_id')
pois['floor_family'] = pois['poi_id'].map(_puo['family']).fillna(pois['poi_id'].map(slots.set_index('unit_id')['family'])).fillna('')
pois['occupied_m2'] = pois['poi_id'].map(_puo.loc[_puo['no_unit_reason'].eq('') | _puo['restored'], 'O']).fillna(0.0)
pois['no_unit_reason'] = np.select([pois['non_occupant'], pois['vacancy'] != '',
                                    pois['building_id'].isin(_vac_b) & ~pois['poi_id'].isin(_puo.index[_puo['no_unit_reason'].eq('')])],
                                   ['non-occupant', pois['vacancy'].where(pois['vacancy'].eq('closed'), 'vacant').to_numpy(dtype=object),
                                    'vacant building'],
                                   default=pois['poi_id'].map(_puo['no_unit_reason']).fillna(''))
pois.loc[pois['poi_id'].isin(_restore.loc[_restore['kind'] == 'poi', 'unit_id']), 'no_unit_reason'] = ''
_ab = areas.pivot_table(index='building_id', columns='label', values='area_m2', aggfunc='sum', fill_value=0.0).reindex(columns=AREA_LABELS, fill_value=0.0)
_ap = areas[areas['kind'] == 'poi'].pivot_table(index='unit_id', columns='label', values='area_m2', aggfunc='sum', fill_value=0.0).reindex(columns=AREA_LABELS, fill_value=0.0)
for l in AREA_LABELS:
    bld[f'area_{l}_m2'] = bld['building_id'].map(_ab[l]).fillna(0.0)
    pois[f'area_{l}_m2'] = pois['poi_id'].map(_ap[l]).fillna(0.0)
# the values of the labels without a zone total: visitors per working day = area x rate / 100.
# A POI unit looks up its own group and use, a building unit its only POI's, a rest unit only its building's class.
_i = {'low': 0, 'mid': 1, 'high': 2}[VISITOR_RATE_SCENARIO]
_pq = pois.set_index('poi_id')
_staff = re.compile(VISITOR_STAFF_ONLY_PATTERN, re.IGNORECASE)
_ub = uo[units_ok].groupby('building_id')['labels'].agg(lambda s: set().union(*s)).to_dict()
for b_, l_ in zip(units['building_id'], units['label']):          # with the rows the last-carrier guard restored
    _ub.setdefault(b_, set()).add(l_)
_gone = edu_drops.groupby('building_id')['label'].agg(set)         # labels the who-carries rules dropped: no school any more
_co = {b_: bool({'school', 'university'} & ((set(_L[b_]) - _gone.get(b_, set())) | _ub.get(b_, set()))) for b_ in areas['building_id'].unique()}
_ul = uo.set_index('unit_id')['labels']
_uO = uo.set_index('unit_id')['O']
_ukall = uo.set_index('unit_id')['k_all']
_VR = dict(group_rates=VISITOR_GROUP_RATES, class_rates=VISITOR_CLASS_RATES, type_rates=VISITOR_TYPE_RATES,
           default_rates=VISITOR_DEFAULT_RATES, sport_uses=VISITOR_SPORT_USES, dance_uses=VISITOR_DANCE_USES,
           events_uses=VISITOR_EVENTS_USES, car_uses=VISITOR_CAR_USES, gastro_class='restaurants / gastronomy',
           hotel_classes={'hotels', 'hotels with conference areas'}, industry_class='Industrial operations / Production')
from types import SimpleNamespace as _NS
def _txt(x):
    return x if isinstance(x, str) else ''
_R8 = VISITOR_RULES                             # the visitor rates as meant (lib/visitor_rates.py)
_az = _R8.get('audience_zero') or {}            # the audience zero only as far as the venue's Leisure share holds it
_aud_net = set()                                # (kind, unit, label) the loop gave back their audience rate, netted after the loop
_aud_keep0 = frozenset(_az.get('keep_zero_buildings', ()))
_acc_re = re.compile(r'"access": "([^"]+)"')
def _own_text(b_, uid, kind):                    # the unit's own names and type
    if kind == 'poi':
        return f"{_txt(_pq.at[uid, 'name'])} | {_txt(_pq.at[uid, 'poi_type'])}"
    s_ = f"{_txt(_b.at[b_, 'name'])} | {_txt(_b.at[b_, 'osm_twin_name'])} | {_txt(_b.at[b_, 'llm_type'])}"
    if kind == 'building' and _b.at[b_, 'n_pois'] == 1 and b_ in _sole.index:
        s_ += f" | {_txt(_sole.at[b_, 'name'])}"
    return s_
def _access(uid, kind):                          # a POI's OSM access tag
    if kind != 'poi':
        return ''
    m_ = _acc_re.search(_txt(_pq.at[uid, 'tags']))
    return m_.group(1) if m_ else ''
_vals = []
for b_, uid, kind, lab, area in areas[['building_id', 'unit_id', 'kind', 'label', 'area_m2']].itertuples(index=False):
    r = _BX[b_]
    cls, btype, head_ = _b.at[b_, 'bosserhof_class'], _b.at[b_, 'llm_type'], _b.at[b_, 'head']
    if kind == 'poi':
        g, use = _pq.at[uid, 'group'], _pq.at[uid, 'poi_use']
        staff = bool(_staff.search(f"{_txt(_pq.at[uid, 'name'])} {_txt(_pq.at[uid, 'poi_type'])}")) or (
            g in VISITOR_STAFF_ONLY_GROUPS and cls in VISITOR_STAFF_ONLY_CLASSES)
    else:
        has = kind == 'building' and r.n_pois == 1 and b_ in _sole.index
        g, use = (_sole.at[b_, 'group'], _sole.at[b_, 'poi_use']) if has else (None, None)
        _hp = _R8.get('heritage_poi')           # a heritage POI names the building's past, not its use
        if _hp and use in _hp['uses'] and cls not in _hp['keep_classes']:
            g, use = None, None
        staff = bool(_staff.search(_txt(head_)))
    staff = staff or (kind, uid) in staff_shop_keys      # a staff shop inside a plant
    u = _NS(group=g, use=use, cls=cls, btype=_txt(btype), staff_only=staff, leisure='leisure' in _ul[uid], co_school=_co[b_],
            plain_home=kind == 'building' and bool(r.pure_home) and r.n_pois == 0, parking=bool(r.parking),
            text=_own_text(b_, uid, kind), zoned=bool(pd.notna(_b.at[b_, 'zone_idx'])), labels=_ul[uid], access=_access(uid, kind), kind=kind)
    O, k = float(_uO[uid]), max(int(_ukall[uid]), 1)
    if lab == 'errands' and use in VISITOR_PER_OUTLET and not staff:
        if _R8.get('no_zone_zero') and not u.zoned:        # a building in no zone: VISUM never reads it
            _vals.append((b_, uid, kind, lab, 0.0, 'ZERO', 'zero: no zone', False))
            continue
        lo_mid_hi = VISITOR_PER_OUTLET[use]
        _vals.append((b_, uid, kind, lab, lo_mid_hi[_i] / k, f'per outlet: {use}', 'use', lo_mid_hi[3]))
        continue
    rid, tier = visitor_rates.visitor_rate_id(lab, u, **_VR, rules=_R8)
    if _az.get('net_of_leisure') and tier == 'zero: also leisure (audience rate)' and b_ not in _aud_keep0:
        rid, tier = visitor_rates.visitor_rate_id(lab, u, **_VR, rules={**_R8, 'audience_zero': None})
        _aud_net.add((kind, uid, lab))
    rate, assumed = VISITOR_RATE_ANCHORS[rid][_i], VISITOR_RATE_ANCHORS[rid][3]
    capm2 = POINT_SIZE_CAP_M2.get(g, POINT_SIZE_CAP_DEFAULT_M2) if isinstance(g, str) else np.inf
    _cc = _R8.get('counter_cap')
    if tier == 'use' and g in VISITOR_SMALL_FORMAT_GROUPS and O > capm2:
        rc, tc = visitor_rates.visitor_rate_id(lab, _NS(**{**vars(u), 'group': None, 'use': None}), **_VR, rules=_R8)
        if _cc and rc == 'E_COUNTER':            # the counter rate covers at most the counter cap, whatever gives it
            rc = _cc['beyond']
        rate_c = VISITOR_RATE_ANCHORS[rc][_i]
        v = (capm2 * rate + (O - capm2) * rate_c) / k / 100
        _vals.append((b_, uid, kind, lab, v, f'{rid} on {capm2:,.0f} m2, {rc} on the rest', 'use + class',
                      (assumed and rate > 0) or (VISITOR_RATE_ANCHORS[rc][3] and rate_c > 0)))
    elif _cc and rid == 'E_COUNTER' and tier != 'use' and O > _cc['cap']:
        rb = VISITOR_RATE_ANCHORS[_cc['beyond']][_i]
        v = (_cc['cap'] * rate + (O - _cc['cap']) * rb) / k / 100
        _vals.append((b_, uid, kind, lab, v, f"{rid} on {_cc['cap']:,.0f} m2, {_cc['beyond']} on the rest", tier + ' (capped)',
                      (assumed and rate > 0) or (VISITOR_RATE_ANCHORS[_cc['beyond']][3] and rb > 0)))
    else:
        f_ = 1.0
        _ob = _R8.get('office_box')               # a big low office box counts business and errands on one storey
        if _ob and lab in _ob['labels'] and kind in ('building', 'rest') and cls in _ob['classes'] and not tier.startswith('zero'):
            A_ = float(_b.at[b_, 'area_m2'])
            e_ = _b.at[b_, 'height_eaves_m']
            e_ = _b.at[b_, 'mean_height_m'] if pd.isna(e_) else e_
            if A_ > _ob['min_footprint'] and pd.notna(e_) and e_ <= _ob['max_eaves'] and O > A_:
                f_ = A_ / O
        _vals.append((b_, uid, kind, lab, area * f_ * rate / 100, rid, tier + (' (one storey)' if f_ < 1 else ''), assumed and rate > 0))
values = pd.DataFrame(_vals, columns=['building_id', 'unit_id', 'kind', 'label', 'value', 'rate', 'tier', 'assumed'])
# the audience venues net of their own Leisure share (VISITOR_RULES audience_zero net_of_leisure): max(0, the meetup at
# the audience rate - the unit's Leisure share); a venue that ends at 0 keeps the zero tier
_lei_u = units[units['category'].eq('Leisure')].groupby(['kind', 'unit_id'])['assigned'].sum()
_an = pd.Series([(k_, u_, l_) in _aud_net for k_, u_, l_ in zip(values['kind'], values['unit_id'], values['label'])], index=values.index)
_av = (values['value'] - pd.Series([_lei_u.get((k_, u_), 0.0) for k_, u_ in zip(values['kind'], values['unit_id'])],
                                   index=values.index)).clip(lower=0.0)
values.loc[_an, 'value'] = _av[_an]
values.loc[_an & _av.le(0), ['rate', 'tier', 'assumed']] = ['ZERO', 'zero: also leisure (audience rate)', False]
values.loc[_an & _av.gt(0), 'tier'] = values.loc[_an & _av.gt(0), 'tier'] + ' (net of its Leisure share)'
if len(values) != len(areas) or not np.isfinite(values['value']).all() or (values['value'] < 0).any():
    raise AssertionError('a label area without a finite, non-negative visitor value')
if (values.loc[values['tier'].str.startswith('zero'), 'value'] != 0).any():
    raise AssertionError('a decided zero (staff only, hotel, parking, plain home, cafe with leisure, school building, academy) with a value')
_val_cols = [f'value_{l}' for l in AREA_LABELS]
for l in AREA_LABELS:
    _v = values[values['label'] == l]
    bld[f'value_{l}'] = bld['building_id'].map(_v.groupby('building_id')['value'].sum()).fillna(0.0)
    pois[f'value_{l}'] = pois['poi_id'].map(_v[_v['kind'] == 'poi'].groupby('unit_id')['value'].sum()).fillna(0.0)
_va = values[values['assumed'] & (values['value'] > 0)]
bld['value_assumed'] = bld['building_id'].map(_va.groupby('building_id')['label'].agg(lambda s: '; '.join(sorted(set(s))))).fillna('')
pois['value_assumed'] = pois['poi_id'].map(_va[_va['kind'] == 'poi'].groupby('unit_id')['label'].agg(lambda s: '; '.join(sorted(set(s))))).fillna('')
pois['visitor_rates'] = pois['poi_id'].map(values[values['kind'] == 'poi'].assign(t=values['label'] + ': ' + values['rate'] + ' (' + values['tier'] + ')')
                                           .groupby('unit_id')['t'].agg('; '.join)).fillna('')
_val_cols += ['value_assumed']

# --- the labels used ------------------------------------------------------------------------------------------------------------
# the labels the place's units carry after all the rules, next to the model's labels (mid_labels_merged, poi_mid_labels): a label
# of the seven with a zone total where a unit of the place in a zone has a row of its category with a weight above 0 (the rows the
# last-carrier guard restored count; a zone whose total is 0 still counts its carriers); one of the five without a total where its
# visitor value is above 0 (a decided zero is no use). A building: all its units - building, POI, rest and remainder; a POI: its
# own unit. A place without a unit, or in no zone, uses none (''). In the catalogue order, ';'-joined like the model's columns
_lu_rows = pd.concat([units.loc[units['zone_idx'].notna() & units['weight'].gt(0), ['building_id', 'unit_id', 'kind', 'label']],
                      values.loc[values['value'].gt(0), ['building_id', 'unit_id', 'kind', 'label']]], ignore_index=True)
def _lu_join(s_):
    return ';'.join(x_ for x_ in LLM_ACTIVITY_LABELS if x_ in set(s_))
bld['labels_used'] = bld['building_id'].map(_lu_rows.groupby('building_id')['label'].agg(_lu_join)).fillna('')
pois['poi_labels_used'] = pois['poi_id'].map(_lu_rows[_lu_rows['kind'].eq('poi')].groupby('unit_id')['label'].agg(_lu_join)).fillna('')
# checks: on every building and POI a label of the seven is used exactly where its summed zoned weight is above 0 and one of the
# five exactly where its summed value is above 0 - nothing else; an assigned value above 0 always has its label, a visitor value
# above 0 its label and a used visitor label its value
_lu_w = units[units['zone_idx'].notna()]
_lu_bad = []
for _lu_t, _lu_k, _lu_c, _lu_poi in ((bld, 'building_id', 'labels_used', False), (pois, 'poi_id', 'poi_labels_used', True)):
    _lu_by = 'unit_id' if _lu_poi else 'building_id'
    _lu_s7 = (_lu_w[_lu_w['kind'].eq('poi')] if _lu_poi else _lu_w).groupby([_lu_by, 'label'])['weight'].sum()
    _lu_s5 = (values[values['kind'].eq('poi')] if _lu_poi else values).groupby([_lu_by, 'label'])['value'].sum()
    _lu_exp = {}
    for (k_, x_), v_ in list(_lu_s7.items()) + list(_lu_s5.items()):
        if v_ > 0:
            _lu_exp.setdefault(k_, set()).add(x_)
    _lu_got = [set(s_.split(';')) - {''} for s_ in _lu_t[_lu_c]]
    _lu_bad += [k_ for k_, g_ in zip(_lu_t[_lu_k], _lu_got) if g_ != _lu_exp.get(k_, set())]
    for x_, c_ in MID_LABEL_TO_ACTIVITY.items():
        _lu_bad += [k_ for k_, a_, g_ in zip(_lu_t[_lu_k], _lu_t[col('assigned', c_)], _lu_got) if a_ > 0 and x_ not in g_]
    for x_ in AREA_LABELS:
        _lu_bad += [k_ for k_, v_, g_ in zip(_lu_t[_lu_k], _lu_t[f'value_{x_}'], _lu_got) if (v_ > 0) != (x_ in g_)]
if _lu_bad:
    raise AssertionError(f'the labels used: {len(_lu_bad)} building(s) / POI(s) whose labels used are not their labels with a weight '
                         f'or a value above 0, e.g. {sorted(set(_lu_bad))[:5]}')
_lu_m = [set(s_.split(';')) for s_ in bld['mid_labels_merged']]
_lu_u = [set(s_.split(';')) - {''} for s_ in bld['labels_used']]
print(f'  ok  the labels used: {sum(m_ != u_ for m_, u_ in zip(_lu_m, _lu_u)):,} of {len(bld):,} buildings and '
      f'{int((pois["poi_labels_used"] != pois["poi_mid_labels"]).sum()):,} of {len(pois):,} asked POIs use other labels than the model '
      f'gave them ({int(bld["labels_used"].eq("").sum()):,} buildings and {int(pois["poi_labels_used"].eq("").sum()):,} POIs none: no unit, '
      'no zone or nothing above 0); a label is used where its zoned weight or its visitor value is above 0. Buildings that gain (+) / '
      'lose (-) a label: ' + ' · '.join(f'{x_} +{sum(x_ in u_ and x_ not in m_ for m_, u_ in zip(_lu_m, _lu_u)):,} '
                                        f'-{sum(x_ in m_ and x_ not in u_ for m_, u_ in zip(_lu_m, _lu_u)):,}' for x_ in LLM_ACTIVITY_LABELS))
_tv = values.groupby('label').agg(units=('unit_id', 'size'), visits=('value', 'sum'))
_tv['assumed %'] = 100 * values[values['assumed']].groupby('label')['value'].sum().reindex(_tv.index).fillna(0) / _tv['visits'].where(_tv['visits'] > 0)
print(f'  ok  visitor values ({VISITOR_RATE_SCENARIO} rates, arrivals per average working day): '
      + ' · '.join(f'{l} {_tv.at[l, "visits"]:,.0f} on {_tv.at[l, "units"]:,} units ({_tv.at[l, "assumed %"]:.0f} % on assumed rates)'
                   for l in AREA_LABELS if l in _tv.index))
print('  ..  decided zeros: ' + ' · '.join(f'{k[6:]} {v:,}' for k, v in values.loc[values['tier'].str.startswith('zero'), 'tier'].value_counts().items())
      + f'; per outlet {int(values["rate"].str.startswith("per outlet").sum()):,}; small-format rates on the capped area only '
      f'{int(values["tier"].eq("use + class").sum()):,}')
print(f'  ok  occupied floor area {bld["occupied_m2"].sum() / 1e6:,.2f} km2 of {bld["floor_area_m2"].sum() / 1e6:,.2f} km2 '
      f'({bld["remainder_m2"].sum() / 1e6:,.2f} km2 of it remainder; no building above its floor area); the five area columns hold '
      + ' · '.join(f'{l} {bld[f"area_{l}_m2"].sum() / 1e3:,.0f}' for l in AREA_LABELS) + ' thousand m2; '
      + f'value columns for {", ".join(_val_cols)}')

# --- diagnostics ------------------------------------------------------------------------------------
_lb = VISITOR_RULES.get('lower_bound_note')
if _lb:                                          # thin OSM coverage: errands and sports probably too low there
    _ln = values['building_id'].map(bld.set_index('building_id')['ags']).fillna('').astype(str).str.startswith(_lb['ags_prefix'])
    _lu = values[_ln & (values['tier'].eq('use') | values['tier'].str.contains('type words', regex=False))]   # own use or type words
    print(f'  ..  note: {_lb["district"]}\'s errands and sports values are probably too low, and Peine\'s nearly as much - OSM '
          f'maps few practices and small shops there '
          f'({_lb["district"]}\'s values from a POI\'s own use or the place\'s own type words: ' + ' · '.join(f'{l} {_lu.loc[_lu["label"].eq(l), "value"].sum():,.0f}'
                                                           for l in ('errands', 'sports', 'lessons')) + ' a day)')
print()
print('  ..  the highest value per m2 of occupied floor area, per category:')
for c in CATS:
    _d = bld[col('assigned', c)] / bld['occupied_m2'].where(bld['occupied_m2'] > 0)
    if _d.notna().any():
        i = _d.idxmax()
        _nm = bld.at[i, 'name'] if isinstance(bld.at[i, 'name'], str) and bld.at[i, 'name'] else bld.at[i, 'label_en']
        print(f'        {c:<17} {_d[i]:.4f} per m2 - {bld.at[i, "building_id"]} "{_nm}" '
              f'({bld.at[i, "occupied_m2"]:,.0f} m2, {bld.at[i, col("assigned", c)]:,.1f}); median over the buildings that received any '
              f'{_d[bld[col("assigned", c)] > 0].median():.5f}')
for c in CATS:
    _z0 = (ztab[col('target', c)] == 0) & ztab.index.isin(units.loc[(units['category'] == c), 'zone_idx'].dropna().astype(int))
    if _z0.any():
        _sr = ztab[col('shared_ring', c)].notna()
        _n0 = int(_z0.sum())
        _ns = int((_z0 & _sr & ztab[col('shared_in', c)].gt(0)).sum())       # join and take a part
        _nj = int((_z0 & _sr & ztab[col('shared_in', c)].le(0)).sum())       # join but take nothing
        _nf = int((_z0 & ~_sr & ztab[col('assigned', c)].gt(0)).sum())       # join nothing, reached by a ring fallback
        _njz = (units[units['category'].eq(c) & units['zone_idx'].notna()].groupby('zone_idx')['no_join'].all()
                if 'no_join' in units.columns else pd.Series(dtype=bool))
        _nn = int((_z0 & ~_sr & ztab[col('assigned', c)].le(0) & ztab.index.isin(_njz.index[_njz.astype(bool)])).sum())   # all no-join
        _mr = REDIST_SHARE_ZERO_TARGET_MAX_RINGS.get(c)
        _nfz = len(far_joins.get(c, {}))                                      # zones with places that join beyond ring 1
        _nfa = int((_z0 & ztab[col('shared_ring', c)].ge(2).fillna(False)).sum())   # ... with all their joining places
        print(f'  ..  {c}: {_n0} zone(s) with target 0 hold carriers of it'
              + (f' - {_ns} share the targets of their ring'
                 + (f' ({_nfz} of them with places that join beyond the touching zones, having nothing to take there; '
                    f'{_nfa} entirely)' if _nfz else '')
                 + (f', {_nj} join but take nothing (no zone they join has a target their places can take)' if _nj else '')
                 + (f', {_nn} join no neighbour (their places are known only from the cadastre or a bare OSM outline)' if _nn else '')
                 + f', {_n0 - _ns - _nj - _nf - _nn} keep 0 (no zone with a target and places of its own '
                 + ('within reach)' if _mr is None else f'within ring {_mr}'
                    + (f', nor one with a total they could take within ring {REDIST_SHARE_ZERO_TARGET_FAR_RINGS[c]})'
                       if c in REDIST_SHARE_ZERO_TARGET_FAR_RINGS else ')'))
                 + (f', {_nf} share nothing but receive through the ring fallback of a zone without a place' if _nf else '')
                 if c in REDIST_SHARE_ZERO_TARGET_CATEGORIES else ''))
    if near_zero.get(c):                          # near-zero totals
        _zl, _nzr = near_zero[c], REDIST_SHARE_NEAR_ZERO_TARGET[c]
        _Ez = units[units['category'].eq(c) & units['zone_idx'].isin(_zl)].groupby('zone_idx')['weight'].sum() / 100.0
        _kz = [z_ for z_ in _zl if pd.isna(ztab.at[z_, col('shared_ring', c)])]
        _az, _sz = ztab.loc[_zl, col('assigned', c)].sum(), ztab.loc[_zl, col('shared_in', c)].sum()
        _az18 = None                              # Workers: the realistic headcounts of section 7 move Workers
        if c == 'Workers':                        # between zones - what the sharing gave them is the allocation's total
            _az18, _az = _az, float(workers_rule['T'].reindex(_zl, fill_value=0.0).sum())
        _rgt = ('touching zone' if _nzr['max_ring'] == 1 else 'zone' if _nzr['max_ring'] is None
                else f'zone within ring {_nzr["max_ring"]}')
        print(f'  ..  {c}: {len(_zl)} zone(s) with a near-zero total (above 0 but under {100 * _nzr["max_share"]:g} % of their own '
              f'expectation, with {_nzr["min_expected"]:,.0f} or more expected) had {ztab.loc[_zl, col("target", c)].sum():,.1f} for '
              f'{_Ez.sum():,.0f} expected and now hold {_az:,.0f}, {_sz:,.0f} of it from the zones they join ({_az / _Ez.sum():.2f} x '
              'the expectation)'
              + (f'; after the realistic headcounts of section 7 they hold {_az18:,.0f}' if _az18 is not None else '')
              + (f'; {len(_kz)} zone(s) find no {_rgt} with a total that is not near zero and places of its own and keep only their '
                 'own total: ' + ', '.join(zones.at[z_, 'NAME'] for z_ in _kz) if _kz else ''))
_wc = bld.groupby('bosserhof_class')[col('assigned', 'Workers')].sum().sort_values(ascending=False)
print('  ..  Workers by Bosserhof class, the largest eight: ' + ' · '.join(f'{k} {v:,.0f}' for k, v in _wc.head(8).items()))
for c in ('Retail_Daily', 'Leisure'):
    _fb = ztab[col('fallback_ring', c)].fillna(0) > 0
    print(f'  ..  {c}: {int(_fb.sum())} zone(s) without a carrier of their own send {ztab.loc[_fb, col("target", c)].sum():,.0f} '
          f'({100 * ztab.loc[_fb, col("target", c)].sum() / ztab[col("target", c)].sum():.2f} %) to their ring (errands and sports / meetup / '
          'lessons feed no category)')

# the Workers sectors - only a check: VISUM's split is one fixed mix per city or district, and a class's
# sector is a reading of the building type (WORKER_SECTOR_CLASSES)
_sec_of = {k: s for s, ks in WORKER_SECTOR_CLASSES.items() for k in ks}
_rest_sec = next(s for s in WORKER_SECTOR_COLUMNS if s not in WORKER_SECTOR_CLASSES)
_wu = units[units['category'] == 'Workers'].dropna(subset=['zone_idx'])
_wcls = _wu['unit_id'].map(pois.set_index('poi_id')['poi_class']).where(_wu['kind'] == 'poi').fillna(_wu['building_id'].map(_b['bosserhof_class']))
_ours = (_wu.assign(sector=_wcls.map(_sec_of).fillna(_rest_sec), district=_wu['zone_idx'].astype(int).map(zones[ZONE_DISTRICT_COL]))
         .pivot_table(index='district', columns='sector', values='assigned', aggfunc='sum', fill_value=0.0).reindex(columns=list(WORKER_SECTOR_COLUMNS), fill_value=0.0))
_vis = zones.groupby(ZONE_DISTRICT_COL)[list(WORKER_SECTOR_COLUMNS.values())].sum().set_axis(list(WORKER_SECTOR_COLUMNS), axis=1)
_sect = pd.concat({'ours %': 100 * _ours.div(_ours.sum(axis=1), axis=0), 'VISUM %': 100 * _vis.div(_vis.sum(axis=1), axis=0)}, axis=1)
print('  ..  Workers per sector and district, ours next to VISUM\'s - a check only, never used: VISUM\'s split is one fixed mix per city '
      'or district, and the statistics count a company\'s offices in its sector where we count an office:')
print(_sect.round(1).to_string())

# --- the file --------------------------------------------------------------------------------------
# only the necessary columns: who and where, the class and labels, the floor the values rest on,
# the seven assigned values, the five visitor values and the mark on our own rates; next to the model's labels the labels
# the rules use (labels_used, poi_labels_used); the rules, vacancy, areas, factors, school types and fallback
# bookkeeping stay in this notebook
_assigned = [col('assigned', c) for c in CATS]
_bout = gpd.GeoDataFrame(bld[['building_id', 'zone_no', 'name', 'label_en', 'bosserhof_class', 'mid_labels_merged', 'labels_used', 'storeys',
                              'floor_area_m2', 'occupied_m2'] + _assigned + _val_cols].copy(), geometry=bld.geometry.values, crs=bld.crs)
_pout = gpd.GeoDataFrame(pois[['poi_id', 'building_id', 'zone_no', 'name', 'poi_mid_labels', 'poi_labels_used', 'poi_class', 'occupied_m2'] + _assigned
                              + _val_cols].copy(), geometry=pois.geometry.values, crs=pois.crs)
if not set(shapely.get_type_id(_zgeom)) <= {3, 6}:               # Polygon, MultiPolygon
    raise AssertionError('make_valid turned a zone into something other than a polygon')
_zwrite = np.array([g if g.geom_type == 'MultiPolygon' else shapely.MultiPolygon([g]) for g in _zgeom], dtype=object)
_zcols = [col(p, c) for c in CATS for p in ('target', 'assigned')]     # assigned: what lies in the zone after the fallback and sharing
_zout = gpd.GeoDataFrame(pd.concat([zones[[ZONE_ID_COL, 'NAME']], ztab[_zcols]], axis=1), geometry=_zwrite, crs=zones.crs)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if REDISTRIBUTION_FILE.exists():
    try:
        REDISTRIBUTION_FILE.unlink()
    except PermissionError as e:
        raise RuntimeError(f'{REDISTRIBUTION_FILE.name} is locked - close it in QGIS and rerun this cell. '
                           f'Original error: {e}') from None
t0 = time.perf_counter()
for _layer, _frame in (('buildings', _bout), ('pois', _pout), ('zones', _zout)):
    print(f'Writing {len(_frame):,} rows x {len(_frame.columns)} columns -> {REDISTRIBUTION_FILE.name} : {_layer} ...', flush=True)
    _frame.to_file(REDISTRIBUTION_FILE, layer=_layer, driver='GPKG')
print(f'  ok  {REDISTRIBUTION_FILE.stat().st_size / 1e6:,.1f} MB  [{time.perf_counter() - t0:,.0f}s]')
_layers = {l[0]: pyogrio.read_info(REDISTRIBUTION_FILE, layer=l[0])['features'] for l in pyogrio.list_layers(REDISTRIBUTION_FILE)}
if _layers != {'buildings': len(_bout), 'pois': len(_pout), 'zones': len(_zout)}:
    raise AssertionError(f'read back {_layers}')
_back = pyogrio.read_dataframe(REDISTRIBUTION_FILE, layer='buildings', read_geometry=False, columns=[col('assigned', c) for c in CATS])
print(f'  ok  read back {_layers}; the buildings layer sums to: '
      + ' · '.join(f'{c} {_back[col("assigned", c)].sum():,.0f}' for c in CATS))

  ok  the buildings add up to the zone totals; a building is the sum of its building, POI, rest and remainder units; the POI layer holds the values of the POI units; the school types add up to School on every building and POI


  ok  the labels used: 7,531 of 39,786 buildings and 910 of 7,864 asked POIs use other labels than the model gave them (1,625 buildings and 202 POIs none: no unit, no zone or nothing above 0); a label is used where its zoned weight or its visitor value is above 0. Buildings that gain (+) / lose (-) a label: work +0 -3,744 · university +0 -61 · school +12 -89 · childcare +0 -53 · retail_daily +104 -468 · retail_non_daily +0 -423 · leisure +218 -548 · sports +0 -23 · errands +4 -2,023 · meetup +0 -1,080 · lessons +0 -210 · business +0 -606
  ok  visitor values (mid rates, arrivals per average working day): business 39,593 on 3,774 units (0 % on assumed rates) · errands 241,928 on 9,522 units (61 % on assumed rates) · sports 109,187 on 1,966 units (19 % on assumed rates) · meetup 35,906 on 3,817 units (15 % on assumed rates) · lessons 14,264 on 834 units (60 % on assumed rates)
  ..  decided zeros: also leisure 788 · school or university building 205 · plain home 191 · no zone 95 · also l

Writing 7,864 rows x 22 columns -> 08_redistribution.gpkg : pois ...


Writing 880 rows x 17 columns -> 08_redistribution.gpkg : zones ...


  ok  26.9 MB  [1s]
  ok  read back {'buildings': 39786, 'pois': 7864, 'zones': 880}; the buildings layer sums to: Workers 591,700 · School 149,999 · University 38,253 · Kindergarten 37,323 · Retail_Daily 908,859 · Retail_Non-Daily 777,989 · Leisure 222,936


## Where this leaves us

Step 08 runs on occupied floor area: every zone target sits on the units of
its zone in proportion to the floor they occupy — or, where a zone has no
carrier of a category, on the nearest ring of zones that has one — and the
totals hold (Workers as far as the buildings can realistically hold them;
below). The places of a zone whose Workers, School, University or
Kindergarten target is 0 share the targets of the nearest ring of zones with a
target and places of their own (Workers at any distance, the other three the
touching zones, up to ring 3 where those hold nothing a place could take); so
do the places of a zone whose Workers total is above 0 but under 1 % of its
own expectation, with the touching zones only — such a zone keeps its small
total and takes no joiners. The labels of steps 06 and 07 are reconciled by
OSM use: eating places carry leisure, pharmacies and fuel stations with a shop
daily shopping, beverage markets, kiosks and tobacco shops daily instead of
non-daily shopping, services no shopping, leisure venues leisure, and a rest
label goes where a POI of its building is its evident source. Staff catering
carries no shopping or leisure, OSM-only buildings on step 04's placeholder
height take the storeys of their class and size, the visitor rates apply as
meant, POI units mapped as points weigh their group's typical size, and the
Workers-0 rules and factors reach the places their tests describe. Big low
office boxes count one storey only with hall evidence (their halls take the
class of what they are), tenants are sized by their format, 7 malls occupy
their published retail floor, staff shops count as staff catering, an audience
venue's meetup counts only beyond its Leisure share, a building known only by
a heritage POI takes its class rate, and the file shows next to the model's
labels the labels the rules use. Places known to be closed and places the OSM
tags say are not in use give no unit, and sport halls and studios count one
storey where only the height gave more.

Workers are a real headcount, so no place
holds more Workers than its kind realistically can — an establishment at most
twice its realistic headcount and never above the published maximum of its
type, an open-ended employer 3 × its Bosserhof value (industry, logistics and
transport 5 ×), 39 named institutions at most their published staff; what an
over-full zone cannot hold goes to the under-filled places of the zones around
it, the touching zones first — but not into the unmapped upper floors of shops
and other establishments, nor into technical structures such as sewage plants,
greenhouses or silos, which hold no more than their realistic headcount
either. 46,998 Workers (7.9 %) spill out of their zone and 303 of 865 zone
totals change; no unit is left above the published maximum of its type (905
before the rule), and the other categories are as they were.

The OSM levels mapped over 4,373 ALKIS buildings count, tall culture halls
count 5 m storeys, food stores carry their published headcounts both ways, and
the 20 under-full named institutions are raised to their published staff from
the over-filled places around them. The 55 halls of the VW Wolfsburg plant
count their OSM levels, so the plant keeps 63,605 of its 70,309 VISUM Workers
(58,530 with one storey per hall). A building whose only site is a landuse area
does not count as having no evidence, so 460 residential-coded buildings keep
their model labels (`NO_EVIDENCE_LANDUSE`).

`08_redistribution.gpkg` carries the buildings, the asked POIs and
the zones with their values per category, the visitor values of the five labels
without a zone total and the floor they rest on — only the necessary columns.

### Next

The review in QGIS.